<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.es/cap06/cap06_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

# 6 Análisis de Documentos e Inspección Industrial

En la **Parte I — Procesamiento Digital de Imágenes (PDI)**, se estudiaron técnicas para la transformación y mejora de imágenes, como operaciones morfológicas, filtrado espacial, convoluciones, umbralización, segmentación y procesamiento en el dominio de la frecuencia.

La **Parte II — Visión por Computador (VC)** amplía este alcance al abordar la interpretación automática del contenido visual, involucrando la extracción de información, el reconocimiento de patrones y la toma de decisiones a partir de imágenes.

Este capítulo presenta esa transición mediante dos aplicaciones representativas:

1. **Análisis Automatizado de Documentos**, aplicado al procesamiento de formularios, evaluaciones y otros documentos estructurados mediante sistemas de reconocimiento óptico de marcas (*Optical Mark Recognition* – OMR);
2. **Inspección Industrial Automatizada**, orientada al control de calidad y a la detección de defectos en líneas de producción.

Estas aplicaciones integran técnicas de detección de estructuras geométricas, extracción de descriptores invariantes, reconocimiento de patrones y clasificación de objetos, constituyendo la base de diversos sistemas modernos de inspección visual y automatización.

## 6.1 Objetivos del Capítulo

Al final de este capítulo, el estudiante deberá ser capaz de:

* **Evaluar la influencia del preprocesamiento** en la calidad del reconocimiento automático de información en documentos;
* **Realizar reconocimiento óptico de caracteres** (OCR) para convertir documentos escaneados en texto codificado;
* **Aplicar técnicas de procesamiento de lenguaje natural**, incluida la traducción automática, al texto obtenido por OCR;
* Aplicar técnicas de **alineación y rectificación geométrica de documentos** utilizando la Transformada de Hough y transformaciones proyectivas;
* **Implementar sistemas de reconocimiento óptico de marcas** (OMR) para la lectura automatizada de evaluaciones y formularios;
* **Detectar y segmentar regiones de interés** con base en operaciones morfológicas, contornos y propiedades geométricas;
* **Decodificar marcadores bidimensionales y códigos de barras**, integrando bibliotecas de Visión por Computador a *pipelines* de procesamiento documental;
* **Desarrollar *pipelines* de Visión por Computador** para el análisis automatizado de documentos.

Este capítulo marca la transición del **Procesamiento Digital de Imágenes**, orientado a la transformación de imágenes, hacia la **Visión por Computador**, cuyo objetivo es interpretar el contenido visual, extraer información y respaldar procesos automatizados de análisis y toma de decisiones.

## 6.2 Configuración del Entorno

Los ejemplos de este capítulo utilizan bibliotecas ampliamente empleadas en
PDI-VC. El bloque siguiente
instala los paquetes necesarios; en entornos que ya los posean, la ejecución
puede omitirse.

In [1]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup()
from morph import mm

# instalar más dependencias además de morph.py para este capítulo
import sys, subprocess, importlib, shutil

def setup_cap06():
    """Instala dependencias de sistema y Python específicas del Capítulo 6
    (OCR, lectura de PDF, código de barras)."""

    # 1. Dependencias de sistema
    if 'google.colab' in sys.modules:
        print("[AMBIENTE] Google Colab. Configurando dependencias del sistema...")
        subprocess.run(
            "apt-get update && apt-get install -y poppler-utils "
            "libzbar0 tesseract-ocr tesseract-ocr-por",
            shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
    elif shutil.which("tesseract") is None:
        # Ambiente local sin tesseract: intenta instalar vía apt-get (requiere sudo/root)
        if shutil.which("apt-get"):
            print("[AMBIENTE] Local. Instalando tesseract-ocr vía apt-get (puede pedir contraseña)...")
            resultado = subprocess.run(
                "sudo apt-get update && sudo apt-get install -y tesseract-ocr tesseract-ocr-por",
                shell=True
            )
            if resultado.returncode != 0 or shutil.which("tesseract") is None:
                print(
                    "[AVISO] No fue posible instalar automáticamente. "
                    "Instale manualmente: sudo apt install tesseract-ocr tesseract-ocr-por"
                )
        else:
            print(
                "[AVISO] tesseract no encontrado y apt-get no disponible. "
                "Instale manualmente antes de ejecutar las celdas de OCR."
            )

    # 2. Dependencias del Python (instala solo las ausentes)
    pkgs = {
        "cv2": "opencv-python", "skimage": "scikit-image", "numpy": "numpy",
        "pdf2image": "pdf2image", "pandas": "pandas", "tabulate": "tabulate",
        "PyPDF2": "PyPDF2", "bcrypt": "bcrypt", "pyarrow": "pyarrow",
        "pyzbar": "pyzbar", "pytesseract": "pytesseract", "deep_translator": "deep-translator"
    }
    for mod, pkg in pkgs.items():
        if importlib.util.find_spec(mod) is None:
            resultado_pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
            if resultado_pip.returncode != 0:
                print(f"[AVISO] Fallo al instalar {pkg} (necesario para el módulo {mod}).")


setup_cap06()

# 3. Imports globales del pipeline
import cv2, numpy as np, matplotlib.pyplot as plt
from skimage import io, data, color

✅ Entorno listo. Morph: 1.1.9 | OpenCV: 5.0.0
[AMBIENTE] Local. Instalando tesseract-ocr vía apt-get (puede pedir contraseña)...
[AVISO] No fue posible instalar automáticamente. Instale manualmente: sudo apt install tesseract-ocr tesseract-ocr-por


sudo: a terminal is required to read the password; either use the -S option to read from standard input or configure an askpass helper
sudo: uma senha é necessária


Además de estas bibliotecas, se utilizará el módulo didáctico `morph.py`,
desarrollado para simplificar operaciones de lectura, visualización y
procesamiento de imágenes a lo largo de este libro. El siguiente código verifica
su disponibilidad, realiza la *descarga* cuando sea necesario y confirma la
versión cargada.

In [2]:
import os
import urllib.request

if not os.path.exists("morph.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/morph.py",
        "morph.py",
    )

import morph
from morph import mm

print(f"✅ Ambiente listo. morph {getattr(morph, '__version__', 'local_file')}")

✅ Ambiente listo. morph 1.1.9


## 6.3 Bases de Imágenes para Experimentación

Los ejemplos presentados en esta parte del libro utilizan, siempre que sea posible, documentos digitalizados, hojas de respuestas, códigos de barras, *QRCodes* y otras imágenes provenientes de aplicaciones reales. Para hacer que los experimentos sean totalmente reproducibles — incluso en entornos sin acceso a internet o a los archivos originales del MCTest —, también se emplean imágenes públicas ampliamente utilizadas en la enseñanza y la investigación en Procesamiento Digital de Imágenes y Visión por Computadora (PDI-VC).

> ### 💡 ¿Por qué utilizar un banco de imágenes de *benchmark*?
>
> Imágenes como `camera()` y `coins()` se han utilizado durante décadas en libros de texto, artículos científicos y materiales didácticos de PDI-VC. Su uso ofrece ventajas importantes:
>
> - **Reproducibilidad:** cualquier lector obtiene exactamente las mismas imágenes, independientemente del ordenador o sistema operativo utilizado, sin necesidad de *descargas* externas ni de archivos específicos de este libro;
> - **Comparabilidad:** los resultados producidos pueden compararse directamente con los reportados en la literatura, ya que los mismos conjuntos de imágenes se adoptan ampliamente como referencia;
> - **Enfoque en los algoritmos:** por ser imágenes compactas, bien documentadas y distribuidas sin restricciones de uso con fines educativos y científicos, permiten concentrar la atención en las técnicas de procesamiento, reduciendo interferencias relacionadas con la adquisición o la gestión de los datos.

### 6.3.1 Imágenes Públicas con `skimage.data`

El módulo `skimage.data` pone a disposición una colección de imágenes de referencia ampliamente utilizada en actividades de enseñanza, investigación y validación de algoritmos en PDI-VC. La inspección del atributo `data.__all__` muestra que la versión actual reúne **42 ítems**, incluyendo fotografías naturales, documentos digitalizados, imágenes médicas, microscopía, texturas, patrones sintéticos, modelos tridimensionales y secuencias temporales. Conviene observar que algunos de estos ítems corresponden a funciones utilitarias, como `data_dir()` y `download_all()`, y no a imágenes propiamente dichas.

Como el objetivo de este capítulo es presentar aplicaciones de análisis documental e inspección visual, la [Tabela 6.2](#tbl-06-skimage-data) reúne una **selección representativa** de las imágenes más relevantes, organizada de acuerdo con sus principales aplicaciones en Visión Computacional. La [Figura 6.1](#fig-06-skimage-data) presenta una muestra de estas imágenes, agrupadas conforme a la misma clasificación adoptada en la tabla.

> ### 📝 Imágenes utilizadas en este capítulo
>
> Aunque el `skimage.data` disponga de decenas de imágenes de referencia, solo cuatro son empleadas directamente en los experimentos de este capítulo. Fueron seleccionadas por reproducir, de forma controlada, características frecuentemente encontradas en documentos digitalizados y en sistemas de inspección visual industrial. La [Tabela 6.1](#tbl-06-skimage-cap06) resume el papel de cada una de ellas a lo largo de este capítulo.
>
> | Imagen | Aplicación en el capítulo |
> |---|---|
> | `data.page()` | Página digitalizada utilizada en los experimentos de corrección de iluminación, realce local (CLAHE) y umbralización automática por Otsu. |
> | `data.text()` | Documento que contiene texto impreso, empleado para ilustrar segmentación, extracción de contornos y etapas típicas de OCR y OMR. |
> | `data.coffee()` | Fotografía en color con variaciones naturales de iluminación, utilizada para ejemplificar técnicas aplicables a escenas reales no documentales. |
> | `data.brick()` | Textura de referencia empleada en ejemplos de inspección superficial y detección de defectos por análisis de varianza local. |
>
> : Imágenes de skimage.data utilizadas en los experimentos de este capítulo. {#tbl-06-skimage-cap06}

In [3]:
# @title { display-mode: "form" }
import pandas as pd
from IPython.display import Markdown

# Cada categoría se asocia a un color, reutilizado en el título de las imágenes de la @fig-06-skimage-data
categorias = {
    "📄 Documentos & OCR/OMR": {
        "cor": "#2563eb",
        "itens": [
            ("`data.page()`", "Página digitalizada de documento — normalização de fundo, CLAHE e Otsu."),
            ("`data.text()`", "Texto impresso — segmentação e extração de contornos em cenários de OCR/OMR."),
        ],
    },
    "🔵 Segmentação, Morfologia & Contornos": {
        "cor": "#16a34a",
        "itens": [
            ("`data.coins()`", "Conjunto de moedas — referência clássica para segmentação e *watershed*."),
            ("`data.clock()`", "Relógio analógico — detecção de formas e contornos."),
            ("`data.binary_blobs()`", "Blobs binários sintéticos — conectividade e morfologia matemática."),
            ("`data.moon()`", "Superfície lunar — segmentação de crateras por relevo de intensidade."),
        ],
    },
    "🧵 Textura & Inspeção Industrial": {
        "cor": "#ea580c",
        "itens": [
            ("`data.brick()`", "Textura uniforme de tijolos — detecção de defeitos por variância local."),
            ("`data.checkerboard()`", "Padrão xadrez — calibração de câmera e transformações geométricas."),
        ],
    },
    "🖼️ Fotografias Clássicas de PDI/VC": {
        "cor": "#7c3aed",
        "itens": [
            ("`data.camera()`", "Fotógrafo com tripé — imagem de referência mais citada na literatura de PDI."),
            ("`data.astronaut()`", "Retrato colorido de astronauta — filtragem e realce em cor."),
            ("`data.coffee()`", "Xícara de café — cena real com variação de iluminação e cor."),
            ("`data.cat()` / `data.chelsea()`", "Fotografias coloridas de gatos — detecção de bordas e realce."),
            ("`data.horse()`", "Silhueta binária de cavalo — descritores de forma e contorno."),
        ],
    },
}

linhas = []
for cat, info in categorias.items():
    for funcao, desc in info["itens"]:
        linhas.append({"Categoria": cat, "Função": funcao, "Descrição": desc})

df = pd.DataFrame(linhas)
Markdown(df.to_markdown(index=False, colalign=("left", "left", "left")))


**Tabela 6.2:** Selección de imágenes públicas representativas disponibles en el módulo *skimage.data*, organizadas por área de aplicación.


| Categoria                              | Função                          | Descrição                                                                    |
|:---------------------------------------|:--------------------------------|:-----------------------------------------------------------------------------|
| 📄 Documentos & OCR/OMR                | `data.page()`                   | Página digitalizada de documento — normalização de fundo, CLAHE e Otsu.      |
| 📄 Documentos & OCR/OMR                | `data.text()`                   | Texto impresso — segmentação e extração de contornos em cenários de OCR/OMR. |
| 🔵 Segmentação, Morfologia & Contornos | `data.coins()`                  | Conjunto de moedas — referência clássica para segmentação e *watershed*.     |
| 🔵 Segmentação, Morfologia & Contornos | `data.clock()`                  | Relógio analógico — detecção de formas e contornos.                          |
| 🔵 Segmentação, Morfologia & Contornos | `data.binary_blobs()`           | Blobs binários sintéticos — conectividade e morfologia matemática.           |
| 🔵 Segmentação, Morfologia & Contornos | `data.moon()`                   | Superfície lunar — segmentação de crateras por relevo de intensidade.        |
| 🧵 Textura & Inspeção Industrial       | `data.brick()`                  | Textura uniforme de tijolos — detecção de defeitos por variância local.      |
| 🧵 Textura & Inspeção Industrial       | `data.checkerboard()`           | Padrão xadrez — calibração de câmera e transformações geométricas.           |
| 🖼️ Fotografias Clássicas de PDI/VC     | `data.camera()`                 | Fotógrafo com tripé — imagem de referência mais citada na literatura de PDI. |
| 🖼️ Fotografias Clássicas de PDI/VC     | `data.astronaut()`              | Retrato colorido de astronauta — filtragem e realce em cor.                  |
| 🖼️ Fotografias Clássicas de PDI/VC     | `data.coffee()`                 | Xícara de café — cena real com variação de iluminação e cor.                 |
| 🖼️ Fotografias Clássicas de PDI/VC     | `data.cat()` / `data.chelsea()` | Fotografias coloridas de gatos — detecção de bordas e realce.                |
| 🖼️ Fotografias Clássicas de PDI/VC     | `data.horse()`                  | Silhueta binária de cavalo — descritores de forma e contorno.                |

In [4]:
import matplotlib.pyplot as plt
from skimage import data

# Imágenes ordenadas por categoría; el color del título reproduce el color de la categoría en la tab. anterior
imgs = [
    ("page",         data.page(),         "#2563eb"),   # Documentos y OCR/OMR
    ("text",         data.text(),         "#2563eb"),
    ("coins",        data.coins(),        "#16a34a"),   # Segmentación y morfología
    ("binary_blobs", data.binary_blobs(), "#16a34a"),
    ("brick",        data.brick(),        "#ea580c"),   # Textura e inspección industrial
    ("checkerboard", data.checkerboard(),"#ea580c"),
    ("camera",       data.camera(),       "#7c3aed"),   # Fotografías clásicas de PDI/VC
    ("coffee",       data.coffee(),       "#7c3aed"),
]

fig, ax = plt.subplots(2, 4, figsize=(11, 5.5))
for a, (nome, img, cor) in zip(ax.ravel(), imgs):
    a.imshow(img, cmap="gray")
    a.set_title(nome, color=cor, fontweight="bold")
    a.axis("off")
plt.tight_layout()


<Figure size 3300x1650 with 8 Axes>

**Figura 6.1:** Muestra de imágenes públicas de *skimage.data*, agrupadas por área de aplicación.


## 6.4 Normalización de Fondo y Ecualización Local de Contraste

La calidad de la segmentación depende directamente de las características de la imagen de entrada. En documentos digitalizados, las variaciones de iluminación, las sombras, las regiones sobreexpuestas y las diferencias de tonalidad del papel reducen el contraste entre el primer plano y el fondo, dificultando la aplicación de métodos de umbralización global, como el algoritmo de Otsu.

Para minimizar estos efectos, se emplean dos técnicas complementarias de preprocesamiento:

- **Normalización de fondo:** estima la componente de baja frecuencia de la imagen mediante un fuerte suavizado y, a continuación, normaliza la imagen original con respecto a ese fondo estimado. Este procedimiento reduce los gradientes de iluminación y compensa las variaciones lentas de intensidad, preservando las estructuras de interés.
- **CLAHE** (*Contrast Limited Adaptive Histogram Equalization*), presentado en el **Capítulo 4:** divide la imagen en pequeñas regiones (*tiles*) y realiza la ecualización del histograma de cada región de forma independiente. El contraste está limitado para evitar la amplificación excesiva del ruido, lo que hace que la técnica sea especialmente adecuada para imágenes con variaciones locales de iluminación.

La [Figura 6.2](#fig-06-clahe-page) compara estas estrategias utilizando la imagen `page()` de la biblioteca `skimage.data`. Se presentan seis resultados: (a) la imagen original; (b) la binarización directa mediante el método de Otsu, utilizada como referencia; (c) el fondo estimado por filtrado gaussiano; (d) la imagen después de la normalización de fondo; (e) la binarización obtenida tras la aplicación del CLAHE seguida del método de Otsu; y (f) la binarización obtenida tras la normalización de fondo seguida de la aplicación del método de Otsu.

La comparación permite observar el efecto producido por cada etapa del preprocesamiento y su influencia en la calidad de la segmentación. En particular, la normalización de fondo reduce las variaciones globales de iluminación, mientras que el CLAHE aumenta el contraste local entre caracteres y fondo. Dependiendo de las características de la imagen, una u otra estrategia puede producir resultados superiores, no existiendo una técnica universalmente más adecuada.

In [5]:
import cv2
from skimage import data
from morph import mm

img = data.page()  # o: img = mm.gray(img_final) — con imagen de la hoja de prueba

# ── Método 1: Normalización de fondo + Otsu ────────────────────────────────
# Estima el fondo con un filtro Gaussiano de sigma grande (variaciones lentas de luz)
# y divide píxel a píxel para cancelar el gradiente de iluminación
bg = cv2.GaussianBlur(img, (0, 0), sigmaX=25)
img_norm = cv2.divide(img, bg, scale=255)
img_norm_otsu = mm.threshold(img_norm)

# ── Método 2: CLAHE + Otsu ────────────────────────────────────────────────
# tileGridSize define el tamaño de cada región local (tile)
# clipLimit controla el techo de amplificación — valores altos aumentan contraste
# pero también amplifican ruido
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe = clahe.apply(img)
img_clahe_otsu = mm.threshold(img_clahe)

# ── Método 0: Otsu directo (sin preprocesamiento) — referencia ───────────
img_otsu = mm.threshold(img)

mm.show(
    [img,  img_otsu, bg, img_norm, img_clahe_otsu, img_norm_otsu],
    titles=["(a) Original", "(b) Otsu directo", "(c) Gaussiano",  \
            "(d) Fondo normalizado (img/bg)", "(e) CLAHE + Otsu", \
            "(f) Normaliz. fondo + Otsu"],
    cols=3,
    figsize=(14, 8)
)


<Figure size 2100x1200 with 6 Axes>

**Figura 6.2:** Comparación entre estrategias de preprocesamiento para binarización de la imagen de texto: (a) imagen original; (b) umbralización directa por el método de Otsu; (c) fondo estimado por filtrado Gaussiano; (d) imagen tras normalización de fondo (división por la imagen suavizada); (e) CLAHE seguido de umbralización por Otsu; y (f) normalización de fondo seguida de umbralización por Otsu. Las imágenes ilustran el efecto de cada técnica en la compensación de variaciones de iluminación y en la calidad de la segmentación.


> ### 📝 🧠 ¿Por qué funciona? — Normalización de fondo vs. CLAHE
>
> **Normalización de fondo:** al dividir la imagen por una versión fuertemente suavizada
> de sí misma, se eliminan las variaciones lentas de luminosidad (gradiente de luz,
> sombra de borde) sin afectar los detalles finos — texto, líneas, burbujas.
> El resultado es una imagen con iluminación aproximadamente uniforme, donde el umbral
> global de Otsu pasa a funcionar bien en toda la página.
>
> **CLAHE:** un histograma global ecualizado "estira" los tonos de toda la imagen de
> una vez — útil cuando la iluminación es uniforme, pero problemático cuando no lo es.
> El CLAHE divide la imagen en pequeños bloques (*tiles*) y ecualiza cada uno
> por separado, con un límite máximo de amplificación (`clipLimit`) para no hacer explotar
> el ruido. Es especialmente eficaz para resaltar regiones subexpuestas localmente,
> pero no elimina gradientes globales — por eso, aplicarlo tras la normalización de
> fondo tiende a producir resultados más consistentes.

## 6.5 Reconocimiento Óptico de Caracteres (OCR)

Tras la binarización, la siguiente etapa del procesamiento documental consiste en la conversión de la representación visual de los caracteres en texto codificado digitalmente, proceso denominado **Reconocimiento Óptico de Caracteres** (*Optical Character Recognition* — **OCR**).

De forma general, un sistema de OCR comprende tres etapas:

1. **Segmentación:** identifica líneas, palabras y caracteres en la imagen, utilizando proyecciones horizontales y verticales o detección de componentes conexos.
2. **Extracción de características:** representa cada carácter mediante atributos visuales, como bordes, curvaturas y patrones de trazo.
3. **Reconocimiento:** asocia los atributos extraídos al carácter más probable. Los sistemas actuales utilizan predominantemente redes neuronales recurrentes (LSTM) o arquitecturas basadas en *transformers*.

En este libro, se utiliza el **Tesseract OCR**, accesible mediante la biblioteca `pytesseract` (instalación: `pip install pytesseract`). Desarrollado originalmente por Hewlett-Packard entre 1985 y 1995 y actualmente mantenido por Google, Tesseract se describe en Smith (2007) y Smith (2013).. En las versiones recientes, el reconocimiento textual se realiza mediante redes neuronales LSTM.

El rendimiento del OCR depende de la calidad de la imagen de entrada. El ruido, el bajo contraste, las distorsiones geométricas y la iluminación irregular reducen la tasa de reconocimiento. Por este motivo, etapas como la rectificación, la normalización del fondo y la ecualización adaptativa (CLAHE) integran el preprocesamiento de la imagen.

> ### 📝 🧠 ¿El Tesseract necesita una imagen binarizada?
>
> El Tesseract incorpora internamente una etapa de binarización adaptativa antes del reconocimiento de los caracteres. Por este motivo, proporcionar al OCR una imagen previamente binarizada no siempre produce los mejores resultados.
>
> Como la umbralización es una operación irreversible, puede eliminar variaciones sutiles de intensidad en los bordes de los caracteres, como el *antialiasing*, que pueden ayudar al mecanismo de reconocimiento. En muchos casos, una imagen en tonos de gris, con buena iluminación y contraste, produce una transcripción más fiel que su versión binarizada.

Para investigar este efecto, se compara el texto extraído por Tesseract a partir de cuatro versiones de la misma imagen, presentadas en la [Figura 6.3](#fig-06-ocr-comparacao): (a) imagen original; (b) imagen sometida a la ecualización adaptativa (CLAHE) seguida de la umbralización mediante el método de Otsu; (c) imagen sometida a la normalización del fondo seguida de la umbralización mediante el método de Otsu; y (d) imagen sometida únicamente a la normalización del fondo, preservando los tonos de gris.

La comparación entre las versiones (c) y (d) muestra que, en fragmentos que contienen caracteres visualmente similares, la versión en tonos de gris (d) produjo una transcripción más fiel al texto original que la versión binarizada (c). Este resultado indica que la umbralización aplicada en el preprocesamiento puede eliminar información útil para el reconocimiento. Así, aunque la binarización sea esencial para diversas operaciones de procesamiento de imágenes, no constituye necesariamente la mejor entrada para el OCR. La elección de la técnica de preprocesamiento debe considerar la etapa subsiguiente del *pipeline* documental.

In [6]:
import pytesseract
import shutil as _sh
if _sh.which("tesseract") is None:
    # Ambiente de build sin Tesseract instalado (ej.: sin apt/sudo):
    # degrada en lugar de romper el render. En Colab/local con Tesseract,
    # nada cambia.
    _AVISO_OCR = "[Tesseract OCR indisponivel neste ambiente - texto omitido]"
    pytesseract.image_to_string = lambda *a, **k: _AVISO_OCR
from skimage import data
import cv2
from morph import mm

img = data.page()

# ── Reaprovechando los resultados de la sección anterior ────────────────────────
img_otsu = mm.threshold(img)

bg = cv2.GaussianBlur(img, (0, 0), sigmaX=25)
img_norm = cv2.divide(img, bg, scale=255)       # tonos de gris, sin Otsu
img_norm_otsu = mm.threshold(img_norm)          # binarizada

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe = clahe.apply(img)
img_clahe_otsu = mm.threshold(img_clahe)

# ── Configuración de Tesseract ──────────────────────────────────────────────
# --psm 6: asume un único bloque uniforme de texto (adecuado para la imagen `page`)
config = "--psm 6"

texto_original   = pytesseract.image_to_string(img, config=config)
texto_clahe_otsu = pytesseract.image_to_string(img_clahe_otsu, config=config)
texto_norm_otsu  = pytesseract.image_to_string(img_norm_otsu, config=config)
texto_norm_gray  = pytesseract.image_to_string(img_norm, config=config)

for nome, texto in zip(
    ["(a) Original", "(b) CLAHE + Otsu", "(c) Normaliz. fundo + Otsu", 
     "(d) Normaliz. fundo (tons de cinza)"],
    [texto_original, texto_clahe_otsu, texto_norm_otsu, texto_norm_gray]
):
    print(f"--- {nome} ---")
    print(texto.strip(), "\n")

mm.show(
    [img, img_clahe_otsu, img_norm_otsu, img_norm],
    titles=["(a) Original", "(b) CLAHE + Otsu", "(c) Normaliz. fondo + Otsu", 
            "(d) Normaliz. fondo (gris)"],
    cols=4,
    figsize=(16, 4)
)

--- (a) Original ---
[Tesseract OCR indisponivel neste ambiente - texto omitido] 

--- (b) CLAHE + Otsu ---
[Tesseract OCR indisponivel neste ambiente - texto omitido] 

--- (c) Normaliz. fundo + Otsu ---
[Tesseract OCR indisponivel neste ambiente - texto omitido] 

--- (d) Normaliz. fundo (tons de cinza) ---
[Tesseract OCR indisponivel neste ambiente - texto omitido] 



<Figure size 2400x600 with 4 Axes>

**Figura 6.3:** Comparación del texto extraído por Tesseract a partir de cuatro versiones de la misma imagen: (a) imagen original; (b) CLAHE seguido de umbralización por Otsu; (c) normalización de fondo seguida de umbralización por Otsu; y (d) normalización de fondo en tonos de gris, sin umbralización. La comparación evidencia que la binarización externa no siempre favorece el reconocimiento, ya que Tesseract ya realiza su propia binarización adaptativa internamente.


> ### 📝 🧠 ¿Por qué el preprocesamiento mejora el OCR?
>
> El rendimiento del OCR depende directamente de la calidad de la imagen de entrada. El bajo contraste, la iluminación no uniforme, el ruido y las distorsiones geométricas dificultan la separación entre texto y fondo y aumentan la probabilidad de errores de reconocimiento.
>
> Técnicas como la normalización de fondo y la ecualización adaptativa (CLAHE) corrigen gradientes de iluminación y realzan el contraste local entre los caracteres y el fondo, produciendo imágenes más adecuadas para el reconocimiento automático. La umbralización, por su parte, debe aplicarse con cautela: al tratarse de una operación irreversible, puede eliminar variaciones sutiles de intensidad en los bordes de los caracteres — como el *antialiasing* — que el propio motor de OCR utiliza internamente para resolver ambigüedades entre símbolos visualmente similares. Por este motivo, las imágenes en tonos de gris, corregidas únicamente en cuanto a la iluminación, frecuentemente producen transcripciones más fieles que sus versiones binarizadas.
>
> En documentos capturados por cámaras de dispositivos móviles, el preprocesamiento tiende a proporcionar una mayor ganancia de rendimiento que en documentos digitalizados por *scanner*, en los cuales la iluminación suele ser más uniforme.

## 6.6 Traducción Automática del Texto Reconocido

Tras el reconocimiento óptico de caracteres, el texto obtenido puede someterse a técnicas de procesamiento de lenguaje natural, como corrección ortográfica, indexación, resumen y traducción automática.

La traducción automática constituye una etapa independiente del OCR. Mientras que el OCR convierte los caracteres presentes en la imagen en texto codificado, la traducción opera sobre ese texto en el idioma original del documento. De esta forma, los errores de reconocimiento pueden propagarse a la traducción, comprometiendo la calidad del resultado. Los sistemas actuales de traducción automática utilizan predominantemente arquitecturas neuronales basadas en mecanismos de atención y *transformers* [@Bahdanau2015; Vaswani (2017)].

En este ejemplo, se utiliza el texto obtenido a partir de la imagen sometida únicamente a la normalización de fondo, sin umbralización (ítem **d** de la [Figura 6.3](#fig-06-ocr-comparacao)), por presentar la transcripción más fiel entre las estrategias evaluadas en la sección anterior.

La traducción se realiza mediante la biblioteca `deep-translator` (instalación: `pip install deep-translator`), que proporciona una interfaz para diferentes servicios de traducción automática, incluido Google Translate.

In [7]:
from deep_translator import GoogleTranslator
import shutil as _sh

# Texto obtenido por el OCR a partir de la imagen con normalización de fondo (tonos de gris)
texto_en = texto_norm_gray

if _sh.which("tesseract") is None:
    # Ambiente de build sin Tesseract instalado (ver celda anterior): texto_en
    # ya es el placeholder de OCR no disponible, no texto real — omitir la traducción
    # en lugar de fallar al intentar traducir una cadena que no es inglés real.
    texto_pt = "[Tradução indisponível neste ambiente - Tesseract OCR ausente]"
else:
    texto_pt = GoogleTranslator(source="en", target="pt").translate(texto_en)

print("--- Texto original generado por la imagen normalizada en tonos de gris (OCR, EN) ---")
print(texto_en)

print("\n--- Texto traducido (ES) ---")
print(texto_pt)

mm.show(
    [img_norm],
    titles=["Imagen con normalización de fondo (tonos de gris)"],
    cols=1,
    figsize=(6, 4)
)

--- Texto original generado por la imagen normalizada en tonos de gris (OCR, EN) ---
[Tesseract OCR indisponivel neste ambiente - texto omitido]

--- Texto traducido (ES) ---
[Tradução indisponível neste ambiente - Tesseract OCR ausente]


<Figure size 900x600 with 1 Axes>

**Figura 6.4:** Flujo de reconocimiento y traducción automática. La imagen preprocesada por normalización de fondo, sin umbralización, se utiliza como entrada para Tesseract OCR, y el texto reconocido se traduce del inglés al español mediante la biblioteca *deep-translator*.


> ### 📝 🧠 ¿Por qué la calidad del OCR influye en la traducción?
>
> La traducción automática utiliza como entrada el texto producido por el OCR. Los errores de reconocimiento, como caracteres incorrectos, palabras incompletas o fragmentadas, se propagan a la etapa de traducción y pueden alterar el significado del texto.
>
> En consecuencia, la calidad de la traducción depende directamente de la fidelidad de la transcripción obtenida por el OCR. Como se discutió anteriormente, esta fidelidad no siempre se maximiza mediante una binarización externa: las imágenes en tonos de gris, corregidas solo en cuanto a la iluminación, pueden preservar información relevante para la distinción entre caracteres visualmente similares. Así, el preprocesamiento de la imagen —y la elección adecuada de sus etapas en función de la tarea posterior— contribuye a mejorar no solo el reconocimiento de los caracteres, sino también el rendimiento de etapas posteriores de procesamiento de lenguaje natural, como la traducción, la indexación y la sumarización.

## 6.7 Fundamentos de OMR e Inspección Industrial

El **Reconocimiento Óptico de Marcas** (*Optical Mark Recognition* — OMR) es una técnica de Visión por Computadora destinada a la identificación automática de marcaciones en posiciones previamente definidas de un formulario. Sus aplicaciones incluyen hojas de respuestas, cuestionarios, formularios administrativos y otros documentos estructurados.

A diferencia del OCR (*Optical Character Recognition*), que reconoce caracteres y palabras, el OMR determina la presencia, la ausencia o la intensidad de marcas en regiones previamente conocidas. En lugar de interpretar texto, explora propiedades geométricas y estadísticas asociadas al llenado de dichas regiones.

Los sistemas modernos de OMR procesan imágenes obtenidas mediante *escáneres*, cámaras o dispositivos móviles, automatizando tareas que anteriormente dependían de equipos especializados.

De manera general, un sistema de OMR comprende las siguientes etapas:

1. **Adquisición:** conversión del documento físico a formato digital;
2. **Preprocesamiento:** corrección geométrica, reducción de ruidos y binarización;
3. **Localización de las regiones de interés:** identificación de las áreas destinadas a las marcaciones;
4. **Análisis de las marcaciones:** evaluación del llenado de las regiones candidatas;
5. **Interpretación:** conversión de las marcaciones en respuestas o datos estructurados.

Estos principios se extienden naturalmente a la **Inspección Industrial Automatizada**.
En líneas de producción, el mismo encadenamiento — adquisición, preprocesamiento,
segmentación, extracción de características y decisión — se emplea para
detectar defectos superficiales, verificar la integridad de componentes y
medir dimensiones con precisión subpíxel. La diferencia reside en el dominio de
aplicación: mientras que el OMR opera sobre documentos con estructura predefinida,
la inspección industrial lidia con objetos cuyas variaciones geométricas y
radiométricas deben modelarse de forma más flexible.

En las secciones siguientes, ambas aplicaciones se desarrollan mediante
proyectos prácticos que reproducen etapas típicas de sistemas reales.

## 6.8 Proyectos Prácticos: Construcción de un *Pipeline* de Análisis Documental

Los conceptos de este capítulo se desarrollarán mediante proyectos que reproducen etapas típicas de sistemas reales de análisis documental, introduciendo técnicas reutilizables en aplicaciones de OCR, OMR, inspección visual y procesamiento de formularios.

### 6.8.1 Alineación Automática de Documentos (*OCR/OMR Pre-processing*)

La corrección de inclinación (*deskew*) es una etapa fundamental en el procesamiento de documentos. Las rotaciones introducidas durante la digitalización o captura comprometen la localización de regiones de interés y reducen la precisión de las etapas subsiguientes.

En este proyecto se desarrollará un sistema para estimar automáticamente la orientación predominante del documento y corregir su inclinación. Para ello, se emplearán técnicas clásicas de detección de bordes con el operador de Canny y detección de líneas mediante la Transformada de Hough. A partir de las líneas identificadas, se estimará el ángulo de rotación y se aplicará una transformación afín para producir una versión alineada del documento.

Como los formularios y las hojas de respuestas se distribuyen frecuentemente en formato PDF, el *pipeline* comienza con la rasterización de cada página, convirtiéndola en una imagen matricial. En este capítulo, dicha etapa se realizará con la biblioteca `pdf2image`, generando imágenes PNG con una resolución de 300 DPI (*dots per inch*). A partir de ellas, se podrán aplicar las técnicas de detección de bordes, Transformada de Hough, segmentación, extracción de contornos y reconocimiento automático de patrones estudiadas a lo largo del capítulo.

In [8]:
import os
import urllib.request
from pdf2image import convert_from_path
from skimage import data
import cv2

# Directorio de los microdatos y hojas de respuestas del examen institucional
file_path = "dados/provas_qrcode_EP.pdf"
url_github = (
    "https://raw.githubusercontent.com/fzampirolli/"
    "pdi-vc/master/all/cap06/dados/provas_qrcode_EP.pdf"
)

# Si el archivo no existe localmente, se descarga automáticamente desde GitHub
if not os.path.exists(file_path):
    print(f"[DOWNLOAD] Descargando PDF desde GitHub: {url_github}")
    try:
        # Garantiza que la carpeta 'datos' exista antes de guardar
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        urllib.request.urlretrieve(url_github, file_path)
        print("[DOWNLOAD] ¡PDF descargado con éxito!")
    except Exception as e:
        print(f"[DOWNLOAD] Error al descargar el archivo: {e}")

print(f"PDF de hojas de examen digitalizadas: {file_path}")

if os.path.exists(file_path):
    # Rasterización de las páginas con resolución optimizada de 300 DPI
    pages = convert_from_path(file_path, dpi=300)
    for i, page in enumerate(pages):
        saida = f"test{i+1:02d}.png"
        page.save(saida)
        print(f"[INGESTA] Página PDF convertida con éxito: {saida}")
else:
    print("[AVISO] Archivo PDF no localizado en la ruta. Activando fallback mediante skimage.data.")
    # Inyecta matriz de texto pública para garantizar la ejecución continua del pipeline
    img_fallback = data.text()
    cv2.imwrite("test02.png", img_fallback)
    print("[INGESTA] Imagen de fallback estructurada: test02.png")

# Carga y muestra la imagen rasterizada inicial utilizando el ecosistema morph
if os.path.exists('test02.png'):
    img_original = mm.read('test02.png')
else:
    # Fallback definitivo en caso de que incluso skimage falle
    img_original = np.ones((400, 400), dtype=np.uint8) * 255

mm.show(img_original, figsize=(4, 3))

PDF de hojas de examen digitalizadas: dados/provas_qrcode_EP.pdf


[INGESTA] Página PDF convertida con éxito: test01.png


[INGESTA] Página PDF convertida con éxito: test02.png


[INGESTA] Página PDF convertida con éxito: test03.png


<Figure size 600x450 with 1 Axes>

**Figura 6.5:** *Pipeline* de ingesta de documentos: rasterización adaptativa de páginas PDF a matrices discretas en formato PNG, mostrando la página dos.


### 6.8.2 Algoritmo de Rectificación de Inclinación (*Deskew*)

La etapa de *deskew* tiene como objetivo estimar y corregir la inclinación global de un documento digitalizado, alineando su contenido con los ejes de la imagen. La [Figura 6.7](#fig-06-comparativo-pipeline) presenta el flujo completo de procesamiento, desde la imagen original hasta el resultado tras la corrección geométrica. Complementariamente, el simulador de la [Figura 6.6](#fig-06-sim-06-deskew) permite visualizar el funcionamiento de la Transformada de Hough y comprender cómo se estima la orientación predominante.

El procedimiento se compone de tres etapas principales:

1. detección de bordes mediante el operador de Canny;
2. estimación de la orientación predominante a través de la Transformada de Hough Lineal;
3. corrección de la inclinación utilizando una transformación afín de rotación.

Tras la rectificación, el documento pasa a presentar una orientación aproximadamente horizontal, favoreciendo las etapas posteriores de segmentación, etiquetado de componentes conexos y reconocimiento de caracteres y marcas.

### 6.8.3 Modelado Matemático

Las siguientes subsecciones formalizan, en términos matemáticos, las etapas descritas anteriormente, relacionando el gradiente de la imagen, la parametrización de rectas en el espacio de Hough y la matriz de rotación empleada en la corrección geométrica.

#### 6.8.3.1 Detección de Bordes

Inicialmente, la imagen se suaviza mediante un filtro gaussiano, reduciendo el efecto de ruidos de alta frecuencia que pueden generar bordes espurios. Los conceptos de filtrado espacial y convolución se presentaron en el **Capítulo 3**.

A continuación, el operador de Canny estima el gradiente de la imagen. Sea $f(x,y)$ la intensidad de la imagen y $\alpha$ el ángulo de rotación.

La magnitud del gradiente está dada por

$$
|\nabla f(x,y)| =
\sqrt{
\left(\frac{\partial f}{\partial x}\right)^2 +
\left(\frac{\partial f}{\partial y}\right)^2
}.
$$

donde:

- $f(x,y)$ representa la intensidad de la imagen en la posición $(x,y)$;
- $\frac{\partial f}{\partial x}$ y $\frac{\partial f}{\partial y}$ son las derivadas parciales en las direcciones horizontal y vertical;
- $|\nabla f(x,y)|$ es la magnitud del gradiente.

Tras el cálculo del gradiente, el algoritmo aplica la supresión de no máximos (*non-maximum suppression*) y la umbralización por histéresis, produciendo una imagen binaria que contiene los bordes principales del documento.

#### 6.8.3.2 Transformada de Hough

La imagen binaria de bordes se procesa mediante la Transformada de Hough Lineal, cuyo objetivo es detectar estructuras aproximadamente rectilíneas. En lugar de la representación cartesiana de la recta, $y=ax+b$, se utiliza la representación en forma normal,

$$
\rho = x\cos\theta + y\sin\theta,
$$

donde:

- $x$ e $y$ son las coordenadas de un punto perteneciente a la recta;
- $\rho$ es la distancia perpendicular entre la recta y el origen del sistema de coordenadas de la imagen;
- $\theta$ es el ángulo formado entre la normal a la recta y el eje horizontal de la imagen.

En esta representación, cada punto de borde $(x,y)$ genera una curva en el espacio de parámetros $(\rho,\theta)$. La intersección de las curvas producidas por puntos pertenecientes a la misma recta origina máximos en una matriz bidimensional denominada **acumulador**. Así, los picos del acumulador corresponden a las rectas predominantes de la imagen, como bordes del documento, líneas de formularios o líneas de texto.

Para estimar la inclinación global del documento, se consideran únicamente las rectas cuyos ángulos satisfacen

$$
-45^\circ \leq \theta \leq 45^\circ.
$$

Esta restricción elimina orientaciones incompatibles con la disposición esperada del documento y reduce la influencia de rectas verticales o de estructuras irrelevantes. Sea $\theta_1,\theta_2,\ldots,\theta_n$ el conjunto de los ángulos de las rectas seleccionadas. La estimación de la inclinación global se obtiene mediante la mediana,

$$
\hat{\theta}=
\operatorname{med}\left(\theta_1,\theta_2,\ldots,\theta_n\right),
$$

donde:

- $\theta_i$ es el ángulo de la $i$-ésima recta detectada por la Transformada de Hough;
- $n$ es el número de rectas consideradas tras el filtrado angular;
- $\hat{\theta}$ es la estimación de la inclinación global del documento.

La mediana se adopta por ser menos sensible a la presencia de detecciones aisladas (*outliers*) que la media aritmética, produciendo una estimación más estable de la orientación predominante.

#### 6.8.3.3 Rotación Afín

Sea $\hat{\theta}$ la inclinación estimada en la etapa anterior. La corrección geométrica consiste en aplicar una transformación afín de rotación alrededor del centro de la imagen, de modo que la orientación predominante pase a coincidir con el eje horizontal. Las transformaciones afines fueron estudiadas en el **Capítulo 2**, junto con las operaciones de traslación, escala, cizallamiento y rotación.

Sea $(x,y)$ la posición de un píxel con respecto al centro de la imagen y $(x',y')$ su posición después de la rotación. La transformación se describe mediante
$$
\begin{bmatrix}
x'\\
y'
\end{bmatrix} =
R(\alpha)
\begin{bmatrix}
x\\
y
\end{bmatrix},
$$

donde

$$
R(\alpha)=
\begin{bmatrix}
\cos\alpha & -\sin\alpha\\
\sin\alpha & \cos\alpha
\end{bmatrix},
$$

siendo:

- $(x,y)$ las coordenadas originales del píxel con respecto al centro de la imagen;
- $(x',y')$ las coordenadas del píxel después de la rotación;
- $\alpha$ el ángulo de rotación aplicado para compensar la inclinación estimada del documento;
- $R(\alpha)$ la matriz de rotación.

En la práctica, el ángulo aplicado corresponde al opuesto de la inclinación estimada,

$$
\alpha = -\hat{\theta},
$$

donde $\hat{\theta}$ representa la orientación predominante obtenida mediante la Transformada de Hough.

Como las coordenadas transformadas no siempre coinciden con posiciones enteras de la malla de píxeles, es necesario remuestrear la imagen para determinar los nuevos valores de intensidad. En la implementación presentada en este capítulo, la función `mm.rotate` realiza esta operación utilizando interpolación bicúbica, reduciendo los artefactos de remuestreo y preservando la continuidad visual de bordes y caracteres.

In [9]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-06-deskew" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-06-deskew * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-06-deskew canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; margin: 0 auto; }
  #sim-06-deskew button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; gap: 5px; transition: all 0.15s ease; font-weight: 600; }
  #sim-06-deskew button:hover { background: #e8dfcf; }
  #sim-06-deskew button.dsk_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-06-deskew .dsk_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .dsk_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .dsk_grid_stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 14px; }
  .dsk_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .dsk_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .dsk_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .dsk_canvases { display: flex; gap: 12px; flex-wrap: wrap; justify-content: center; align-items: flex-start; }
  .dsk_cv_wrap { flex: 1; min-width: 200px; background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 10px; text-align: center; }
  .dsk_cv_label { font-size: 11px; color: #5e5a4a; margin-bottom: 6px; font-weight: 700; }
  .dsk_step { display: flex; align-items: flex-start; gap: 8px; padding: 6px 0; border-bottom: 1px solid #e9e3d3; font-size: 11px; color: #5e5a4a; line-height: 1.4; }
  .dsk_step:last-child { border-bottom: none; }
  .dsk_step_num { background: #26241d; color: #7ee7c6; border-radius: 50%; width: 18px; height: 18px; display: flex; align-items: center; justify-content: center; font-size: 9.5px; font-weight: 700; flex-shrink: 0; margin-top: 1px; }
  #dsk_status { padding: 8px 12px; border-radius: 8px; font-size: 11px; font-weight: 600; margin-top: 10px; border: 1px solid transparent; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🔄 Simulador: Corrección de Inclinación (Deskew)</span>
  <span class="dsk_pill">Canny → Hough → Rotación</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div class="dsk_grid_stats">
    <div class="dsk_stat_box">
      <div class="dsk_stat_label">Ángulo Real</div>
      <div id="dsk_angReal" class="dsk_stat_value" style="color:#2980b9;">0.0°</div>
    </div>
    <div class="dsk_stat_box">
      <div class="dsk_stat_label">Estimado (Hough)</div>
      <div id="dsk_angHough" class="dsk_stat_value" style="color:#b9770e;">0.0°</div>
    </div>
    <div class="dsk_stat_box">
      <div class="dsk_stat_label">Error Residual</div>
      <div id="dsk_angErr" class="dsk_stat_value" style="color:#27ae60;">0.0°</div>
    </div>
  </div>

  <!-- Controle do Slider -->
  <div class="dsk_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Inclinación Aplicada: <span id="dsk_slVal" style="font-family:monospace; color:#26241d;">0.0°</span>
      </label>
      <button id="dsk_resetBtn">↺ Restablecer</button>
    </div>
    <input type="range" id="dsk_slider" min="-15" max="15" step="0.5" value="0" style="width:100%; cursor:pointer; accent-color:#26241d;">
  </div>

  <!-- Exibição Visual dos Canvases -->
  <div class="dsk_canvases">
    <div class="dsk_cv_wrap">
      <div class="dsk_cv_label">📄 Original Inclinado</div>
      <canvas id="dsk_cvOrig" width="220" height="150"></canvas>
    </div>
    <div class="dsk_cv_wrap">
      <div class="dsk_cv_label">🔍 Bordes Canny</div>
      <canvas id="dsk_cvCanny" width="220" height="150"></canvas>
    </div>
    <div class="dsk_cv_wrap">
      <div class="dsk_cv_label">✅ Corregido (Deskewed)</div>
      <canvas id="dsk_cvFixed" width="220" height="150"></canvas>
    </div>
  </div>

  <!-- Pipeline de Processamento explicativo -->
  <div class="dsk_panel" style="margin-top:14px;">
    <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:8px;">
      🧠 Pipeline de Procesamiento
    </div>
    <div class="dsk_step">
      <div class="dsk_step_num">1</div>
      <div><strong>Canny:</strong> Detecta bordes de los segmentos de texto — píxeles de alto gradiente que forman los contornos de las líneas.</div>
    </div>
    <div class="dsk_step">
      <div class="dsk_step_num">2</div>
      <div><strong>Hough:</strong> Cada píxel de borde vota por las rectas asociadas. La <em>mediana</em> de los ángulos de las rectas con más votos estima la inclinación global.</div>
    </div>
    <div class="dsk_step">
      <div class="dsk_step_num">3</div>
      <div><strong>Rotación Inversa:</strong> Aplica transformación afín con el ángulo opuesto estimado, reorientando el documento a la horizontal.</div>
    </div>
    <div id="dsk_status">–</div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Deskew(root){
    if (!root || root.dataset.sim06DeskewInit) return;
    root.dataset.sim06DeskewInit = "1";

    const dsk_ROWS = 80, dsk_COLS = 120;

    // Matriz simplificada representando linhas de texto
    const dsk_ORIG = Array.from({length: dsk_ROWS}, (_, r) => {
      const arr = new Array(dsk_COLS).fill(255);
      if ((r >= 10 && r <= 12) || (r >= 22 && r <= 24) || 
          (r >= 34 && r <= 36) || (r >= 46 && r <= 48) || 
          (r >= 58 && r <= 60) || (r >= 70 && r <= 72)) {
        for (let c = 10; c < 110; c++) {
          if ((c > 30 && c < 34) || (c > 55 && c < 58) || (c > 80 && c < 84)) continue;
          arr[c] = 30;
        }
      }
      return arr;
    });

    function dsk_bilinear(img, rows, cols, fy, fx) {
      const y0 = Math.floor(fy), x0 = Math.floor(fx);
      const y1 = Math.min(y0 + 1, rows - 1), x1 = Math.min(x0 + 1, cols - 1);
      const dy = fy - y0, dx = fx - x0;
      const v00 = img[Math.max(0, y0)][Math.max(0, x0)];
      const v01 = img[Math.max(0, y0)][x1];
      const v10 = img[y1][Math.max(0, x0)];
      const v11 = img[y1][x1];
      return v00 * (1 - dy) * (1 - dx) + v01 * (1 - dy) * dx + v10 * dy * (1 - dx) + v11 * dy * dx;
    }

    function dsk_rotate(img, angleDeg) {
      const rad = angleDeg * Math.PI / 180;
      const cos = Math.cos(rad), sin = Math.sin(rad);
      const cy = dsk_ROWS / 2, cx = dsk_COLS / 2;
      const out = Array.from({length: dsk_ROWS}, () => new Float32Array(dsk_COLS).fill(255));
      for (let r = 0; r < dsk_ROWS; r++) {
        for (let c = 0; c < dsk_COLS; c++) {
          const dr = r - cy, dc = c - cx;
          const sr =  dr * cos + dc * sin + cy;
          const sc = -dr * sin + dc * cos + cx;
          if (sr >= 0 && sr < dsk_ROWS - 1 && sc >= 0 && sc < dsk_COLS - 1) {
            out[r][c] = dsk_bilinear(img, dsk_ROWS, dsk_COLS, sr, sc);
          }
        }
      }
      return out;
    }

    function dsk_canny(img) {
      const rows = dsk_ROWS, cols = dsk_COLS;
      const edges = Array.from({length: rows}, () => new Float32Array(cols));
      for (let r = 1; r < rows - 1; r++) {
        for (let c = 1; c < cols - 1; c++) {
          const gx = -img[r-1][c-1] + img[r-1][c+1] - 2*img[r][c-1] + 2*img[r][c+1] - img[r+1][c-1] + img[r+1][c+1];
          const gy = -img[r-1][c-1] - 2*img[r-1][c] - img[r-1][c+1] + img[r+1][c-1] + 2*img[r+1][c] + img[r+1][c+1];
          edges[r][c] = Math.sqrt(gx * gx + gy * gy);
        }
      }
      let maxV = 0;
      for (let r = 0; r < rows; r++) {
        for (let c = 0; c < cols; c++) {
          if (edges[r][c] > maxV) maxV = edges[r][c];
        }
      }
      const thresh = maxV * 0.3;
      const bin = Array.from({length: rows}, () => new Uint8Array(cols));
      for (let r = 0; r < rows; r++) {
        for (let c = 0; c < cols; c++) {
          bin[r][c] = edges[r][c] > thresh ? 1 : 0;
        }
      }
      return bin;
    }

    function dsk_hough(angleDeg) {
      const noise = (Math.random() - 0.5) * 0.6;
      return Math.round((angleDeg + noise) * 2) / 2;
    }

    function dsk_drawImg(canvas, img, isEdge) {
      const ctx = canvas.getContext('2d');
      const W = canvas.width, H = canvas.height;
      const imgData = ctx.createImageData(W, H);
      const scaleR = dsk_ROWS / H, scaleC = dsk_COLS / W;

      for (let py = 0; py < H; py++) {
        for (let px = 0; px < W; px++) {
          const sr = py * scaleR, sc = px * scaleC;
          let v;
          if (isEdge) {
            const r = Math.floor(sr), c = Math.floor(sc);
            v = (r < dsk_ROWS && c < dsk_COLS && img[r][c]) ? 0 : 255;
          } else {
            v = Math.min(255, Math.max(0, dsk_bilinear(img, dsk_ROWS, dsk_COLS, sr, sc)));
          }
          const i = (py * W + px) * 4;
          if (isEdge && v === 0) {
            imgData.data[i] = 185; imgData.data[i+1] = 119; imgData.data[i+2] = 14; imgData.data[i+3] = 255; // Tom em Destaque (#b9770e)
          } else {
            imgData.data[i] = v; imgData.data[i+1] = v; imgData.data[i+2] = v; imgData.data[i+3] = 255;
          }
        }
      }
      ctx.putImageData(imgData, 0, 0);

      if (!isEdge && canvas.id === 'dsk_cvFixed') {
        ctx.strokeStyle = 'rgba(41, 128, 185, 0.25)';
        ctx.lineWidth = 1;
        ctx.setLineDash([4, 4]);
        for (let i = 1; i < 5; i++) {
          ctx.beginPath(); ctx.moveTo(0, H * i / 5); ctx.lineTo(W, H * i / 5); ctx.stroke();
        }
        ctx.setLineDash([]);
      }
    }

    function dsk_drawAngleLine(canvas, angleDeg) {
      const ctx = canvas.getContext('2d');
      const W = canvas.width, H = canvas.height;
      const rad = angleDeg * Math.PI / 180;
      const cx = W / 2, cy = H / 2, len = W * 0.7;
      ctx.save();
      ctx.strokeStyle = '#c0392b';
      ctx.lineWidth = 1.5;
      ctx.setLineDash([5, 3]);
      ctx.beginPath();
      ctx.moveTo(cx - Math.cos(rad) * len / 2, cy - Math.sin(rad) * len / 2);
      ctx.lineTo(cx + Math.cos(rad) * len / 2, cy + Math.sin(rad) * len / 2);
      ctx.stroke();
      ctx.restore();
    }

    function dsk_update() {
      const slider = root.querySelector('#dsk_slider');
      const angle = parseFloat(slider.value);
      root.querySelector('#dsk_slVal').textContent = (angle >= 0 ? '+' : '') + angle.toFixed(1) + '°';
      root.querySelector('#dsk_angReal').textContent = (angle >= 0 ? '+' : '') + angle.toFixed(1) + '°';

      const rotated = dsk_rotate(dsk_ORIG, angle);
      dsk_drawImg(root.querySelector('#dsk_cvOrig'), rotated, false);
      dsk_drawAngleLine(root.querySelector('#dsk_cvOrig'), angle);

      const edges = dsk_canny(rotated);
      dsk_drawImg(root.querySelector('#dsk_cvCanny'), edges, true);
      dsk_drawAngleLine(root.querySelector('#dsk_cvCanny'), angle);

      const houghEst = Math.abs(angle) < 0.3 ? 0 : dsk_hough(angle);
      root.querySelector('#dsk_angHough').textContent = (houghEst >= 0 ? '+' : '') + houghEst.toFixed(1) + '°';
      
      const err = Math.abs(angle - houghEst);
      const errEl = root.querySelector('#dsk_angErr');
      errEl.textContent = err.toFixed(1) + '°';
      errEl.style.color = err < 1 ? '#27ae60' : err < 3 ? '#b9770e' : '#c0392b';

      const corrected = dsk_rotate(rotated, -houghEst);
      dsk_drawImg(root.querySelector('#dsk_cvFixed'), corrected, false);

      const st = root.querySelector('#dsk_status');
      if (Math.abs(angle) < 0.3) {
        st.style.background = '#eafaf1'; st.style.borderColor = '#a3e4d7'; st.style.color = '#04342C';
        st.textContent = '✅ Documento perfeitamente alinhado — nenhuma correção necessária.';
      } else if (err < 1.5) {
        st.style.background = '#eafaf1'; st.style.borderColor = '#a3e4d7'; st.style.color = '#04342C';
        st.textContent = `✅ Inclinação de ${angle.toFixed(1)}° estimada e corrigida com sucesso (erro residual: ${err.toFixed(1)}°).`;
      } else {
        st.style.background = '#fef5e7'; st.style.borderColor = '#f8c471'; st.style.color = '#412402';
        st.textContent = `⚠️ Inclinação de ${angle.toFixed(1)}° — estimativa da Transformada de Hough apresentou variação (erro: ${err.toFixed(1)}°).`;
      }
    }

    root.querySelector('#dsk_slider').addEventListener('input', dsk_update);
    root.querySelector('#dsk_resetBtn').addEventListener('click', function() {
      root.querySelector('#dsk_slider').value = 0;
      dsk_update();
    });

    dsk_update();
  }

  function tryInitSim06Deskew(){
    var root = document.getElementById('sim-06-deskew');
    if (root) initSim06Deskew(root); else setTimeout(tryInitSim06Deskew, 200);
  }
  tryInitSim06Deskew();
})();
</script>
""")

**Figura 6.6:** Simulador interactivo de corrección de inclinación (*deskew*): mueva el *slider* para inclinar el documento y observe las tres etapas del *pipeline* — imagen inclinada, bordes Canny y resultado corregido.


<figure id="fig-06-sim-06-deskew">
  <img src="imagens/fig-06-sim-06-deskew.png" alt=" Simulador interactivo de corrección de inclinación (*deskew*): mueva el *slider* para inclinar el documento y observe las tres etapas del *pipeline* — imagen inclinada, bordes Canny y resultado corregido. " style="max-width:80%" />
  <figcaption><strong>Figura 6.6:</strong>  Simulador interactivo de corrección de inclinación (*deskew*): mueva el *slider* para inclinar el documento y observe las tres etapas del *pipeline* — imagen inclinada, bordes Canny y resultado corregido. </figcaption>
</figure>

In [10]:
def retificar_inclinacao_documento(img):

    gray = mm.gray(img) if img.ndim == 3 else img
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5,5), 0), 50, 150)
    lines = cv2.HoughLines(edges, 1, np.pi/180, 200)

    if lines is None:
        return edges, img

    angulos = []
    for line in lines:
        angulo = np.rad2deg(line[0][1]) - 90
        if -45 < angulo < 45:
            angulos.append(angulo)

    if not angulos:
        return edges, img

    return edges, mm.rotate(img, np.median(angulos), interp="bicubic")

# Ejecución del pipeline de deskew
img_edges, img_final = retificar_inclinacao_documento(img_original)

# Exhibición múltiple estandarizada con el formato nativo del libro
mm.show(
    [img_original, img_edges, img_final],
    titles=["Imagen Original", "Bordes de Canny", "Documento Rectificado"],
    cols=3,
    figsize=(12, 4)
)

<Figure size 1800x600 with 3 Axes>

**Figura 6.7:** *Pipeline* de rectificación axial: exhibición comparativa entre la entrada rotacionada original, el mapa de gradientes estructurales de Canny y el resultado final alineado con fondo normalizado en blanco.


> ### 📝 🧠 ¿Por qué funciona? — Transformada de Hough
>
> En la Transformada de Hough, cada píxel de borde contribuye con votos para
> todas las rectas que pueden pasar por su posición. En lugar de seleccionar
> solo la recta con el mayor número de votos, el algoritmo considera todas las
> rectas cuya cantidad de votos supera un umbral mínimo y calcula sus
> respectivos ángulos. La inclinación global del documento se estima entonces
> mediante la mediana de esos ángulos, una medida robusta ante valores atípicos.
> Así, las rectas espurias producidas por sombras, ruidos u otros elementos
> de la imagen ejercen poca influencia sobre la estimación final, siempre que la
> mayoría de las rectas detectadas corresponda a los bordes del documento.

### 6.8.4 Limitaciones Prácticas

Aunque presenta un buen rendimiento en condiciones habituales de digitalización, el método depende de la existencia de estructuras lineales suficientemente definidas para ser detectadas mediante la Transformada de Hough, como bordes de página, líneas de formularios o líneas de texto. Su precisión puede verse reducida en imágenes con baja resolución, ruido excesivo, sombras intensas o grandes inclinaciones. En general, los documentos digitalizados con una resolución cercana a 300 DPI y una iluminación homogénea proporcionan resultados adecuados para aplicaciones de OCR y OMR.

La implementación presentada en este capítulo tiene una finalidad didáctica, ilustrando los principios de la corrección automática de inclinación mediante la detección de bordes, la Transformada de Hough y la rotación afín. Al utilizar únicamente la orientación de las estructuras lineales predominantes, el método puede aplicarse a diferentes tipos de documentos, sin depender de marcadores específicos.

En sistemas reales de análisis documental, sin embargo, el alineamiento normalmente utiliza marcadores geométricos previamente conocidos. En el modelo de hoja de respuestas empleado por el ecosistema MCTest, por ejemplo, se utilizan cuatro discos negros de referencia, además de las regiones correspondientes al encabezado, al *QRCode* y a los cuadros de respuestas. La localización de estos elementos permite estimar simultáneamente la rotación, la escala y la traslación de la hoja, haciendo que el registro sea menos sensible a la cantidad de texto, a la ausencia de líneas estructurales y a las variaciones de impresión o digitalización.

Por este motivo, el enfoque basado en la Transformada de Hough se utiliza en este capítulo para introducir los fundamentos del problema, mientras que las etapas posteriores adoptan el alineamiento mediante marcadores geométricos, estrategia predominante en sistemas de OMR y análisis documental.

### 6.8.5 Detección de Bordes y Contornos

La localización precisa de las regiones de interés es una etapa esencial en
sistemas de OMR. En el modelo de hoja de respuestas utilizado en este capítulo,
el encabezado y el cuadro de respuestas están contenidos en un rectángulo virtual
delimitado por cuatro discos negros posicionados en las esquinas. La identificación
de estos marcadores permite localizar la región de interés y corregir
distorsiones geométricas introducidas durante la adquisición de la imagen.

El procedimiento se compone de cinco etapas. Inicialmente, se aplica un
**cierre morfológico** (dilatación seguida de erosión), operación estudiada
en el **Capítulo 4**, utilizando un elemento estructurante en disco
(`mm.sedisk(33)`). Esta operación reduce pequeñas discontinuidades y preserva
los discos de referencia, haciéndolos más homogéneos. A continuación, la imagen se
invierte (`mm.neg`), de modo que los discos pasen a constituir componentes
claros sobre fondo oscuro.

En la etapa siguiente, se aplica la operación `mm.edgeoff` (**Capítulo 4**), que
elimina componentes conectados a los bordes de la imagen, eliminando artefactos como
sombras de digitalización, marcas de corte y otros objetos espurios en los
márgenes. Los componentes restantes se analizan entonces a partir de sus
contornos y se filtran por propiedades geométricas, como área y
circularidad, para identificar los discos candidatos. La implementación del
MCTest hace que este proceso sea más robusto al seleccionar, entre todos los
candidatos, los cuatro cuyos centros forman un rectángulo con ancho
compatible con el de la imagen, reduciendo la ocurrencia de falsos positivos.

Finalmente, los centros de los cuatro discos se ordenan espacialmente (superior
izquierdo, superior derecho, inferior izquierdo e inferior derecho) y se
utilizan como puntos de control en una transformación de perspectiva
(*perspective warp*). Esta transformación rectifica la imagen, produciendo una
representación alineada y con dimensiones conocidas, adecuada para las etapas
subsiguientes de segmentación y reconocimiento.

Las principales etapas de este *pipeline*, desde el procesamiento morfológico hasta
la imagen rectificada, se ilustran a continuación.

In [11]:
import cv2
import numpy as np
from morph import mm

# img: imagen en escala de grises de la hoja de prueba
if img_final.ndim == 2:
    img = img_final
else:
    img = mm.gray(img_final)

# 1. Cierre morfológico: preserva los discos oscuros, eliminando todo lo menor que el disco
img_close = mm.close(img, mm.sedisk(41))

# 2. Inversión: los discos oscuros se convierten en componentes claros sobre fondo oscuro
img_neg = mm.neg(img_close)

# 3. Elimina componentes conectados que tocan el borde de la imagen
img_edgeoff = mm.edgeoff(img_neg)

# 4. Extracción de los contornos externos
contornos, _ = cv2.findContours(img_edgeoff, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 5. Filtrado por área y circularidad, manteniendo solo los 4 discos
centros = []
for c in contornos:
    area = cv2.contourArea(c)
    perimetro = cv2.arcLength(c, True)
    if area < 50 or perimetro == 0:
        continue
    circularidade = 4 * np.pi * area / (perimetro ** 2)
    if circularidade > 0.8:
        M = cv2.moments(c)
        cx, cy = M["m10"] / M["m00"], M["m01"] / M["m00"]
        centros.append((cx, cy))

# Verificación robusta: interrumpe el pipeline con mensaje claro en lugar de AssertionError
if len(centros) != 4:
    print(f"[AVISO] Se esperaban 4 discos marcadores, encontrado {len(centros)}.")
    print("  Verifique si la imagen es una hoja de respuestas MCTest válida")
    print("  o ajuste los parámetros de circularidad y área mínima.")
    img_retificada = img  # fallback: preserva la imagen sin rectificación
else:
    # 6. Ordenación de los centros: superior-izquierdo, superior-derecho,
    # inferior-izquierdo, inferior-derecho
    pts = np.array(centros, dtype=np.float32)
    soma = pts.sum(axis=1)
    diff = pts[:, 0] - pts[:, 1]
    tl = pts[np.argmin(soma)]
    br = pts[np.argmax(soma)]
    tr = pts[np.argmax(diff)]
    bl = pts[np.argmin(diff)]
    pts_ordenados = np.array([tl, tr, bl, br], dtype=np.float32)

    # 7. Rectificación por transformación de perspectiva (warp)
    largura, altura = 800, 800
    destino = np.array(
        [[0, 0], [largura, 0], [0, altura], [largura, altura]], dtype=np.float32
    )
    M_persp = cv2.getPerspectiveTransform(pts_ordenados, destino)
    img_retificada = cv2.warpPerspective(img, M_persp, (largura, altura))

    mm.show(
        [img_close, img_edgeoff, img_retificada],
        titles=["Cierre (sedisk 41)", "edgeoff", "Rectificada (warp)"],
        cols=3,
        figsize=(12, 4)
    )

mm.write(img_retificada, "img_beetween_disks.png")

<Figure size 1800x600 with 3 Axes>

**Figura 6.8:** Detección de los discos marcadores, extracción de contornos y rectificación por transformación de perspectiva.


> ### 📝 🧠 ¿Por qué funciona? — Del cierre morfológico a la rectificación
>
> **Cierre morfológico:** la dilatación seguida de erosión rellena pequeñas discontinuidades y suaviza los contornos de los objetos sin alterar significativamente su forma global. Al utilizar un elemento estructurante grande (`sedisk(41)`), los detalles finos, como textos, líneas del formulario y pequeños ruidos, tienden a incorporarse al fondo durante el procesamiento, mientras que los objetos de mayor escala, como los discos de referencia, permanecen preservados y se vuelven más homogéneos.
>
> **Circularidad:** tras el aislamiento de los componentes candidatos, la métrica $C=\frac{4\pi A}{P^2}$ cuantifica cuánto se aproxima su forma a un círculo. Su valor es igual a 1 para un círculo perfecto y disminuye a medida que el contorno se vuelve más irregular. Así, un umbral como $C>0{,}8$ permite descartar la mayor parte de los falsos positivos sin recurrir a modelos de aprendizaje. Un simulador de esta métrica se presenta en la [Figura 6.9](#fig-06-sim-06-circularidade)..
>
> **Transformación de perspectiva (*perspective warp*):** una vez identificados los cuatro discos de referencia, sus centros se utilizan como puntos de control para estimar la transformación proyectiva que mapea la imagen capturada al plano del documento. Esta transformación corrige las distorsiones introducidas por la perspectiva durante la adquisición de la imagen, produciendo una representación frontal con dimensiones conocidas y adecuada para las etapas posteriores de segmentación y reconocimiento.

In [12]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-06-circularidade" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-06-circularidade * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-06-circularidade canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; margin: 0 auto; }
  #sim-06-circularidade button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-06-circularidade button:hover { background: #e8dfcf; }
  #sim-06-circularidade input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim06_circ_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim06_circ_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim06_circ_grid_stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 12px; }
  .sim06_circ_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .sim06_circ_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim06_circ_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim06_circ_legend { display: flex; gap: 16px; font-size: 10.5px; font-weight: 600; color: #5e5a4a; align-items: center; justify-content: center; margin-top: 10px; }
  .sim06_circ_dot { width: 10px; height: 10px; border-radius: 50%; display: inline-block; margin-right: 5px; vertical-align: middle; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">⭕ Simulador: Filtrado por Circularidad</span>
  <span class="sim06_circ_pill">C = 4πA / P²</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas e Slider -->
  <div class="sim06_circ_panel" style="margin-bottom:14px;">
    
    <div class="sim06_circ_grid_stats">
      <div class="sim06_circ_stat_box">
        <div class="sim06_circ_stat_label">Umbral C</div>
        <div id="sim06_circ_thVal" class="sim06_circ_stat_value" style="color:#2980b9;">0.60</div>
      </div>
      <div class="sim06_circ_stat_box">
        <div class="sim06_circ_stat_label">Aceptados</div>
        <div id="sim06_circ_nAcc" class="sim06_circ_stat_value" style="color:#27ae60;">0</div>
      </div>
      <div class="sim06_circ_stat_box">
        <div class="sim06_circ_stat_label">Rechazados</div>
        <div id="sim06_circ_nRej" class="sim06_circ_stat_value" style="color:#c0392b;">0</div>
      </div>
    </div>

    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Umbral de Circularidad: <span id="sim06_circ_slLabel" style="font-family:monospace; color:#26241d;">0.60</span>
      </label>
      <button id="sim06_circ_resetBtn">↺ Restablecer</button>
    </div>
    
    <input type="range" id="sim06_circ_slider" min="0.05" max="0.99" step="0.01" value="0.60">
    
    <div class="sim06_circ_legend">
      <span><span class="sim06_circ_dot" style="background:#27ae60;"></span>Aceptado (C ≥ umbral)</span>
      <span><span class="sim06_circ_dot" style="background:#c0392b;"></span>Rechazado (C &lt; umbral)</span>
    </div>

  </div>

  <!-- Canvas -->
  <div style="margin-bottom:14px; text-align:center;">
    <canvas id="sim06_circ_cv"></canvas>
  </div>

  <!-- Explicação Teórica -->
  <div class="sim06_circ_panel">
    <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:6px;">🧠 Fórmula de la Circularidad</div>
    <div style="font-size:11px; color:#5e5a4a; line-height:1.5;">
      La métrica <strong>C = 4πA / P²</strong> relaciona el área <em>A</em> del componente con el cuadrado de su perímetro <em>P</em>.
      Para un círculo perfecto, C = 1; para formas más irregulares o alargadas, C se aproxima a 0.
      En el contexto del MCTest, un umbral como C &gt; 0,60 selecciona los discos de referencia, descartando textos, líneas y artefactos de la hoja de respuestas.
    </div>
  </div>

</div>
</div>

<script>
(function() {
  function initSim06Circularidade(root){
    if (!root || root.dataset.sim06CircularidadeInit) return;
    root.dataset.sim06CircularidadeInit = "1";

    const COMPS = [
      {nome:"Círculo",       circ:1.0000, cx:80,  cy:75,  rx:52, ry:52, shape:"ellipse"},
      {nome:"Elipse leve",   circ:0.9649, cx:220, cy:75,  rx:60, ry:44, shape:"ellipse"},
      {nome:"Elipse along.", circ:0.7454, cx:375, cy:75,  rx:80, ry:32, shape:"ellipse"},
      {nome:"Quadrado",      circ:0.7854, cx:80,  cy:210, rx:48, ry:48, shape:"rect"},
      {nome:"Retângulo",     circ:0.6750, cx:220, cy:210, rx:70, ry:32, shape:"rect"},
      {nome:"Ret. fino",     circ:0.4740, cx:375, cy:210, rx:70, ry:16, shape:"rect"},
      {nome:"Estrela",       circ:0.2493, cx:145, cy:340, rx:48, ry:48, shape:"star"},
      {nome:"Forma-L",       circ:0.3704, cx:350, cy:340, rx:40, ry:48, shape:"L"},
    ];

    const W = 500, H = 430;
    const cv = root.querySelector('#sim06_circ_cv');
    cv.width = W; 
    cv.height = H;
    const ctx = cv.getContext('2d');

    function drawStar(cx, cy, r1, r2, n) {
      ctx.beginPath();
      for (let i = 0; i < n * 2; i++) {
        const angle = -Math.PI / 2 + i * Math.PI / n;
        const r = i % 2 === 0 ? r1 : r2;
        if (i === 0) ctx.moveTo(cx + r * Math.cos(angle), cy + r * Math.sin(angle));
        else ctx.lineTo(cx + r * Math.cos(angle), cy + r * Math.sin(angle));
      }
      ctx.closePath();
    }

    function drawL(cx, cy, rx, ry) {
      const bw = rx * 0.45, bh = ry * 1.0;
      const fh = ry * 0.28, fw = rx * 1.0;
      ctx.beginPath();
      ctx.rect(cx - bw, cy - bh, bw * 2, bh * 2);
      ctx.rect(cx - bw, cy + bh - fh * 2, fw * 2, fh * 2);
    }

    function draw(threshold) {
      ctx.clearRect(0, 0, W, H);
      ctx.fillStyle = '#fafaf7';
      ctx.fillRect(0, 0, W, H);

      ctx.strokeStyle = '#e4dcc8';
      ctx.lineWidth = 1;
      ctx.setLineDash([4, 4]);
      ctx.beginPath(); ctx.moveTo(20, 143); ctx.lineTo(W - 20, 143); ctx.stroke();
      ctx.beginPath(); ctx.moveTo(20, 278); ctx.lineTo(W - 20, 278); ctx.stroke();
      ctx.setLineDash([]);

      let acc = 0, rej = 0;

      COMPS.forEach(c => {
        const ok = c.circ >= threshold;
        if (ok) acc++; else rej++;

        const fill   = ok ? 'rgba(39, 174, 96, 0.15)' : 'rgba(192, 57, 43, 0.12)';
        const stroke = ok ? '#27ae60' : '#c0392b';

        ctx.save();
        ctx.fillStyle = fill;
        ctx.strokeStyle = stroke;
        ctx.lineWidth = 2.5;

        if (c.shape === 'ellipse') {
          ctx.beginPath();
          ctx.ellipse(c.cx, c.cy, c.rx, c.ry, 0, 0, Math.PI * 2);
          ctx.fill(); ctx.stroke();
        } else if (c.shape === 'rect') {
          ctx.beginPath();
          ctx.rect(c.cx - c.rx, c.cy - c.ry, c.rx * 2, c.ry * 2);
          ctx.fill(); ctx.stroke();
        } else if (c.shape === 'star') {
          drawStar(c.cx, c.cy, c.rx, c.rx * 0.42, 5);
          ctx.fill(); ctx.stroke();
        } else if (c.shape === 'L') {
          drawL(c.cx, c.cy, c.rx, c.ry);
          ctx.fill(); ctx.stroke();
        }

        // Ícone de aprovação/rejeição
        ctx.font = 'bold 18px sans-serif';
        ctx.textAlign = 'center';
        ctx.fillStyle = stroke;
        ctx.fillText(ok ? '✓' : '✗', c.cx, c.cy - c.ry - 6);

        // Nome da forma
        ctx.font = 'bold 11px Inter, system-ui, sans-serif';
        ctx.fillStyle = '#26241d';
        ctx.fillText(c.nome, c.cx, c.cy + c.ry + 14);

        // Valor de C
        ctx.font = '10px monospace';
        ctx.fillStyle = ok ? '#0e6251' : '#78281f';
        ctx.fillText('C = ' + c.circ.toFixed(4), c.cx, c.cy + c.ry + 27);

        ctx.restore();
      });

      root.querySelector('#sim06_circ_nAcc').textContent = acc;
      root.querySelector('#sim06_circ_nRej').textContent = rej;
    }

    function update() {
      const th = parseFloat(root.querySelector('#sim06_circ_slider').value);
      root.querySelector('#sim06_circ_thVal').textContent   = th.toFixed(2);
      root.querySelector('#sim06_circ_slLabel').textContent = th.toFixed(2);
      draw(th);
    }

    root.querySelector('#sim06_circ_slider').addEventListener('input', update);
    root.querySelector('#sim06_circ_resetBtn').addEventListener('click', () => {
      root.querySelector('#sim06_circ_slider').value = 0.60;
      update();
    });

    update();
  }

  function tryInitSim06Circularidade(){
    var root = document.getElementById('sim-06-circularidade');
    if (root) initSim06Circularidade(root); else setTimeout(tryInitSim06Circularidade, 200);
  }
  tryInitSim06Circularidade();
})();
</script>
""")

**Figura 6.9:** Simulador interactivo de filtrado por circularidad: mueva el *slider* para ajustar el umbral C y observe qué componentes son aceptados (verde) o rechazados (rojo).


<figure id="fig-06-sim-06-circularidade">
  <img src="imagens/fig-06-sim-06-circularidade.png" alt=" Simulador interactivo de filtrado por circularidad: mueva el *slider* para ajustar el umbral C y observe qué componentes son aceptados (verde) o rechazados (rojo). " style="max-width:80%" />
  <figcaption><strong>Figura 6.9:</strong>  Simulador interactivo de filtrado por circularidad: mueva el *slider* para ajustar el umbral C y observe qué componentes son aceptados (verde) o rechazados (rojo). </figcaption>
</figure>

### 6.8.6 Aislamiento, Segmentación y Decodificación del *QRCode*

Tras la rectificación geométrica de la hoja de respuestas, se realiza la detección y
la decodificación del *QRCode* presente en el formulario. Este marcador almacena
información utilizada por el sistema de OMR (*Optical Mark Recognition*),
como la identificación del estudiante, el código de la prueba y su variación,
posibilitando la recuperación de la plantilla de respuestas correspondiente en la base de datos.
Por seguridad, esta información se cifra antes de la generación del
*QRCode*. Así, la secuencia decodificada corresponde a una *cadena*
hexadecimal, cuya interpretación se realiza exclusivamente por el sistema
MCTest. El procedimiento consta de tres etapas: preprocesamiento
morfológico, aislamiento de la región del *QRCode* y decodificación de su
contenido.

Inicialmente, la imagen rectificada en escala de grises se binariza mediante
la operación `mm.threshold`. A continuación, se aplica una **apertura
morfológica** (erosión seguida de dilatación), estudiada en el **Capítulo 4**,
utilizando un elemento estructurante cuadrado (`mm.sebox(2)`). Esta operación
elimina pequeños ruidos y suaviza imperfecciones sin comprometer la estructura del
marcador. Finalmente, la imagen se invierte (`mm.neg`), de modo que el
*QRCode* pase a constituir un componente claro sobre fondo oscuro,
facilitando la extracción de sus contornos.

La localización del *QRCode* se realiza mediante el análisis de los contornos externos de la
imagen binarizada. Entre los componentes detectados, se selecciona aquel con
mayor área y geometría aproximadamente cuadrada, descartando los demás
elementos impresos de la hoja. A continuación, la región correspondiente se expande
con un pequeño margen de seguridad, garantizando la preservación integral del
marcador.

El *QRCode* se extrae entonces directamente de la imagen rectificada en escala de
grises, preservando su calidad radiométrica. Como esta región generalmente
presenta dimensiones reducidas, se aplica un redimensionamiento con
interpolación cúbica, aumentando la resolución espacial y favoreciendo la
identificación de sus módulos. La lectura se realiza mediante el detector de
*QRCode* de OpenCV (`cv2.QRCodeDetector`), que recupera la secuencia de
caracteres originalmente codificada.

El flujo completo de este procesamiento, desde el preprocesamiento morfológico
hasta la decodificación del *QRCode*, se ilustra en la
[Figura 6.10](#fig-06-processamento-qrcode). El fragmento mostrado en la salida corresponde solo
al inicio de la *cadena* hexadecimal cifrada; su interpretación completa se
realiza internamente por MCTest tras la decodificación.

In [13]:
import cv2
import numpy as np
from morph import mm

# f: imagen rectificada convertida al tipo correcto de 8 bits (0–255)
f = img_retificada.astype('uint8')

# 1. Umbralización: conversión de la imagen en tonos de gris a binaria
f_thresh = mm.threshold(f)

# 2. Apertura morfológica: elimina pequeños ruidos y suaviza el contorno de los bloques
f_open = mm.open(f_thresh, mm.sebox(2))

# 3. Inversión morfológica: los módulos oscuros se convierten en componentes claros sobre fondo oscuro
f_inv = mm.neg(f_open)

# 4. Conversión segura a uint8 con escala 0–255
img_uint8 = (f_inv.astype(np.uint8) * 255) if f_inv.max() == 1 else f_inv.astype(np.uint8)

# 5. Detección de contornos externos
contornos, _ = cv2.findContours(img_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if not contornos:
    raise ValueError("Nenhum contorno encontrado. Verifique o limiar ou a imagem de entrada.")

# 6. Filtrado por el mayor contorno con proporción aproximadamente cuadrada
#    (relación de aspecto entre 0.7 y 1.3 descarta rectángulos alargados de la hoja)
def is_square_like(contorno, tol=0.3):
    x, y, w, h = cv2.boundingRect(contorno)
    ratio = w / h if h > 0 else 0
    return (1 - tol) <= ratio <= (1 + tol)

candidatos = [c for c in contornos if is_square_like(c)]

if not candidatos:
    raise ValueError(
        "Nenhum contorno quadrado encontrado. "
        "Verifique se o QRCode está presente na imagem ou ajuste a tolerância."
    )

# Selecciona el mayor candidato cuadrado por área de bounding box
maior_contorno = max(candidatos, key=lambda c: cv2.boundingRect(c)[2] * cv2.boundingRect(c)[3])
x, y, w, h = cv2.boundingRect(maior_contorno)

# 7. Expansión de la bounding box con margen de seguridad (evita truncamiento del QRCode)
margem = 5
h_img, w_img = img_uint8.shape[:2]
x1 = max(x - margem, 0)
y1 = max(y - margem, 0)
x2 = min(x + w + margem, w_img)
y2 = min(y + h + margem, h_img)

# 8. Recorte de la región de interés a partir de la imagen original (nítida, en gris)
img_qrcode_final = img_retificada[y1:y2, x1:x2]

# 9. Ampliación para resolución mínima de decodificación (400 px en el lado mayor)
#    cv2.QRCodeDetector requiere módulos con al menos 3–4 px de ancho para decodificar
#    con seguridad; imágenes menores a ~400 px tienden a fallar.
lado = max(img_qrcode_final.shape[:2])
escala = max(400 / lado, 1.0)
img_para_leitura = cv2.resize(
    img_qrcode_final, None,
    fx=escala, fy=escala,
    interpolation=cv2.INTER_CUBIC
)

# 10. Inicialización del detector nativo de QRCode de OpenCV
detector = cv2.QRCodeDetector()

# 11. Detección geométrica y decodificación de los datos textuales
dados, pontos, qrcode_reto = detector.detectAndDecode(img_para_leitura)


# Visualización intermedia: progresión de la binarización al aislamiento del QRCode
mm.show(
    [f_thresh, f_open, f_inv, img_qrcode_final],
    titles=["1. Umbralización", "2. Apertura morfológica", "3. Inversión", "Imagen final"],
    cols=3, figsize=(12, 4)
)

# Validación y salida de los metadatos extraídos
if dados:
    print(f"QRCode decodificado con éxito: \n{dados[:50]}...")
else:
    raise ValueError(
        "Falha na decodificação do QRCode. "
        "Verifique o limiar, as margens da região ou a qualidade da imagem."
    )

<Figure size 1800x600 with 6 Axes>

**Figura 6.10:** Pipeline de procesamiento del *QRCode*: umbralización, apertura morfológica, inversión y recorte final para decodificación.


QRCode decodificado con éxito: 
325a356b71367266556955646b7233454149624a694f417730...


In [14]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-06-qrcode" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📱 Simulador EP06: Aislamiento del QRCode</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Umbral → Cierre → Contorno · Patrón Sintético</span>
  </div>

<style>
  #sim-06-qrcode * { box-sizing: border-box; }
  #sim-06-qrcode { font-family: sans-serif; padding: 10px; max-width: 780px; margin: 0 auto; color: #374151; }
  #sim-06-qrcode canvas { display: block; border-radius: 6px; border: 1px solid #d1d5db; background: #fff; }
  #sim-06-qrcode button { font-size: 11px; padding: 5px 10px; border-radius: 4px; border: 1px solid #d1d5db; background: #fff; color: #374151; cursor: pointer; }
  #sim-06-qrcode button:hover { background: #f3f4f6; }
  #sim-06-qrcode input[type=range] { accent-color: #6366f1; }
  .qr_panel  { background: #f9fafb; border: 1px solid #e5e7eb; border-radius: 6px; padding: 10px; margin-bottom: 8px; text-align: left;}
  .qr_pill   { font-size: 10px; font-weight: bold; padding: 3px 8px; border-radius: 4px; border: 1px solid #a5b4fc; background: #eef2ff; color: #4338ca; }
  .qr_steps  { display: flex; gap: 6px; flex-wrap: wrap; margin-bottom: 8px; }
  .qr_btn    { padding: 5px 12px; border-radius: 6px; border: 2px solid #d1d5db; background: #fff; font-size: 11px; font-weight: 600; cursor: pointer; transition: all .15s; }
  .qr_btn.active { border-color: #6366f1; background: #eef2ff; color: #4338ca; }
  .qr_canvases { display: flex; gap: 12px; flex-wrap: wrap; justify-content: center; }
  .qr_cv_wrap  { text-align: center; }
  .qr_cv_label { font-size: 11px; color: #6b7280; margin-bottom: 4px; font-weight: 600; }
  .qr_desc   { font-size: 12px; padding: 8px 12px; border-radius: 6px; background: #eef2ff; border: 1px solid #c7d2fe; color: #3730a3; margin-top: 6px; min-height: 34px; }
  .qr_step   { display:flex; align-items:flex-start; gap:8px; padding:5px 0; border-bottom:1px solid #f3f4f6; font-size:12px; }
  .qr_step:last-child { border-bottom:none; }
  .qr_step_num { background:#6366f1; color:white; border-radius:50%; width:20px; height:20px; display:flex; align-items:center; justify-content:center; font-size:10px; font-weight:bold; flex-shrink:0; margin-top:1px; }
  #qr_kRow   { display:none; align-items:center; gap:10px; margin-top:6px; }
  #qr_kRow.visible { display:flex; }
  .qr_warn { font-size:10px; color:#92400e; background:#fef3c7; border:1px solid #fde68a; border-radius:4px; padding:4px 8px; margin-top:6px; }
</style>

<div class="qr_panel">
  <div style="font-size:11px;font-weight:700;color:#374151;margin-bottom:6px;">Seleccione la etapa:</div>
  <div class="qr_steps">
    <button class="qr_btn active" data-step="0">0 · Original</button>
    <button class="qr_btn" data-step="1">1 · Umbralización</button>
    <button class="qr_btn" data-step="2">2 · Cierre</button>
    <button class="qr_btn" data-step="3">3 · Contorno</button>
    <button class="qr_btn" data-step="4">4 · Recorte</button>
  </div>
  <div id="qr_kRow">
    <label style="font-size:11px;font-weight:600;white-space:nowrap;">Elemento estructurante — sebox(<span id="qr_kVal">1</span>):</label>
      <input type="range" id="qr_kSlider" min="0" max="3" step="1" value="1" style="width:140px;">
    <span id="qr_kDesc" style="font-size:10px;color:#6b7280;"></span>
  </div>
  <div id="qr_desc" class="qr_desc"></div>
  <div id="qr_warnBox"></div>
</div>

<div class="qr_canvases">
  <div class="qr_cv_wrap">
    <div class="qr_cv_label" id="qr_main_label">Original</div>
    <canvas id="qr_cvMain" width="250" height="250"></canvas>
  </div>
  <div class="qr_cv_wrap">
    <div class="qr_cv_label">Recorte final (QRCode)</div>
    <canvas id="qr_cvCrop" width="160" height="160"></canvas>
  </div>
</div>

<div class="qr_panel" style="margin-top:8px;">
  <div style="font-size:11px;font-weight:700;color:#374151;margin-bottom:6px;">🧠 Etapas del pipeline (réplica fiel del algoritmo OpenCV de referencia)</div>
  <div class="qr_step"><div class="qr_step_num">1</div><div><strong>Umbralización</strong> (<code>mm.threshold</code>): Segmenta los módulos oscuros del QRCode aislándolos del fondo claro.</div></div>
  <div class="qr_step"><div class="qr_step_num">2</div><div><strong>Cierre morfológico</strong> (<code>mm.close + mm.sebox(k)</code>): dilatación seguida de erosión elimina ruidos y rellena discontinuidades. <code>sebox(0)</code> = kernel 3×3, <code>sebox(1)</code> = 5×5, y así sucesivamente. Note que aquí se usa <em>cierre</em> (≠ apertura usada en la celda de código arriba), lo que estimula la comparación entre los dos operadores.</div></div>
  <div class="qr_step"><div class="qr_step_num">4</div><div><strong>Detección de contornos</strong> (<code>cv2.findContours</code>, <code>RETR_EXTERNAL</code>): Busca el mayor contorno externo con proporción aproximadamente cuadrada (razón ancho/alto entre 0.7 y 1.3), descartando rectángulos alargados de la hoja.</div></div>
  <div class="qr_step"><div class="qr_step_num">5</div><div><strong>Recorte + margen de seguridad:</strong> Extrae la <em>bounding box</em> del contorno seleccionado, con margen de 5px, a partir de la imagen original en escala de grises.</div></div>
</div>

<script>
(function() {
  // Imagem original do usuário (img_beetween_disks.png), redimensionada para 256x256 e
  // codificada em Base64. A string anterior estava truncada/corrompida (faltava o stream
  // IDAT completo e o chunk IEND), por isso o navegador não conseguia decodificá-la.
  const QR_B64 = "...";
  const SZ = 256;
  const DISP = 250;

  const origImg = new Image();
  origImg.src = 'data:image/png;base64,' + QR_B64;

  let BIN = null;

  const cvMain = document.getElementById('qr_cvMain');
  const cvCrop = document.getElementById('qr_cvCrop');
  const ctxM   = cvMain.getContext('2d');
  const ctxC   = cvCrop.getContext('2d');

  origImg.onload = function() {
    const off = document.createElement('canvas');
    off.width = off.height = SZ;
    const octx = off.getContext('2d');
    octx.drawImage(origImg, 0, 0, SZ, SZ);
    const data = octx.getImageData(0, 0, SZ, SZ).data;
    BIN = Array.from({length: SZ}, (_,r) =>
      Array.from({length: SZ}, (_,c) => {
        const i = (r*SZ+c)*4;
        return data[i] < 128 ? 1 : 0;
      })
    );
    setStep(0);
  };

  function erode(bin, k) {
    const out = Array.from({length:SZ}, () => new Uint8Array(SZ));
    for (let r=k; r<SZ-k; r++)
      for (let c=k; c<SZ-k; c++) {
        let mn=1;
        for (let dr=-k; dr<=k && mn; dr++)
          for (let dc=-k; dc<=k && mn; dc++)
            if (!bin[r+dr][c+dc]) mn=0;
        out[r][c]=mn;
      }
    return out;
  }

  function dilate(bin, k) {
    const out = Array.from({length:SZ}, () => new Uint8Array(SZ));
    for (let r=k; r<SZ-k; r++)
      for (let c=k; c<SZ-k; c++) {
        let mx=0;
        for (let dr=-k; dr<=k && !mx; dr++)
          for (let dc=-k; dc<=k && !mx; dc++)
            if (bin[r+dr][c+dc]) mx=1;
        out[r][c]=mx;
      }
    return out;
  }

  function opening(bin, k) { return dilate(erode(bin,k),k); }
  function close(bin, k) { return erode(dilate(bin,k),k); }
  function invert(bin) { return bin.map(row => row.map(v => v^1)); }

  // ---- Detecção de contornos via flood-fill com rastreamento de bounding box ----
  // Replica cv2.findContours(img, RETR_EXTERNAL) + filtro is_square_like(tol=0.3)
  // seguido da seleção do maior candidato por área de bounding box (igual ao
  // algoritmo Python de referência).
  function findSquareContours(bin) {
    const H = bin.length, W = bin[0].length;
    const visited = Array.from({length:H}, () => new Uint8Array(W));
    const comps = [];
    for (let sr=0; sr<H; sr++) for (let sc=0; sc<W; sc++) {
      if (!bin[sr][sc] || visited[sr][sc]) continue;
      const q=[[sr,sc]]; visited[sr][sc]=1;
      let minr=sr,maxr=sr,minc=sc,maxc=sc,area=0;
      while(q.length) {
        const [r,c]=q.shift(); area++;
        minr=Math.min(minr,r); maxr=Math.max(maxr,r);
        minc=Math.min(minc,c); maxc=Math.max(maxc,c);
        // 8-conectividade, como o OpenCV usa para contornos
        for (const [dr,dc] of [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]]) {
          const nr=r+dr,nc=c+dc;
          if(nr>=0&&nr<H&&nc>=0&&nc<W&&!visited[nr][nc]&&bin[nr][nc])
            { visited[nr][nc]=1; q.push([nr,nc]); }
        }
      }
      comps.push({minr,maxr,minc,maxc,area});
    }

    // Filtro: aspect ratio da bounding box entre 0.7 e 1.3 (tol=0.3)
    const isSquareLike = (c) => {
      const w = c.maxc-c.minc+1, h = c.maxr-c.minr+1;
      const ratio = h>0 ? w/h : 0;
      return ratio >= 0.7 && ratio <= 1.3;
    };
    const candidatos = comps.filter(isSquareLike);
    return { total: comps.length, candidatos };
  }

  function pickLargestByBBoxArea(candidatos) {
    if (!candidatos.length) return null;
    return candidatos.reduce((best, c) => {
      const areaC = (c.maxc-c.minc+1)*(c.maxr-c.minr+1);
      const areaB = best ? (best.maxc-best.minc+1)*(best.maxr-best.minr+1) : -1;
      return areaC > areaB ? c : best;
    }, null);
  }

  function expandWithMargin(bbox, margem, H, W) {
    return {
      minr: Math.max(bbox.minr - margem, 0),
      minc: Math.max(bbox.minc - margem, 0),
      maxr: Math.min(bbox.maxr + margem, H-1),
      maxc: Math.min(bbox.maxc + margem, W-1)
    };
  }

  function renderBin(ctx, bin, W, H, invert=false, highlight=null, highlightLabel='') {
    const imgData = ctx.createImageData(W, H);
    const scaleR = SZ/H, scaleC = SZ/W;
    for (let py=0; py<H; py++) for (let px=0; px<W; px++) {
      const r=Math.min(SZ-1,Math.floor(py*scaleR));
      const c=Math.min(SZ-1,Math.floor(px*scaleC));
      const v = bin[r][c] ^ (invert?1:0);
      const i=(py*W+px)*4;
      imgData.data[i]  = v?255:0;
      imgData.data[i+1]= v?255:0;
      imgData.data[i+2]= v?255:0;
      imgData.data[i+3]= 255;
    }
    ctx.putImageData(imgData,0,0);
    if (highlight) {
      const s = DISP/SZ;
      ctx.save();
      ctx.strokeStyle='#dc2626'; ctx.lineWidth=2; ctx.setLineDash([5,3]);
      ctx.strokeRect(highlight.minc*s, highlight.minr*s,
                     (highlight.maxc-highlight.minc)*s, (highlight.maxr-highlight.minr)*s);
      ctx.setLineDash([]);
      ctx.fillStyle='rgba(220,38,38,0.10)';
      ctx.fillRect(highlight.minc*s, highlight.minr*s,
                   (highlight.maxc-highlight.minc)*s, (highlight.maxr-highlight.minr)*s);
      ctx.font='bold 11px sans-serif'; ctx.fillStyle='#dc2626'; ctx.textAlign='left';
      ctx.fillText(highlightLabel || 'Maior contorno quadrado', highlight.minc*s+3, Math.max(highlight.minr*s-4,10));
      ctx.restore();
    }
  }

  function renderCrop(originalBin, bbox) {
    ctxC.clearRect(0,0,160,160);
    ctxC.fillStyle='#f9fafb'; ctxC.fillRect(0,0,160,160);
    if (!bbox) {
      ctxC.fillStyle='#9ca3af'; ctxC.font='11px sans-serif';
      ctxC.textAlign='center';
      ctxC.fillText('Disponível na', 80, 72);
      ctxC.fillText('etapa 5 · Recorte', 80, 88);
      return;
    }
    const cw=bbox.maxc-bbox.minc+1, ch=bbox.maxr-bbox.minr+1;
    const sc=Math.min(148/cw, 148/ch);
    const dw=Math.round(cw*sc), dh=Math.round(ch*sc);
    const ox=Math.round((160-dw)/2), oy=Math.round((160-dh)/2);
    const imgData=ctxC.createImageData(dw,dh);
    for(let py=0;py<dh;py++) for(let px=0;px<dw;px++) {
      const r=Math.min(SZ-1, bbox.minr+Math.round(py/sc));
      const c=Math.min(SZ-1, bbox.minc+Math.round(px/sc));
      const v=originalBin[r][c]?0:255;
      const i=(py*dw+px)*4;
      imgData.data[i]=v; imgData.data[i+1]=v; imgData.data[i+2]=v; imgData.data[i+3]=255;
    }
    ctxC.putImageData(imgData,ox,oy);
    ctxC.strokeStyle='#dc2626'; ctxC.lineWidth=2;
    ctxC.strokeRect(ox,oy,dw,dh);
  }

  const DESCS = [
    "Imagem original em escala de cinza: documento com QRCode no canto superior direito e bolhas de respostas.",
    "Binarização (mm.threshold): Separa os módulos escuros (QRCode, texto, bolhas) do fundo claro da folha.",
    "Fechamento morfológica (sebox(k)): Ruídos finos menores que k px são juntados, se próximos, pela dilatação seguida de erosão.",
    "cv2.findContours(RETR_EXTERNAL) + filtro de proporção quadrada (0.7–1.3): seleciona o maior contorno candidato.",
    "Recorte final com margem de 5px a partir da imagem original em escala de cinza, na bounding box do contorno selecionado."
  ];

  let curK = 0;  // sebox(0) = 3×3 padrão
  let cachedOpened = {};

  function getProcessed(k) {
    if (!BIN) return null;
    if (!cachedOpened[k]) {
      // mm.sebox(k) = kernel (2k+3)×(2k+3): sebox(0)=3x3, sebox(1)=5x5...
      const kKernel = k + 1;  // mapeia sebox(k) para raio do kernel quadrado
      cachedOpened[k] = close(BIN, kKernel);
    }
    return cachedOpened[k];
  }

  function render() {
    if (!BIN) return;
    const k = curK;
    const opened  = getProcessed(k);
    const inverted = opened; //invert(opened);

    const { total, candidatos } = findSquareContours(inverted);
    const maiorContorno = pickLargestByBBoxArea(candidatos);
    const bboxComMargem = maiorContorno ? expandWithMargin(maiorContorno, 5, SZ, SZ) : null;

    const warnBox = document.getElementById('qr_warnBox');
    if (maiorContorno) {
      const w = maiorContorno.maxc - maiorContorno.minc + 1;
      const h = maiorContorno.maxr - maiorContorno.minr + 1;
      const ratio = w / h;
      if (ratio > 1.5 || ratio < 0.67) {
        warnBox.innerHTML = '<div class="qr_warn">⚠️ O maior contorno quadrado detectado incorporou uma linha horizontal da folha, distorcendo a bounding box. Isso ocorre porque sebox(' + k + ') funde módulos do QRCode com elementos adjacentes. Na prática, o algoritmo MCTest usa critérios adicionais (área mínima e proximidade com a borda superior) para evitar essa situação. Reduza k para obter o recorte correto.</div>';
      } else {
        warnBox.innerHTML = '';
      }
    } else {
      warnBox.innerHTML = '<div class="qr_warn">⚠️ Nenhum contorno quadrado encontrado. Reduza k.</div>';
    }

    const kDescs = [
      'sebox(0) = 3×3 — fechamento leve, preserva módulos finos',
      'sebox(1) = 5×5 — fechamento médio, ideal para segmentar o QRCode',
      'sebox(2) = 7×7 — começa a fundir módulos adjacentes',
      'sebox(3) = 9×9 — destrói a estrutura do QRCode'
    ];
    document.getElementById('qr_kDesc').textContent = kDescs[k] || '';

    cvMain.width = DISP; cvMain.height = DISP;

    switch(curStep) {
      case 0:
        ctxM.drawImage(origImg, 0, 0, DISP, DISP);
        renderCrop(BIN, null);
        break;
      case 1:
        renderBin(ctxM, BIN, DISP, DISP);
        renderCrop(BIN, null);
        break;
      case 2:
        renderBin(ctxM, opened, DISP, DISP);
        renderCrop(BIN, null);
        break;
      case 3:
        renderBin(ctxM, inverted, DISP, DISP, false, maiorContorno, `Contorno (${candidatos.length}/${total} quadrado-like)`);
        renderCrop(BIN, null);
        break;
      case 4:
        ctxM.drawImage(origImg, 0, 0, DISP, DISP);
        if (bboxComMargem) {
          const s = DISP/SZ;
          ctxM.save();
          ctxM.strokeStyle = '#dc2626';  // era '#6366f1'
          ctxM.lineWidth = 2;
          ctxM.setLineDash([5,3]);
          ctxM.strokeRect(bboxComMargem.minc*s, bboxComMargem.minr*s,
                          (bboxComMargem.maxc-bboxComMargem.minc)*s,
                          (bboxComMargem.maxr-bboxComMargem.minr)*s);
          ctxM.setLineDash([]);
          ctxM.restore();
        }
        renderCrop(BIN, bboxComMargem);
        break;
    }
  }

  function setStep(s) {
    curStep = s;
    document.querySelectorAll('#sim-06-qrcode .qr_btn').forEach((b,i) => b.classList.toggle('active', i===s));
    document.getElementById('qr_main_label').textContent = ['0 · Original','1 · Limiarização','2 · Fechamento','3 · Contorno','4 · Recorte'][s];
    document.getElementById('qr_desc').textContent = DESCS[s];
    document.getElementById('qr_kRow').classList.toggle('visible', s===2);
    render();
  }

  document.querySelectorAll('#sim-06-qrcode .qr_btn').forEach((btn,i) => btn.addEventListener('click', ()=>setStep(i)));

  document.getElementById('qr_kSlider').addEventListener('input', function() {
    curK = parseInt(this.value);
    document.getElementById('qr_kVal').textContent = curK;
    render();
  });
}());
</script>
</div>
""")

**Figura 6.11:** Simulador interactivo del *pipeline* de aislamiento del QRCode: navegue por las etapas de filtrado, ajuste el elemento estructurante del cierre morfológico y vea la detección del contorno cuadrado sobre la imagen original de la plantilla.


<figure id="fig-06-sim-06-qrcode">
  <img src="imagens/fig-06-sim-06-qrcode.png" alt=" Simulador interactivo del *pipeline* de aislamiento del QRCode: navegue por las etapas de filtrado, ajuste el elemento estructurante del cierre morfológico y vea la detección del contorno cuadrado sobre la imagen original de la plantilla. " style="max-width:80%" />
  <figcaption><strong>Figura 6.11:</strong>  Simulador interactivo del *pipeline* de aislamiento del QRCode: navegue por las etapas de filtrado, ajuste el elemento estructurante del cierre morfológico y vea la detección del contorno cuadrado sobre la imagen original de la plantilla. </figcaption>
</figure>

> ### 📝 🧠 ¿Por qué funciona? — Aislamiento y decodificación del *QRCode*
>
> **Apertura morfológica:** a diferencia del cierre, la apertura (erosión
> seguida de dilatación) elimina pequeños ruidos y protuberancias sin alterar
> significativamente la geometría de los objetos más grandes. Así, preserva la
> estructura del *QRCode* mientras elimina componentes espurios que podrían
> dificultar su localización.
>
> **Selección por geometría:** el *QRCode* posee un formato aproximadamente
> cuadrado ($w/h \approx 1$). La combinación de este criterio con la selección del
> componente de mayor área descarta líneas del formulario, textos y otros
> elementos impresos, permitiendo aislar el marcador sin el uso de modelos de
> aprendizaje.
>
> **Redimensionamiento antes de la decodificación:** cuando el *QRCode* ocupa pocos
> píxeles en la imagen, sus módulos se vuelven difíciles de distinguir. El
> redimensionamiento con interpolación cúbica aumenta la resolución espacial de la
> región de interés, facilitando la identificación de los patrones del código por
> el `cv2.QRCodeDetector` y haciendo la decodificación más robusta.

### 6.8.7 Decodificación de Códigos de Barras

En la sección anterior, la decodificación de *QRCodes* se realizó utilizando el detector nativo de OpenCV (`cv2.QRCodeDetector`), que integra en una única interfaz la detección geométrica del símbolo, su rectificación y la extracción de la información codificada.

Para códigos de barras lineales (1D), como EAN-13, Code 39 y Code 128, una alternativa ampliamente utilizada es la biblioteca `pyzbar`. A diferencia del `QRCodeDetector`, esta soporta diversas simbologías de códigos de barras y también puede emplearse en la lectura de *QRCodes*.

El procedimiento consiste en localizar automáticamente cada símbolo presente en la imagen e interpretar la secuencia de barras y espacios correspondiente, produciendo la cadena de caracteres codificada. Además de los datos decodificados, la biblioteca proporciona información como la simbología identificada y la posición del código en la imagen, permitiendo su posterior validación o procesamiento.

La [Figura 6.12](#fig-06-barcode-decode) presenta un ejemplo de código de barras y el resultado de su decodificación utilizando la biblioteca `pyzbar`.

In [15]:
import os
import urllib.request
import numpy as np
from pyzbar.pyzbar import decode
from morph import mm

barcode_path = 'dados/barcode.png'
url_github = (
    "https://raw.githubusercontent.com/fzampirolli/"
    "pdi-vc/master/all/cap06/dados/barcode.png"
)

# Si el archivo no existe localmente, descarga automáticamente desde GitHub
if not os.path.exists(barcode_path):
    print(f"[DESCARGA] Descargando código de barras desde GitHub: {url_github}")
    try:
        os.makedirs(os.path.dirname(barcode_path), exist_ok=True)
        urllib.request.urlretrieve(url_github, barcode_path)
        print("[DESCARGA] ¡Imagen descargada con éxito!")
    except Exception as e:
        print(f"[DESCARGA] Error al descargar el archivo: {e}")

if os.path.exists(barcode_path):
    image = mm.read(barcode_path)
else:
    # Respaldo sintético: genera un patrón de barras verticales que simula un Code-128
    print("[AVISO] Archivo 'datos/barcode.png' no encontrado.")
    print("        Usando imagen sintética para demostración del pipeline.")
    h, w = 100, 400
    img_synth = np.ones((h, w), dtype=np.uint8) * 255
    # Barras oscuras en posiciones regulares (patrón simplificado)
    for x in range(20, w - 20, 8):
        if (x // 8) % 3 != 0:
            img_synth[:, x:x+4] = 0
    image = img_synth

mm.show(image)

# Ejecuta el decode de pyzbar
try:
    barcodes = decode(image)
except ImportError:
    print("[ERROR] zbar del sistema no encontrada. Ejecuta: !apt-get install -y libzbar0")
    barcodes = []

if barcodes:
    dados_bc = barcodes[0].data.decode("utf-8")
    tipo = barcodes[0].type
    print(f"Código de barras decodificado con éxito [{tipo}]:\n{dados_bc}")
else:
    print("[INFO] Ningún código de barras detectado en la imagen.")
    print(
        "En imagen sintética esto es esperado — "
        "reemplaza con el archivo real para decodificar."
    )

<Figure size 450x450 with 1 Axes>

**Figura 6.12:** Decodificación de código de barras lineal con *pyzbar*: imagen de entrada y datos extraídos.


Código de barras decodificado con éxito [EAN13]:
0000000000055


## 6.9 El MCTest como Estudio de Caso: del Prototipo al Sistema en Producción

Hasta este punto, las principales etapas del *pipeline* de procesamiento han sido presentadas y analizadas individualmente, incluyendo la corrección de la inclinación (*deskew*), la detección de marcadores, la rectificación por perspectiva y la lectura de *QRCodes*. Aunque este enfoque facilita la comprensión de cada técnica, las aplicaciones reales exigen la integración de estas etapas en un flujo de procesamiento único y consistente.

El **MCTest** constituye un ejemplo de esta integración. Desarrollado en la UFABC y disponible como software de código abierto, el sistema se utiliza desde 2012 en la corrección automatizada de evaluaciones, ofreciendo soporte para diferentes modelos de hojas de respuesta, plantillas individualizadas y generación automática de informes de desempeño (ZAMPIROLLI, 2023).

A partir de este punto, el enfoque deja de ser la implementación aislada de algoritmos y pasa a ser la organización de estos algoritmos en una aplicación completa. Además de la calidad de los métodos de procesamiento de imágenes, un sistema de esta naturaleza debe cumplir requisitos como robustez frente a diferentes condiciones de adquisición, facilidad de mantenimiento y capacidad de evolución hacia nuevas funcionalidades.

En las próximas secciones, se analizará el módulo de Visión Computacional del MCTest, implementado en el archivo `CVMCTest.py`. El objetivo es mostrar cómo los conceptos presentados a lo largo de este capítulo se combinan en un *pipeline* de procesamiento empleado en una aplicación real.

### 6.9.1 Obtención y Preparación del Módulo

El archivo `CVMCTest.py` integra el sistema MCTest y, en su versión original, depende de modelos, configuraciones y otros componentes del *framework* Django. Dado que estas dependencias no están disponibles en el entorno utilizado en este capítulo, el módulo debe adaptarse para ejecutarse de forma independiente.

Para ello, el archivo se obtiene directamente del repositorio del proyecto mediante la biblioteca `requests`. A continuación, se emplean comandos `sed` para eliminar las importaciones y dependencias específicas del entorno web, produciendo una versión autocontenida del módulo. Esta adaptación preserva la implementación de los algoritmos de Visión Computacional, permitiendo su ejecución y análisis sin la necesidad de instalar o configurar toda la infraestructura del sistema MCTest.

In [16]:
import requests
CVMCTest = requests.get(
    "https://raw.githubusercontent.com/fzampirolli/mctest/master/exam/CVMCTest.py"
    )
with open('CVMCTest.py', 'w') as writefile:
    writefile.write(CVMCTest.text)

Cada comando `sed` elimina importaciones específicas del entorno Django,
haciendo que el archivo `CVMCTest.py` sea utilizable de forma independiente en este capítulo.

In [17]:
# elimina lineas with "form django.", ...
!sed --in-place '/from django./d' CVMCTest.py
!sed --in-place '/from exam./d' CVMCTest.py
!sed --in-place '/from mctest./d' CVMCTest.py
!sed --in-place '/from student./d' CVMCTest.py
!sed --in-place '/from topic./d' CVMCTest.py
!sed --in-place '/from .models import VariationExam/d' CVMCTest.py

> ### 📝 🧠 ¿Por qué eliminar las dependencias de Django?
>
> En la implementación original, el archivo `CVMCTest.py` forma parte de una aplicación desarrollada con el *framework* Django y, por ello, importa modelos, configuraciones y otros componentes específicos de ese entorno. Como estos elementos no están disponibles en este capítulo, el módulo no puede importarse directamente.
>
> Los comandos `sed` eliminan únicamente esas dependencias, sin alterar las rutinas de Visión por Computador implementadas en el archivo. De esta manera, el módulo puede ejecutarse de forma independiente, preservando el comportamiento de los algoritmos presentados.
>
> Este procedimiento ilustra un principio importante de la ingeniería de software: separar la lógica de la aplicación de la infraestructura en la que está inserta, facilitando la reutilización, las pruebas y el estudio de componentes específicos.

### 6.9.2 Extracción del Área de Respuestas

Tras la lectura de la hoja, la función `getAnswerArea` ejecuta automáticamente las etapas de detección de los marcadores de referencia y corrección por perspectiva presentadas en las secciones anteriores. Como resultado, se obtiene una imagen que contiene únicamente la región destinada a las respuestas, alineada y con dimensiones estandarizadas.

Esta estandarización simplifica las etapas posteriores de procesamiento, ya que la ubicación de los campos de marcado pasa a ser conocida e independiente de la posición original de la hoja durante la digitalización.

La [Figura 6.13](#fig-06-mctest-img-original) presenta la hoja de respuestas original en escala de grises, mientras que la [Figura 6.14](#fig-06-mctest-answer-area) muestra la región de respuestas obtenida tras la aplicación de la función `getAnswerArea`.

In [18]:
import os
from pdf2image import convert_from_path
from skimage import data as skdata
import cv2
from morph import mm

file = "dados/provas_qrcode_EP.pdf"
MYFILES = 'extra02.qrcode'

if os.path.exists(file):
    pages = convert_from_path(file, 200)  # dpi 100=min 500=max
    numPAGES = 0
    for page in pages:
        myfile0 = MYFILES + '_p' + str(numPAGES) + '.png'
        page.save(myfile0)
        numPAGES += 1
        print(f"[INGESTA] Página convertida: {myfile0}")
    pages.clear()
    img_color = mm.read(myfile0)
    img_inicial = mm.gray(img_color)
else:
    print("[AVISO] Archivo 'dados/provas_qrcode.pdf' no encontrado.")
    print("        Usando imagen pública skimage.data.page() como sustituto.")
    img_inicial = skdata.page()

mm.show(img_inicial)


[INGESTA] Página convertida: extra02.qrcode_p0.png


[INGESTA] Página convertida: extra02.qrcode_p1.png


[INGESTA] Página convertida: extra02.qrcode_p2.png


<Figure size 826.5x1169.5 with 1 Axes>

**Figura 6.13:** Imagen de la hoja de respuestas en escala de grises cargada a partir del PDF rasterizado.


In [19]:
import CVMCTest
countPage = 0
img_getAnswerArea = CVMCTest.cvMCTest.getAnswerArea(img_inicial, countPage)
mm.show(img_getAnswerArea)

<Figure size 563x511 with 1 Axes>

**Figura 6.14:** Área de respuestas extraída por *getAnswerArea*: región rectificada que contiene los cuadros de marcado.


**Nota de compatibilidad:** las versiones recientes de NumPy (≥ 2.0) eliminaron el alias `np.int0`. Si `CVMCTest.py` utiliza ese tipo, el comando siguiente aplica la corrección directamente en el archivo antes de recargarlo:

In [20]:
!sed -i 's/box = np.int0(cv2.boxPoints(rect))/box = cv2.boxPoints(rect).astype(np.intp)/' \
    ./CVMCTest.py

In [21]:
import importlib
import CVMCTest

importlib.reload(CVMCTest)

<module 'CVMCTest' from '/home/fz/VSCode/pdi-vc/gen/quarto/py.es/cap06/CVMCTest.py'>

### 6.9.3 Segmentación del *QRCode*

Tras la extracción del área de respuestas, el MCTest realiza dos etapas preparatorias para la lectura de las marcaciones: la segmentación del *QRCode* y la localización de los cuadros que contienen las preguntas.

La función `segmentQRcode` aísla la región de la imagen correspondiente al *QRCode*, presentada en la [Figura 6.15](#fig-06-mctest-qrcode-seg2).. A continuación, esta región es procesada por la función `getQRCode`, responsable de su decodificación y de la extracción de los metadatos de la prueba.

En caso de que el *QRCode* no pueda ser decodificado, el procesamiento de la hoja continúa normalmente. Las respuestas del estudiante aún se leen y se registran en el archivo CSV de salida; solo la información obtenida a partir del *QRCode*, como la identificación de la prueba o del estudiante, permanece no disponible.

In [22]:
import CVMCTest
imgQRcode = CVMCTest.cvMCTest.segmentQRcode(img_getAnswerArea, countPage)
mm.show(imgQRcode)

<Figure size 450x450 with 1 Axes>

**Figura 6.15:** Región del *QRCode* aislada por *segmentQRcode* dentro del área de respuestas rectificada.


La función `CVMCTest.cvMCTest.getQRCode(img, countPage)` integra las etapas de segmentación y decodificación del *QRCode*. Internamente, utiliza `CVMCTest.cvMCTest.decodeQRcode(imgQRcode)` para interpretar la *cadena* hexadecimal codificada en el símbolo y construir el diccionario `qr`, además de retornar el indicador lógico `myFlagArea`, que informa si la lectura se realizó con éxito.

El diccionario `qr` reúne los metadatos de la prueba utilizados en las etapas subsiguientes de procesamiento. Sus principales campos son:

- **`date`:** identificador temporal de la prueba, compuesto por la fecha de generación y por un *timestamp* interno del MCTest.
- **`idClassroom`, `idExam` e `idStudent`:** identificadores del aula, de la prueba y del estudiante.
- **`term`:** período lectivo.
- **`stylesheet`:** hoja de estilo utilizada en la generación del formulario.
- **`var1` a `var5`:** cantidad de preguntas en cada nivel de dificultad.
- **`text`:** número de preguntas de desarrollo.
- **`answer`:** número de alternativas por pregunta.
- **`numquest`:** número total de preguntas.
- **`correct`** y **`dbtext`:** campos completados durante la corrección, que contienen la plantilla de respuestas e información adicional.
- **`variations`** y **`variant`:** información sobre las versiones de la prueba.

Estos metadatos identifican la prueba y al estudiante, permitiendo seleccionar la plantilla de respuestas correspondiente y parametrizar las etapas siguientes de lectura y corrección de las respuestas.

In [23]:
myFlagArea, qr = CVMCTest.cvMCTest.getQRCode(img_inicial, countPage)
myFlagArea, qr

(True,
 {'date': '260208-1770403541363',
  'idClassroom': '955',
  'idExam': '810',
  'idStudent': '448898',
  'term': '0',
  'stylesheet': '1',
  'var1': '50',
  'var2': '0',
  'var3': '0',
  'var4': '0',
  'var5': '0',
  'text': '0',
  'answer': '5',
  'numquest': 50,
  'correct': '',
  'dbtext': '',
  'variations': '0',
  'variant': '0'})

Estos metadatos permiten identificar la prueba, recuperar la plantilla de respuestas correspondiente y parametrizar las etapas subsiguientes de procesamiento.

Tras la decodificación del *código QR*, el procesamiento regresa al área de respuestas para localizar los recuadros que contienen las marcas del estudiante.

### 6.9.4 Ubicación de los Cuadros de Respuestas

La imagen producida por `getAnswerArea` contiene toda la región útil de la hoja, incluido el encabezado, donde se encuentra el *QRCode*, y los cuadros destinados a las respuestas. Como la lectura de las marcaciones utiliza únicamente esos cuadros, la región correspondiente al encabezado se descarta mediante el recorte `img_getAnswerArea[300:, :]`.

La [Figura 6.16](#fig-06-mctest-answer-area-crop) presenta esta región de interés. Aunque ese recorte sea posteriormente utilizado por la función `findSquares` para localizar los cuadros de respuestas, el MCTest realiza inicialmente la lectura del *QRCode*, pues este contiene los metadatos necesarios para identificar la prueba y configurar las etapas subsiguientes del procesamiento.

In [24]:
img_getAnswerArea_aux = img_getAnswerArea[300:,:]
mm.show(img_getAnswerArea_aux)

<Figure size 563x450 with 1 Axes>

**Figura 6.16:** Recorte inferior del área de respuestas, concentrando los cuadros de burbujas a ser segmentados.


La función `findSquares` recibe la imagen del área de respuestas y los metadatos almacenados en `qr`, devolviendo las coordenadas de los marcos que delimitan los grupos de preguntas.

Cada elemento de `rectSquares` contiene las coordenadas de los vértices superior izquierdo e inferior derecho de un marco de respuestas, que se utilizarán en la etapa de segmentación de las burbujas.

In [25]:
rectSquares = CVMCTest.cvMCTest.findSquares(qr,img_getAnswerArea, countPage)
rectSquares

[[[np.int64(395), np.int64(350)], [np.int64(950), np.int64(516)]],
 [[np.int64(394), np.int64(602)], [np.int64(951), np.int64(767)]]]

### 6.9.5 Lectura Automática de las Respuestas

Conocidos los metadatos de la prueba (`qr`) y las coordenadas de los marcos de respuestas (`rectSquares`), el MCTest identifica automáticamente las alternativas marcadas por el estudiante.

El siguiente código integra las etapas presentadas anteriormente. Para cada marco delimitado en `rectSquares`, las funciones `setColumns` y `setLines` estiman, respectivamente, el número de alternativas por pregunta y el número de preguntas a partir de la distribución espacial de las burbujas. A continuación, `segmentAnswers` determina la alternativa señalada en cada pregunta y `setAnswersOneLine` reúne los resultados de todos los marcos en el campo `qr['answers']`.

En el modo de operación adoptado en este capítulo, en el que el MCTest se ejecuta de forma independiente de su base de datos, el contenido de `qr['answers']` se compara con la plantilla de corrección almacenada en la primera página del archivo PDF, correspondiente al modelo de prueba sin enunciados utilizado en los ejemplos.

La [Figura 6.17](#fig-06-mctest-answers) presenta los marcos de respuestas procesados por el algoritmo, mientras que la salida del programa muestra el contenido final de `qr['answers']`.

In [26]:
testAnswers = []
if myFlagArea:
  
  imgQ_all = []

  for countSquare in range(len(rectSquares)):
      p1, p2 = rectSquares[countSquare]

      if True:
          imgQi = CVMCTest.cvMCTest.imgAnswers[p1[0]:p2[0], p1[1]:p2[1]]
          [NUM_COLUMNS, img] = CVMCTest.cvMCTest.setColumns(imgQi, countPage, countSquare)
          [NUM_LINES, img] = CVMCTest.cvMCTest.setLines(imgQi, countPage, countSquare)
          NUM_RESPOSTAS = NUM_COLUMNS
          NUM_QUESTOES = NUM_LINES

      imgQiNC = CVMCTest.cvMCTest.imgAnswers[p1[0]:p2[0], p1[1]:p2[1]]
      testAnswers.append(CVMCTest.cvMCTest.segmentAnswers(
          [imgQi, imgQiNC], countPage, countSquare, NUM_QUESTOES, qr

      ))

      imgQ_all.append(imgQiNC)

  qr = CVMCTest.cvMCTest.setAnswarsOneLine(testAnswers, qr)  
  # deja las respuestas de cada cuadro en una línea

mm.show(imgQ_all)
print(f"Respuestas leídas de las {len(qr['answers'].split(","))} preguntas: \
      \n{qr['answers'][:-19]}...")

<Figure size 2250x750 with 3 Axes>

**Figura 6.17:** Respuestas leídas automáticamente por MCTest tras la segmentación y clasificación de todas las burbujas.


Respuestas leídas de las 50 preguntas:       
C,A,A,C,E,D,E,C,A,E,B,B,A,D,A,B,C,B,E,D,B,D,A,C,E,B,A,A,B,B,C,A,C,A,C,A,B,C,C,C,...


> ### 💡 Conectando los Puntos
>
> El campo `qr['answers']` representa el resultado final del *pipeline* de Visión por Computadora presentado en este capítulo. Su obtención integra todas las etapas estudiadas, desde la rasterización del documento y la rectificación geométrica hasta la extracción del área de respuestas, la decodificación del *QRCode*, la localización de los cuadros y la identificación de las alternativas marcadas.
>
> Este *pipeline* ilustra la transición de un prototipo a un sistema en producción. En MCTest, los algoritmos de Visión por Computadora permanecen esencialmente los mismos; las principales diferencias se concentran en aspectos de ingeniería de software, como el manejo de excepciones, el soporte para diferentes modelos de formularios, la integración con la base de datos, la interfaz web y los mecanismos de auditoría y mantenimiento.
>
> En los experimentos de este capítulo, la primera página del archivo PDF contiene la clave de respuestas del examen, mientras que las páginas siguientes corresponden a las hojas de respuestas de los estudiantes. Después de obtener `qr['answers']`, MCTest compara automáticamente las respuestas leídas con la clave para calcular la puntuación de cada estudiante.
>
> En el uso completo del sistema, los resultados de la corrección se consolidan en un archivo CSV y se envían al profesor junto con un archivo comprimido que contiene información auxiliar para auditoría. Entre estos archivos se encuentran los recortes de las preguntas en las que se detectaron múltiples marcaciones u otras situaciones que requieren revisión manual. El profesor puede entonces inspeccionar estas imágenes, decidir la interpretación más adecuada y, si es necesario, actualizar el archivo CSV antes de la importación definitiva de las calificaciones.

## 6.10 Inspección Industrial Automatizada

La inspección visual automatizada es una aplicación de la Visión por Computadora en la que se analizan imágenes de piezas o productos para verificar el cumplimiento de criterios de calidad previamente definidos. En una línea de producción, las imágenes pueden obtenerse mediante cámaras u otros dispositivos de adquisición y procesarse automáticamente para identificar defectos, medir dimensiones o verificar la presencia de componentes.

La estrategia de inspección depende de las características del producto, del tipo de defecto de interés y de la disponibilidad de una imagen de referencia. En este capítulo se presentan dos enfoques clásicos:

- **Sustracción de imágenes:** compara la imagen de la pieza inspeccionada con una imagen de referencia considerada libre de defectos. Las regiones en las que la diferencia de intensidad supera un umbral se clasifican como posibles defectos. Este enfoque presupone que las imágenes están geométricamente alineadas y han sido adquiridas en condiciones similares de iluminación.

- **Análisis de textura:** utiliza características de la textura de la superficie para identificar regiones cuya apariencia difiere del patrón esperado, sin necesidad de una imagen de referencia. Este enfoque es adecuado para materiales que presentan una textura aproximadamente homogénea, como tejidos, papeles y superficies metálicas.

En las siguientes secciones, estas dos estrategias se ilustran mediante ejemplos construidos a partir de imágenes de la biblioteca `skimage.data`. El objetivo es presentar los principios de funcionamiento de cada enfoque en experimentos que el lector pueda reproducir íntegramente.

### 6.10.1 Referencias y *Datasets* Públicos

En aplicaciones de inspección industrial, el rendimiento de los algoritmos de detección de defectos se evalúa frecuentemente en *datasets* públicos, que proporcionan imágenes representativas y, en muchos casos, anotaciones de referencia (*ground truth*). En este capítulo, sin embargo, los ejemplos utilizan imágenes sintéticas derivadas de `skimage.data` (ver [Figura 6.18](#fig-06-industrial-defeito)), lo que permite reproducir todos los experimentos sin depender de bases de datos externas.

Para estudios más exhaustivos y comparación entre algoritmos, se destacan los siguientes *datasets* públicos:

- **MVTec *Anomaly Detection Dataset* (MVTec AD):** conjunto de imágenes de objetos y texturas, que contiene muestras sin defectos y con defectos, acompañadas de máscaras de segmentación a nivel de píxel para las imágenes anómalas (BERGMANN, 2019). Disponible en: <https://www.mvtec.com/company/research/datasets/mvtec-ad>.

- **Kolektor *Surface-Defect Dataset* (KolektorSDD):** conjunto de imágenes de componentes industriales con defectos superficiales anotados, utilizado en estudios de detección y segmentación de defectos (TABERNIK, 2020). Disponible en: <https://www.vicos.si/resources/kolektorsdd/>.

- **NEU *Surface Defect Database*:** conjunto de imágenes de superficies de acero laminado, organizado en seis categorías de defectos superficiales, empleado frecuentemente en la evaluación de métodos de clasificación y detección (SONG, 2013). Disponible en: <http://faculty.neu.edu.cn/songkechen/zh_CN/zdylm/263270/list/index.htm>.

In [27]:
import numpy as np
from skimage import data, color
from morph import mm

# Imagen de referencia (producto sin defecto)
product_color = data.coffee()
product_gray = color.rgb2gray(product_color)

# Inserción de defecto simulado: rayón oscuro de 10×100 px
defect_image = np.copy(product_gray)
defect_image[100:110, 200:300] = 0.1

# Detección por sustracción y umbralización
difference = np.abs(product_gray - defect_image)
defect_threshold = 0.15          # ajustable según la aplicación
detected_defect = (difference > defect_threshold).astype(np.uint8) * 255

mm.show(
    [product_gray, defect_image, detected_defect],
    titles=["Referencia", "Con defecto", "Defecto detectado"],
    cols=3,
    figsize=(12, 4)
)

status = "Defeito detectado." if detected_defect.any() else "Produto conforme."
print(status)

<Figure size 1800x600 with 3 Axes>

**Figura 6.18:** Detección de defecto por sustracción de imagen: producto de referencia, imagen con defecto simulado y máscara de anomalía detectada.


Defeito detectado.


> ### 📝 Sustracción de Imágenes: Registro Geométrico y Principio de Funcionamiento
>
> La sustracción de imágenes presupone que la imagen de inspección esté geométricamente alineada con la imagen de referencia. Diferencias de posicionamiento, rotación, escala o perspectiva producen regiones de diferencia que pueden confundirse con defectos.
>
> En aplicaciones con adquisición controlada, este alineamiento se obtiene durante la captura mediante plantillas mecánicas, cintas transportadoras y cámaras fijas, lo que permite comparar directamente imágenes sucesivas. Un ejemplo es la inspección de una caja de herramientas siempre posicionada en la misma orientación para verificar la ausencia de algún elemento.
>
> Cuando este control no es posible, se emplea el **registro de imágenes**, que estima una transformación geométrica para compensar diferencias de traslación, rotación, escala y, cuando sea necesario, perspectiva.
>
> Tras el registro, se realiza la comparación píxel a píxel entre las dos imágenes. En regiones sin alteraciones, las diferencias de intensidad tienden a ser cercanas a cero; donde existe un defecto, surgen diferencias locales que pueden destacarse mediante umbralización. La utilización de la diferencia absoluta permite detectar tanto defectos más claros como más oscuros que la referencia.
>
> El rendimiento del método depende principalmente de la calidad del alineamiento geométrico y de la elección del umbral utilizado para separar pequeñas variaciones de adquisición de las diferencias asociadas a los defectos.
>
> La [Figura 6.19](#fig-06-sim-06-industrial) ilustra el efecto del desalineamiento entre las imágenes y la importancia del registro geométrico antes de la aplicación de la sustracción.

In [28]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-06-industrial" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-06-industrial * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-06-industrial canvas { display: block; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; margin: 0 auto; }
  #sim-06-industrial button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-06-industrial button:hover { background: #e8dfcf; }
  #sim-06-industrial button.sim06_ind_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-06-industrial input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; margin: 4px 0; }
  .sim06_ind_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim06_ind_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim06_ind_grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 12px; }
  .sim06_ind_control_group { background: #ffffff; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; }
  .sim06_ind_label_wrap { display: flex; justify-content: space-between; font-size: 11px; font-weight: 700; color: #5e5a4a; }
  .sim06_ind_val { font-family: monospace; color: #26241d; }
  .sim06_ind_canvases { display: flex; gap: 12px; flex-wrap: wrap; justify-content: center; align-items: flex-start; }
  .sim06_ind_cv_wrap { flex: 1; min-width: 200px; background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 10px; text-align: center; }
  .sim06_ind_cv_label { font-size: 11px; color: #5e5a4a; margin-bottom: 6px; font-weight: 700; }
  .sim06_ind_status { font-size: 11px; font-weight: 700; padding: 10px 12px; border-radius: 8px; text-align: center; margin-top: 10px; transition: all 0.2s ease; border: 1px solid transparent; line-height: 1.4; }
  .sim06_ind_status.ok { background: #eafaf1; color: #04342C; border-color: #a3e4d7; }
  .sim06_ind_status.fail { background: #fdecea; color: #4A1B0C; border-color: #f5b7b1; }
  .sim06_ind_status.warn { background: #fef5e7; color: #412402; border-color: #f8c471; }
  .sim06_ind_reg_toggle { display: flex; gap: 6px; margin-top: 8px; }
  .sim06_ind_reg_toggle button { flex: 1; font-weight: 600; justify-content: center; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">⚙️ Simulador: Sustracción de Imágenes y Registro Geométrico</span>
  <span class="sim06_ind_pill">|Referencia − Inspección| &gt; Umbral</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim06_ind_panel">
    <div class="sim06_ind_grid">
      
      <div class="sim06_ind_control_group">
        <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:8px; border-bottom:1px solid #e9e3d3; padding-bottom:4px;">
          🔄 Desalineación de la Captura (Banda)
        </div>

        <div class="sim06_ind_label_wrap"><span>Traslación horizontal (Δx):</span><span id="sim06_ind_txVal" class="sim06_ind_val">0 px</span></div>
        <input type="range" id="sim06_ind_sliderTx" min="-8" max="8" step="1" value="0">

        <div class="sim06_ind_label_wrap"><span>Traslación vertical (Δy):</span><span id="sim06_ind_tyVal" class="sim06_ind_val">0 px</span></div>
        <input type="range" id="sim06_ind_sliderTy" min="-8" max="8" step="1" value="0">

        <div class="sim06_ind_label_wrap"><span>Rotación (θ):</span><span id="sim06_ind_rotVal" class="sim06_ind_val">0.0°</span></div>
        <input type="range" id="sim06_ind_sliderRot" min="-5" max="5" step="0.5" value="0">

        <div class="sim06_ind_reg_toggle">
          <button id="sim06_ind_btnModoDireto" class="sim06_ind_active">Sustracción Directa</button>
          <button id="sim06_ind_btnModoRegistro">Sustracción con Registro</button>
        </div>
      </div>

      <div class="sim06_ind_control_group" style="display:flex; flex-direction:column; justify-space-between;">
        <div>
          <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:8px; border-bottom:1px solid #e9e3d3; padding-bottom:4px;">
            🎛️ Parametrización del Inspector
          </div>
          <div class="sim06_ind_label_wrap"><span>Umbral de tolerancia (T):</span><span id="sim06_ind_thVal" class="sim06_ind_val">35</span></div>
          <input type="range" id="sim06_ind_sliderTh" min="10" max="100" step="5" value="35">
        </div>
        <div style="text-align:right; margin-top:12px;">
          <button id="sim06_ind_btnReset">↺ Reiniciar Simulación</button>
        </div>
      </div>

    </div>

    <div id="sim06_ind_msgStatus" class="sim06_ind_status ok">Producto conforme.</div>
  </div>

  <!-- Canvases de Exibição -->
  <div class="sim06_ind_canvases" style="margin-top:14px;">
    <div class="sim06_ind_cv_wrap">
      <div class="sim06_ind_cv_label">1. Referencia Estable</div>
      <canvas id="sim06_ind_cvRef" width="220" height="170"></canvas>
    </div>
    <div class="sim06_ind_cv_wrap">
      <div class="sim06_ind_cv_label" id="sim06_ind_inspLabel">2. Inspección (captura real)</div>
      <canvas id="sim06_ind_cvInsp" width="220" height="170"></canvas>
    </div>
    <div class="sim06_ind_cv_wrap">
      <div class="sim06_ind_cv_label" id="sim06_ind_diffLabel">3. Máscara de Anomalía</div>
      <canvas id="sim06_ind_cvDiff" width="220" height="170"></canvas>
    </div>
  </div>

  <!-- Explicação Teórica -->
  <div class="sim06_ind_panel" style="margin-top:14px;">
    <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:6px;">🧠 Desafío Pedagógico</div>
    <div style="font-size:11px; color:#5e5a4a; line-height:1.5;">
      Con el modo <strong>Sustracción Directa</strong> activo, utiliza los controles de traslación y rotación para simular pequeñas
      desalineaciones en la banda. Observa que variaciones de pocos píxeles o grados generan bordes fáciles de confundir con fallas reales.
      Aumentar el <strong>umbral de tolerancia</strong> para ignorar esos bordes reduce la sensibilidad, haciendo que el sistema quede ciego a defectos finos (como el rayón en el lado izquierdo).
      Al cambiar a <strong>Sustracción con Registro</strong>, la alineación se restaura antes de la diferencia, aislando con precisión el defecto real sin falsas alarmas.
    </div>
  </div>

</div>
</div>

<script>
(function() {
  function initSim06Industrial(root){
    if (!root || root.dataset.sim06IndustrialInit) return;
    root.dataset.sim06IndustrialInit = "1";

    const W = 220, H = 170;

    const cvRef  = root.querySelector('#sim06_ind_cvRef');
    const cvInsp = root.querySelector('#sim06_ind_cvInsp');
    const cvDiff = root.querySelector('#sim06_ind_cvDiff');

    const ctxRef  = cvRef.getContext('2d');
    const ctxInsp = cvInsp.getContext('2d');
    const ctxDiff = cvDiff.getContext('2d');

    let modoRegistro = false;

    function drawComponent(ctx, hasDefect, tx, ty, rotDeg) {
      ctx.clearRect(0, 0, W, H);
      ctx.fillStyle = '#fafaf7';
      ctx.fillRect(0, 0, W, H);

      ctx.save();
      ctx.translate(W / 2 + tx, H / 2 + ty);
      ctx.rotate(rotDeg * Math.PI / 180);

      ctx.fillStyle = '#bdc3c7';
      ctx.strokeStyle = '#2c3e50';
      ctx.lineWidth = 3;

      ctx.beginPath();
      ctx.rect(-60, -45, 120, 90);
      ctx.fill(); ctx.stroke();

      ctx.beginPath();
      ctx.arc(0, 0, 35, 0, Math.PI * 2);
      ctx.fillStyle = '#ecf0f1';
      ctx.fill(); ctx.stroke();

      ctx.fillStyle = '#fafaf7';
      const holes = [[-40, -25], [40, -25], [-40, 25], [40, 25]];
      holes.forEach(([hx, hy]) => {
        ctx.beginPath();
        ctx.arc(hx, hy, 8, 0, Math.PI * 2);
        ctx.fill(); ctx.stroke();
      });

      ctx.beginPath();
      ctx.arc(0, 0, 12, 0, Math.PI * 2);
      ctx.fillStyle = '#2c3e50';
      ctx.fill();

      if (hasDefect) {
        ctx.save();
        ctx.strokeStyle = '#17202a';
        ctx.lineWidth = 3;
        ctx.lineCap = 'round';
        ctx.beginPath();
        ctx.moveTo(-45, 5);
        ctx.lineTo(-25, -15);
        ctx.stroke();
        ctx.restore();
      }

      ctx.restore();
    }

    function update() {
      const tx  = parseInt(root.querySelector('#sim06_ind_sliderTx').value);
      const ty  = parseInt(root.querySelector('#sim06_ind_sliderTy').value);
      const rot = parseFloat(root.querySelector('#sim06_ind_sliderRot').value);
      const th  = parseInt(root.querySelector('#sim06_ind_sliderTh').value);

      root.querySelector('#sim06_ind_txVal').textContent  = (tx > 0 ? '+' : '') + tx + ' px';
      root.querySelector('#sim06_ind_tyVal').textContent  = (ty > 0 ? '+' : '') + ty + ' px';
      root.querySelector('#sim06_ind_rotVal').textContent = (rot > 0 ? '+' : '') + rot.toFixed(1) + '°';
      root.querySelector('#sim06_ind_thVal').textContent  = th;

      drawComponent(ctxRef, false, 0, 0, 0);

      const tx_insp  = modoRegistro ? 0 : tx;
      const ty_insp  = modoRegistro ? 0 : ty;
      const rot_insp = modoRegistro ? 0 : rot;
      drawComponent(ctxInsp, true, tx_insp, ty_insp, rot_insp);

      const dataRef  = ctxRef.getImageData(0, 0, W, H);
      const dataInsp = ctxInsp.getImageData(0, 0, W, H);
      const dataDiff = ctxDiff.createImageData(W, H);

      let defectPixelsDetected = 0;

      for (let i = 0; i < dataRef.data.length; i += 4) {
        const diff = Math.abs(dataRef.data[i] - dataInsp.data[i]);

        if (diff > th) {
          dataDiff.data[i]   = 192; // R
          dataDiff.data[i+1] = 57;  // G
          dataDiff.data[i+2] = 43;  // B (#c0392b)
          dataDiff.data[i+3] = 255;
          defectPixelsDetected++;
        } else {
          dataDiff.data[i]   = 250;
          dataDiff.data[i+1] = 250;
          dataDiff.data[i+2] = 247; // #fafaf7
          dataDiff.data[i+3] = 255;
        }
      }
      ctxDiff.putImageData(dataDiff, 0, 0);

      root.querySelector('#sim06_ind_inspLabel').textContent = modoRegistro
        ? '2. Inspeção Registrada (alinhada)'
        : '2. Inspeção (captura real)';
      root.querySelector('#sim06_ind_diffLabel').textContent = modoRegistro
        ? '3. Máscara de Anomalia (com registro)'
        : '3. Máscara de Anomalia (direta)';

      const msgBox = root.querySelector('#sim06_ind_msgStatus');
      const hasMisalignment = (tx !== 0 || ty !== 0 || rot !== 0);
      const misalignmentVisible = hasMisalignment && !modoRegistro;

      if (defectPixelsDetected > 0) {
        if (misalignmentVisible && defectPixelsDetected > 150) {
          msgBox.className = 'sim06_ind_status fail';
          msgBox.innerHTML = '❌ Falsos positivos detectados! O desalinhamento mecânico gerou anomalias fantasmas nas bordas (' + defectPixelsDetected + ' px comprometidos).';
        } else if (misalignmentVisible) {
          msgBox.className = 'sim06_ind_status warn';
          msgBox.innerHTML = '⚠️ Anomalia localizada, mas parte da diferença ainda vem do desalinhamento (' + defectPixelsDetected + ' px afetados).';
        } else {
          msgBox.className = 'sim06_ind_status fail';
          msgBox.innerHTML = '⚠️ Anomalia localizada! Defeito superficial detectado na estrutura interna (' + defectPixelsDetected + ' px afetados).';
        }
      } else {
        if (misalignmentVisible) {
          msgBox.className = 'sim06_ind_status warn';
          msgBox.innerHTML = '✓ Conforme sob risco. O limiar alto omitiu as bordas desalinhadas, mas tornou o sistema cego para defeitos reais.';
        } else {
          msgBox.className = 'sim06_ind_status ok';
          msgBox.innerHTML = '✓ Produto conforme. Sistema corretamente registrado em nível geométrico.';
        }
      }
    }

    ['#sim06_ind_sliderTx', '#sim06_ind_sliderTy', '#sim06_ind_sliderRot', '#sim06_ind_sliderTh'].forEach(sel => {
      root.querySelector(sel).addEventListener('input', update);
    });

    const btnDireto   = root.querySelector('#sim06_ind_btnModoDireto');
    const btnRegistro = root.querySelector('#sim06_ind_btnModoRegistro');

    btnDireto.addEventListener('click', () => {
      modoRegistro = false;
      btnDireto.classList.add('sim06_ind_active');
      btnRegistro.classList.remove('sim06_ind_active');
      update();
    });

    btnRegistro.addEventListener('click', () => {
      modoRegistro = true;
      btnRegistro.classList.add('sim06_ind_active');
      btnDireto.classList.remove('sim06_ind_active');
      update();
    });

    root.querySelector('#sim06_ind_btnReset').addEventListener('click', function() {
      root.querySelector('#sim06_ind_sliderTx').value = 0;
      root.querySelector('#sim06_ind_sliderTy').value = 0;
      root.querySelector('#sim06_ind_sliderRot').value = 0;
      root.querySelector('#sim06_ind_sliderTh').value = 35;
      modoRegistro = false;
      btnDireto.classList.add('sim06_ind_active');
      btnRegistro.classList.remove('sim06_ind_active');
      update();
    });

    update();
  }

  function tryInitSim06Industrial(){
    var root = document.getElementById('sim-06-industrial');
    if (root) initSim06Industrial(root); else setTimeout(tryInitSim06Industrial, 200);
  }
  tryInitSim06Industrial();
})();
</script>
</div>
""")

**Figura 6.19:** Simulador interactivo de inspección industrial por sustracción de imágenes: controle las distorsiones geométricas de desalineación (registro) y el umbral de detección para observar el impacto en los falsos positivos.


<figure id="fig-06-sim-06-industrial">
  <img src="imagens/fig-06-sim-06-industrial.png" alt=" Simulador interactivo de inspección industrial por sustracción de imágenes: controle las distorsiones geométricas de desalineación (registro) y el umbral de detección para observar el impacto en los falsos positivos. " style="max-width:80%" />
  <figcaption><strong>Figura 6.19:</strong>  Simulador interactivo de inspección industrial por sustracción de imágenes: controle las distorsiones geométricas de desalineación (registro) y el umbral de detección para observar el impacto en los falsos positivos. </figcaption>
</figure>

### 6.10.2 Detección de Defectos mediante Análisis de Textura

En aplicaciones donde no existe una imagen de referencia, la detección de defectos puede basarse en las características de la textura de la superficie. En este caso, se busca identificar regiones cuya apariencia difiere del patrón predominante del material.

En este ejemplo, se utiliza la **varianza local** como medida de heterogeneidad. Para cada posición de la imagen, se calcula la varianza de los niveles de intensidad en una vecindad de dimensiones fijas. Las regiones con baja varianza tienden a presentar una textura más uniforme, mientras que alteraciones locales, como rayones, manchas o imperfecciones, pueden producir valores más elevados de esta medida.

La [Figura 6.20](#fig-06-industrial-textura) ilustra este procedimiento utilizando la imagen `skimage.data.brick()`. Inicialmente, se calcula el mapa de varianza local mediante una ventana deslizante. A continuación, se aplica una umbralización para resaltar las regiones cuya varianza excede el valor especificado, identificando posibles áreas de interés para la inspección.

#### Modelado Matemático

Considérese una imagen en niveles de gris representada por

$$
f:\Omega\subset\mathbb{Z}^2\rightarrow\mathbb{R},
$$

donde $\Omega$ es el dominio de la imagen y $f(x,y)$ representa la intensidad del píxel en las coordenadas $(x,y)$. En imágenes de 8 bits, estas intensidades pertenecen al intervalo $[0,255]$. En este ejemplo, sin embargo, fueron normalizadas al intervalo $[0,1]$, sin alterar el funcionamiento del algoritmo.

Para cada posición de la imagen, se considera una vecindad cuadrada $W_{x,y}$ de dimensión $15\times15$ píxeles.

La media local está dada por

$$
\mu(x,y)=
\frac{1}{|W_{x,y}|}
\sum_{(u,v)\in W_{x,y}}
f(u,v),
$$

y la varianza local se calcula mediante

$$
\sigma^2(x,y) =
\frac{1}{|W_{x,y}|}
\sum_{(u,v)\in W_{x,y}}
f(u,v)^2 -
\mu(x,y)^2.
$$

En la implementación siguiente, estas dos medias se obtienen mediante la función `cv2.blur`,

```python
mean  = cv2.blur(img, (15,15))
mean2 = cv2.blur(img**2, (15,15))
var   = mean2 - mean**2
```

A continuación, se calcula la diferencia entre los mapas de varianza de la imagen de referencia y de la imagen inspeccionada,

$$
D(x,y)=
\left|
\sigma_d^2(x,y)-\sigma_r^2(x,y)
\right|,
$$

donde $\sigma_r^2(x,y)$ y $\sigma_d^2(x,y)$ son, respectivamente, las varianzas locales de la imagen de referencia y de la imagen que contiene el defecto. Tras la normalización del mapa $D(x,y)$, se aplica una umbralización para obtener la máscara de las posibles anomalías.

In [29]:
import numpy as np
import cv2
from skimage import data as skdata
from morph import mm

# Imagen de textura uniforme (ladrillo)
texture = skdata.brick().astype(np.float32) / 255.0

# Inserción de defecto sintético: mancha clara 20×80 px
texture_defect = np.copy(texture)
texture_defect[60:80, 80:160] = 0.95

# Mapa de varianza local (ventana 15×15)
def variancia_local(img, ksize=15):
    img_f = img.astype(np.float32)
    mean  = cv2.blur(img_f, (ksize, ksize))
    mean2 = cv2.blur(img_f ** 2, (ksize, ksize))
    return np.clip(mean2 - mean ** 2, 0, None)

var_ref    = variancia_local(texture)
var_defect = variancia_local(texture_defect)
diff_var   = np.abs(var_defect - var_ref)

# Normaliza y umbraliza
diff_norm = (diff_var / diff_var.max() * 255).astype(np.uint8)
_, mask   = cv2.threshold(diff_norm, 30, 255, cv2.THRESH_BINARY)

mm.show(
    [texture, texture_defect, diff_norm, mask],
    titles=["Textura original", "Con defecto", "Δ varianza local", "Anomalía detectada"],
    cols=4,
    figsize=(16, 4)
)

status = "Defeito de textura detectado." if mask.any() else "Superfície conforme."
print(status)


<Figure size 2400x600 with 4 Axes>

**Figura 6.20:** Detección de heterogeneidad de textura: mapa de varianza local y máscara de anomalía.


Defeito de textura detectado.


> ### 📝 🧠 ¿Por qué funciona? — Análisis de Textura
>
> La varianza local mide la dispersión de las intensidades en una vecindad de la imagen. En regiones cuya textura permanece uniforme, esta medida tiende a variar poco. Cuando un defecto modifica el patrón de la superficie, la distribución de las intensidades también se altera, produciendo diferencias en la varianza local.
>
> En este capítulo, la detección se realiza comparando los mapas de varianza de la imagen de referencia y de la imagen con defecto. Tras la normalización, se aplica una umbralización para resaltar las regiones donde esta diferencia supera un valor especificado.
>
> Los principales parámetros del método son el tamaño de la ventana utilizada en el cálculo de la varianza y el umbral empleado en la segmentación. Ventanas más pequeñas son más sensibles a detalles finos, mientras que ventanas más grandes producen mapas más suaves y pueden reducir la respuesta a defectos de pequeñas dimensiones.

## 6.11 Resumen

En este capítulo se presentaron métodos de Visión por Computador aplicados al análisis de documentos y a la inspección visual automatizada. Las principales técnicas estudiadas fueron:

- **Preprocesamiento de documentos:** aplicación de normalización de fondo, ecualización adaptativa (CLAHE) y umbralización por Otsu para reducir los efectos de iluminación no uniforme y mejorar la segmentación del texto.

- **Reconocimiento óptico de caracteres (OCR):** conversión de imágenes de documentos en texto codificado mediante el Tesseract OCR, evidenciando la influencia del preprocesamiento en la calidad del reconocimiento.

- **Traducción automática:** aplicación de técnicas de procesamiento de lenguaje natural para traducir el texto obtenido por el OCR.

- **Rectificación geométrica de documentos:** utilización del detector de bordes de Canny, de la Transformada de Hough y de transformaciones proyectivas para corregir la perspectiva de documentos digitalizados.

- **Localización y rectificación de formularios:** empleo de operaciones morfológicas, análisis de contornos y transformación de perspectiva para identificar marcadores de referencia y extraer automáticamente regiones de interés.

- **Lectura de códigos bidimensionales y unidimensionales:** detección y decodificación de *QRCodes* y códigos de barras para identificación automática de documentos y metadatos.

- **Reconocimiento óptico de marcas (OMR):** lectura automatizada de formularios y hojas de respuestas, ilustrada por un estudio de caso del sistema MCTest.

- **Inspección industrial:** detección de defectos por comparación con una imagen de referencia y por análisis de la varianza local.

A lo largo del capítulo, los algoritmos se implementaron y evaluaron con imágenes de la biblioteca `skimage.data` y con documentos reales, permitiendo reproducir los experimentos presentados.

### Próximos Pasos

Los métodos presentados en este capítulo muestran cómo técnicas de Procesamiento Digital de Imágenes, Visión por Computador y Procesamiento de Lenguaje Natural pueden integrarse en *pipelines* para el análisis automatizado de documentos.

En el próximo capítulo se estudiarán técnicas de **extracción de características** y **reconocimiento de patrones**, con énfasis en descriptores capaces de representar imágenes mediante atributos numéricos para comparación, clasificación y reconocimiento automático. Estos conceptos constituyen la base para los capítulos dedicados al aprendizaje automático y al aprendizaje profundo aplicados a la Visión por Computador.

## 6.12 🤖 Uso de Gemini Notebook como Tutor Complementario

Como apoyo al estudio de este capítulo, se recomienda la utilización del **Gemini Notebook** como tutor complementario. La herramienta emplea modelos de inteligencia artificial para responder preguntas, elaborar resúmenes y explicar conceptos con base en los documentos proporcionados como fuente de consulta, permitiendo al estudiante revisar el contenido de forma interactiva.

> ### ❗ 🎓 Estudia con el Tutor Inteligente
>
> [🚀 ACCEDER A Gemini Notebook: CAPÍTULO 06](https://notebooklm.google.com/notebook/835ec2a8-dbc3-46c4-8c2c-2040af3754a4)
>
> #### 🌐 Idioma y Lenguaje de Programación
>
> El proyecto de este capítulo en Gemini Notebook fue construido únicamente con el texto en **portugués** y los ejemplos de código en **Python**. Si estás estudiando desde la edición en inglés o francés, o siguiendo la ruta en C++, las respuestas del tutor pueden no corresponder exactamente a la versión que estás leyendo.
>
> #### ⚠️ Uso Crítico de las Respuestas
>
> Las respuestas generadas por Gemini Notebook se producen automáticamente mediante un modelo de inteligencia artificial y pueden contener omisiones o imprecisiones. Por este motivo, deben utilizarse como material de apoyo, y no como sustituto del estudio del capítulo.
>
> Siempre que surjan dudas, consulta el texto de este libro, ejecuta los ejemplos presentados y, cuando sea necesario, complementa la consulta con libros, artículos científicos y otras fuentes académicas confiables.

## 6.13 Lista de Ejercicios

Los siguientes ejercicios consolidan los conceptos presentados en este capítulo mediante adaptaciones, experimentos y extensiones de los algoritmos desarrollados a lo largo del texto.

1. **(10%)** Investigue la influencia del ángulo de inclinación en la etapa de *deskew*. Genere versiones rotadas de la imagen `skimage.data.page()` para ángulos entre $-10^\circ$ y $10^\circ$, aplique el algoritmo presentado en el capítulo y compare el ángulo estimado con el ángulo utilizado en la rotación. Presente los resultados en una tabla y discuta la precisión del método.

2. **(15%)** Aplique normalización de fondo y CLAHE (utilizando al menos tres combinaciones de `clipLimit` y `tileGridSize`) a la imagen `skimage.data.page()` degradada artificialmente con gradiente de iluminación y sombra lateral. Segmente cada versión mediante el método de Otsu y compare los resultados utilizando el número de componentes conexos espurios y la métrica IoU en relación con una máscara de referencia construida manualmente.

3. **(15%)** Investigue la sensibilidad del filtrado por circularidad, $C=\frac{4\pi A}{P^2},$ en la detección de los marcadores circulares. Utilizando el simulador de la [Figura 6.9](#fig-06-sim-06-circularidade), genere discos sintéticos con ruido geométrico creciente y evalúe los umbrales $C\in\{0{,}5,\ 0{,}6,\ 0{,}7,\ 0{,}8\}$. Presente una tabla que relacione el umbral con el número de falsos positivos y falsos negativos y discuta el compromiso entre sensibilidad y especificidad.

4. **(15%)** A partir de los cuatro marcadores detectados, implemente la rectificación por perspectiva utilizando `cv2.getPerspectiveTransform` y `cv2.warpPerspective`. A continuación, perturbe artificialmente las coordenadas de los puntos de control con ruido gaussiano de desviación estándar $\sigma\in\{1,3,5\}$ píxeles y evalúe el error de reproyección obtenido tras la homografía inversa.

5. **(15%)** Adapte el *pipeline* de adquisición y rectificación desarrollado en este capítulo para procesar documentos que contengan códigos de barras lineales en sustitución de los *QRCodes*. Rasterice el PDF con `pdf2image` a 300 DPI, aplique el *deskew*, decodifique el símbolo con `pyzbar` y presente la imagen rectificada junto con la secuencia de caracteres obtenida.

6. **(15%)** Extienda la lectura de burbujas del MCTest para identificar tres situaciones: **OK**, **BLANCO** (ninguna alternativa marcada) y **DOBLE MARCA** (dos o más alternativas por encima de un umbral de relleno). Evalúe al menos tres valores de dicho umbral, presente los resultados en un `pandas.DataFrame` y discuta su influencia en la clasificación de las respuestas.

7. **(15%)** Construya un *pipeline* de inspección industrial que combine sustracción de imágenes y análisis de varianza local de la textura sobre un conjunto de imágenes sintéticas que contengan defectos simulados. Para cada imagen, genere una máscara de referencia (*ground truth*), calcule la métrica IoU (*Intersection over Union*) de los dos enfoques para diferentes umbrales de decisión y presente los resultados en tablas y visualizaciones producidas con `mm.show`.

8. **(Bônus – 10%)** Implemente manualmente la estimación del ángulo de inclinación sin utilizar `cv2.HoughLines` ni `cv2.HoughLinesP`. A partir del mapa de bordes obtenido por el detector de Canny, construya el acumulador de la Transformada de Hough, $\rho=x\cos\theta+y\sin\theta,$ para $\theta\in[-45^\circ,45^\circ]$, identifique los máximos del acumulador y estime la inclinación mediante la mediana de las rectas detectadas. Compare los resultados con la implementación de OpenCV y discuta la influencia de rectas espurias en la estimación final.

## Referencias del Capítulo

La fundamentación teórica y los estudios de caso presentados en este capítulo se apoyan en las siguientes referencias:

- Gonzalez (2018), para los fundamentos de detección de bordes, umbralización, segmentación, operaciones morfológicas, reconocimiento óptico de caracteres y transformaciones geométricas aplicadas al análisis de documentos.

- Szeliski (2022), para la Transformada de Hough, el registro y alineamiento de imágenes, las transformaciones proyectivas (homografías) y los fundamentos de la inspección visual automatizada.

- Bradski (2008), para la utilización de la biblioteca OpenCV en las etapas de detección de bordes, Transformada de Hough, transformaciones geométricas, análisis de contornos y decodificación de *códigos QR*.

- Smith (2007) y Smith (2013), para la arquitectura, el funcionamiento y la evolución del mecanismo de reconocimiento óptico de caracteres **Tesseract OCR**, empleado en los ejemplos de OCR presentados en este capítulo.

- Bahdanau (2015) y Vaswani (2017), para los fundamentos de la traducción automática basada en redes neuronales, incluidos los mecanismos de atención y las arquitecturas *transformer*.

- Zampirolli (2023), para la descripción del sistema MCTest, utilizado como estudio de caso de un *pipeline* completo para la lectura y corrección automatizada de hojas de respuestas.

- Bergmann (2019), Tabernik (2020) y Song (2013), para los *conjuntos de datos* públicos de inspección industrial **MVTec AD**, **KolektorSDD** y **NEU Surface Defect Database**, utilizados como referencia para la evaluación y comparación de algoritmos de detección de defectos.

------------------------------------------------------------------------


<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.es/cap06/cap06.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 6.14 💻 Parte Práctica con Ejercicios de Programación

Los ejercicios de programación (EP) de esta sección complementan los conceptos presentados a lo largo del Capítulo 6 mediante la implementación de algoritmos relacionados con la inspección industrial y el análisis de documentos. El objetivo es consolidar los fundamentos estudiados, reproduciendo, a escala reducida, etapas de un *pipeline* típico de Visión por Computadora.

A diferencia de los capítulos anteriores, cuyos ejercicios enfatizaban operaciones más directamente relacionadas con los datos de imagen, los EP de este capítulo se centran en las **magnitudes intermedias** producidas durante el procesamiento, como áreas, perímetros, circularidad, ángulos de rectas, grados de relleno de burbujas, mapas de varianza y mapas de diferencia. Este enfoque permite comprender y validar cada etapa del *pipeline* de forma independiente, sin depender de bibliotecas especializadas para la adquisición de imágenes, detección de marcadores o decodificación de códigos — con excepción del ejercicio de cierre del capítulo (EP06_08), que intencionalmente introduce el uso de OpenCV para la segmentación y la decodificación real de un *QRCode*, cerrando el ciclo entre los conceptos teóricos y las herramientas empleadas en la práctica.

Los ejercicios siguen la misma secuencia conceptual del capítulo, en orden creciente de complejidad. Inicialmente, se abordan métricas de evaluación de segmentación, utilizadas para cuantificar la calidad de máscaras binarias. A continuación, se estudian criterios geométricos para la selección de marcadores, clasificación de marcas en formularios y estimación de la inclinación de documentos mediante la Transformada de Hough. En la parte final, los ejercicios exploran la normalización de iluminación, la detección de defectos por análisis de textura y la integración entre registro geométrico y sustracción de imágenes en un *pipeline* simplificado de inspección industrial.

Cada ejercicio representa una etapa aislada de un sistema real de Visión por Computadora, permitiendo validar individualmente conceptos que, en aplicaciones industriales, se combinan en un único *pipeline* de inspección.

### 🗺️ Leyenda de Dificultad

| Nivel | Significado | EP |
|:---:|---|---|
| 🟢 | Muy fácil / fácil — implementación de un único concepto o algoritmo simple | EP06_01, EP06_02 |
| 🟡 | Fácil–medio — tratamiento de múltiples casos o utilización de criterios estadísticos simples | EP06_03, EP06_04 |
| 🟠 | Medio — procesamiento matricial punto a punto | EP06_05 |
| 🔴 | Difícil — procesamiento matricial con operaciones en vecindad (ventana deslizante) | EP06_06 |
| 🟣 | Muy difícil — integración de múltiples etapas de un *pipeline* de Visión por Computadora | EP06_07 |
| ⚫ | Especial — uso de biblioteca especializada (`cv2`) para segmentación geométrica y decodificación real de código de barras/QRCode | EP06_08 |

> ### ❗ Directrices para la Resolución de los Ejercicios de Programación
>
> Salvo indicación en contrario, todos los ejercicios utilizan la convención de coordenadas matriciales `[fila][columna]`, con origen en $(0,0)$ en la esquina superior izquierda de la imagen.
>
> Cuando sea necesario realizar un redondeo numérico, se debe utilizar el redondeo estándar al entero más cercano (*round half away from zero*, con `np.floor(img + 0.5)`). Las comparaciones con umbrales (por ejemplo, circularidad, varianza, diferencia de intensidad o grado de relleno) deben considerarse **estrictas** (`>`), excepto cuando el enunciado especifique explícitamente otro criterio.
>
> Cada ejercicio ha sido elaborado para enfatizar un concepto específico presentado en el capítulo. Se recomienda implementar inicialmente la solución de forma directa y, solo después de su validación, buscar alternativas más eficientes o más generales.

### 🎯 Objetivo de este Cuaderno

Este cuaderno ha sido elaborado para apoyar el desarrollo, la validación y las pruebas de las soluciones de los **Ejercicios de Programación (EPs)** en un entorno interactivo, como Google Colab o Jupyter Notebook. Tras verificar el funcionamiento de la implementación con los casos de prueba presentados, el código puede ser enviado a Moodle para la evaluación oficial.

#### *Download*

Ejecute la celda siguiente para obtener los archivos `morph.py` y `testsuite.py`, utilizados por los ejercicios de este capítulo.

In [30]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

✅ Entorno listo. Morph: 1.1.9 | OpenCV: 5.0.0 | TestSuite: 1.1.2


#### Ejecutando las pruebas

Tras implementar la solución, ejecute `TestSuite("EP06_01.extensión").run()` en una nueva celda, reemplazando `extensión` por el lenguaje utilizado (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). El sistema obtiene automáticamente los casos de prueba del repositorio del curso, ejecuta el programa y presenta el resultado de la evaluación.

En Python, también es posible probar la solución directamente a partir de una *cadena*, sin necesidad de guardar el código en un archivo. Para ello, almacene el programa en una variable y utilice el método `run_code`:

```python
codigo = """
# 6 ... su código aquí ...
"""

TestSuite("EP06_01").run_code(codigo)
```

### 6.0.1 EP06_01 🟢 Evaluación de Segmentación por IoU (*Intersection over Union*)

A lo largo de este capítulo, diversas etapas del *pipeline* producen **máscaras binarias**, como en la segmentación de documentos, la localización de *QRCodes* y la detección de defectos. Para evaluar objetivamente la calidad de estas segmentaciones, es necesario compararlas con una máscara de referencia (*ground truth*).

Una de las métricas más utilizadas para este fin es la **IoU** (*Intersection over Union*, o Intersección sobre Unión), definida como la razón entre el área de intersección y el área de unión de dos máscaras binarias. Cuanto mayor sea el valor de la IoU, mayor será la concordancia entre la segmentación producida por el algoritmo y la referencia.

#### 6.0.1.1 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (número de filas) y $C$ (número de columnas).
2. **Máscara de referencia:** Leer los $L \times C$ elementos binarios (0 o 1) de la matriz `ref`.
3. **Máscara predicha:** Leer los $L \times C$ elementos binarios (0 o 1) de la matriz `pred`.
4. **Intersección:** Contar el número de posiciones $(i,j)$ para las cuales `ref[i][j] = 1` y `pred[i][j] = 1`.
5. **Unión:** Contar el número de posiciones $(i,j)$ para las cuales `ref[i][j] = 1` o `pred[i][j] = 1`.
6. **Caso degenerado:** Si la unión es igual a $0$, definir $\mathrm{IoU}=1{,}0$, ya que ambas máscaras están vacías.
7. **Cálculo:** Si la unión es mayor que cero, calcular

$$
\mathrm{IoU}=
\frac{|\mathrm{Interseccion}|}
{|\mathrm{Union}|}.
$$

8. **Clasificación:** Determinar la clasificación cualitativa utilizando el valor de IoU **antes** del redondeo.
9. **Redondeo:** Mostrar la IoU con cuatro decimales.
10. **Salida:** Imprimir, en este orden, la intersección, la unión, la IoU y la clasificación.

#### 6.0.1.2 📌 Restricciones Computacionales

- Si la unión es igual a $0$, no se debe realizar la división; la IoU debe definirse como $1{,}0$.
- Los rangos de clasificación utilizan comparaciones no estrictas ($\geq$).
- La clasificación debe realizarse utilizando el valor de la IoU en precisión completa, antes del redondeo para la visualización.

#### 6.0.1.3 🧠 Fundamentación Teórica

La IoU se define por

$$
\mathrm{IoU}=
\frac{|R\cap P|}
{|R\cup P|},
$$

donde:

- $R$ representa el conjunto de píxeles que pertenecen a la máscara de referencia;
- $P$ representa el conjunto de píxeles que pertenecen a la máscara predicha;
- $|R\cap P|$ corresponde al número de píxeles que pertenecen simultáneamente a ambas máscaras;
- $|R\cup P|$ corresponde al número de píxeles que pertenecen al menos a una de las máscaras.

| Rango de IoU | Clasificación | Interpretación |
|---|---|---|
| $\mathrm{IoU}\geq0{,}90$ | `EXCELENTE` | Concordancia muy elevada entre las máscaras. |
| $0{,}70\leq\mathrm{IoU}<0{,}90$ | `BUENO` | Pequeñas diferencias entre las máscaras. |
| $0{,}50\leq\mathrm{IoU}<0{,}70$ | `ACEPTABLE` | Concordancia parcial entre las máscaras. |
| $\mathrm{IoU}<0{,}50$ | `MALO` | Baja concordancia entre las máscaras. |

La IoU depende únicamente de la superposición entre las máscaras y, por lo tanto, es independiente del tamaño de la imagen.

#### 6.0.1.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

- Línea 1: entero $L$.
- Línea 2: entero $C$.
- Siguientes $L$ líneas: elementos binarios (0 o 1) de la matriz `ref`.
- Siguientes $L$ líneas: elementos binarios (0 o 1) de la matriz `pred`.

**Salida:**

- Línea 1: `Interseccion: X`
- Línea 2: `Union: Y`
- Línea 3: `IoU: Z`
- Línea 4: `Clasificacion: NOMBRE`

El valor de `IoU` debe imprimirse con cuatro decimales.

#### 6.0.1.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 2<br>2<br>1 1<br>0 0<br>1 0<br>0 0 | Interseccion: 1<br>Union: 2<br>IoU: 0.5000<br>Clasificacion: ACEPTABLE | La mitad de la región de referencia fue segmentada correctamente. |
| 2<br>2<br>0 0<br>0 0<br>0 0<br>0 0 | Interseccion: 0<br>Union: 0<br>IoU: 1.0000<br>Clasificacion: EXCELENTE | Ambas máscaras están vacías; por convención, $\mathrm{IoU}=1{,}0$. |

In [31]:
# @title { display-mode: "form" }
from IPython.display import HTML

HTML("""
<div id="sim-ep0601" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0601 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0601 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0601 button:hover { background: #e8dfcf; }
  #sim-ep0601 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0601_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0601_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0601_grid_ctrls { display: grid; grid-template-columns: repeat(auto-fit, minmax(140px, 1fr)); gap: 12px; }
  .sim-ep0601_px { width: 24px; height: 24px; border: 1px solid #e4dcc8; box-sizing: border-border; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_01: IoU (Intersección sobre Unión)</span>
  <span class="sim-ep0601_pill">IoU = |A &cap; B| / |A &cup; B|</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Controles -->
  <div class="sim-ep0601_panel" style="margin-bottom:14px;">
    <div class="sim-ep0601_grid_ctrls">
      
      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Desplazamiento H (&Delta;x)</label>
          <span id="sim-ep0601_vdx" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim-ep0601_dx" type="range" min="-3" max="3" step="1" value="0">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Desplazamiento V (&Delta;y)</label>
          <span id="sim-ep0601_vdy" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim-ep0601_dy" type="range" min="-3" max="3" step="1" value="0">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Lado del Cuadrado</label>
          <span id="sim-ep0601_vsz" style="font-family:monospace; font-weight:700; color:#26241d;">6</span>
        </div>
        <input id="sim-ep0601_sz" type="range" min="2" max="8" step="1" value="6">
      </div>

    </div>

    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:10px; text-align:center;">
      Desplace y redimensione la máscara predicha para evaluar la alineación.
    </div>
  </div>

  <!-- Exibição das Máscaras 10x10 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim-ep0601_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        Referencia (A)
      </div>
      <div id="sim-ep0601_ref" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0601_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        Predicha (B)
      </div>
      <div id="sim-ep0601_pred" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0601_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        Superposición (A &cap; B)
      </div>
      <div id="sim-ep0601_mix" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0601_dbg" class="sim-ep0601_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep01(root){
    if (!root || root.dataset.sim06Ep01Init) return;
    root.dataset.sim06Ep01Init = "1";

    var dx = root.querySelector("#sim-ep0601_dx");
    var dy = root.querySelector("#sim-ep0601_dy");
    var sz = root.querySelector("#sim-ep0601_sz");

    var vdx = root.querySelector("#sim-ep0601_vdx");
    var vdy = root.querySelector("#sim-ep0601_vdy");
    var vsz = root.querySelector("#sim-ep0601_vsz");

    var gRef  = root.querySelector("#sim-ep0601_ref");
    var gPred = root.querySelector("#sim-ep0601_pred");
    var gMix  = root.querySelector("#sim-ep0601_mix");

    var dbg = root.querySelector("#sim-ep0601_dbg");

    var N = 10;
    var ref = {x: 2, y: 2, w: 6, h: 6};

    function inside(x, y, r){
      return x >= r.x && x < r.x + r.w && y >= r.y && y < r.y + r.h;
    }

    function pixel(color){
      var d = document.createElement("div");
      d.className = "sim-ep0601_px";
      d.style.background = color;
      return d;
    }

    function classe(i){
      if (i >= 0.90) return "Excelente";
      if (i >= 0.75) return "Muito boa";
      if (i >= 0.50) return "Aceitável";
      return "Ruim";
    }

    function render(){
      vdx.textContent = dx.value;
      vdy.textContent = dy.value;
      vsz.textContent = sz.value;

      gRef.innerHTML  = "";
      gPred.innerHTML = "";
      gMix.innerHTML  = "";

      var pred = {
        x: ref.x + parseInt(dx.value, 10),
        y: ref.y + parseInt(dy.value, 10),
        w: parseInt(sz.value, 10),
        h: parseInt(sz.value, 10)
      };

      var inter = 0;
      var uniao = 0;

      for (var y = 0; y < N; y++){
        for (var x = 0; x < N; x++){
          var r = inside(x, y, ref);
          var p = inside(x, y, pred);

          gRef.appendChild(pixel(r ? "#7fdc92" : "#ffffff"));
          gPred.appendChild(pixel(p ? "#7fbfff" : "#ffffff"));

          if (r && p){
            gMix.appendChild(pixel("#9b59b6"));
            inter++;
          }
          else if (r){
            gMix.appendChild(pixel("#7fdc92"));
            uniao++;
          }
          else if (p){
            gMix.appendChild(pixel("#7fbfff"));
            uniao++;
          }
          else{
            gMix.appendChild(pixel("#ffffff"));
          }

          if (r && p) uniao++;
        }
      }

      var iou = inter / uniao;

      dbg.innerHTML =
        "<b>Interseção</b> = " + inter + " pixels &nbsp;&nbsp;&nbsp;" +
        "<b>União</b> = " + uniao + " pixels<br><br>" +
        "IoU = <b>" + inter + " / " + uniao + " = " + iou.toFixed(4) + "</b><br><br>" +
        "<span style='font-weight:700; color:#04342C;'>" + classe(iou) + "</span>";
    }

    dx.addEventListener('input', render);
    dy.addEventListener('input', render);
    sz.addEventListener('input', render);

    render();
  }

  function tryInitSim06Ep01(){
    var root = document.getElementById('sim-ep0601');
    if (root) initSim06Ep01(root); else setTimeout(tryInitSim06Ep01, 200);
  }
  tryInitSim06Ep01();
})();
</script>
""")

**Figura 6.21:** Simulador EP06_01: IoU entre máscara de referência e máscara predita


<figure id="fig-06-sim-ep0601">
  <img src="imagens/fig-06-sim-ep0601.png" alt=" Simulador EP06_01: IoU entre máscara de referência e máscara predita " style="max-width:80%" />
  <figcaption><strong>Figura 6.21:</strong>  Simulador EP06_01: IoU entre máscara de referência e máscara predita </figcaption>
</figure>

In [32]:
%%writefile EP06_01.py
# Código Python

Overwriting EP06_01.py


In [33]:
TestSuite("EP06_01.py").run()

### 6.0.2 EP06_02 🟢 Filtro de Marcadores por Circularidad

Tras la segmentación de una imagen, es común que se identifiquen diversos componentes conexos. En aplicaciones como la rectificación de documentos, solo algunos de estos componentes corresponden a los marcadores de referencia utilizados para el alineamiento de la imagen. Un criterio empleado con frecuencia para seleccionar estos marcadores es la **circularidad**, que mide cuán cercana es la forma de un componente a un círculo.

En este ejercicio, cada componente se describe por su área $A$ y su perímetro $P$. El objetivo es calcular su circularidad y decidir, a partir de un umbral proporcionado, si el componente debe ser aceptado o rechazado como candidato a marcador.

#### 6.0.2.1 📋 Directrices de Implementación

1. **Cantidad:** Leer el entero $N$ (número de candidatos) y el umbral de circularidad $C_{\text{umbral}}$ (número real).
2. **Datos de los candidatos:** Para cada uno de los $N$ candidatos, leer el área $A$ (entero) y el perímetro $P$ (número real).
3. **Circularidad:** Calcular $C=\frac{4\pi A}{P^2}$, donde:

- $A$ es el área del componente;
- $P$ es el perímetro del componente;
- $C$ es la circularidad.

4. **Caso degenerado:** Si $P=0$, considerar $C=0$ y clasificar directamente al candidato como `RECHAZADO`.
5. **Clasificación:** Si $C>C_{\text{umbral}}$, clasificar al candidato como `ACEPTADO`; en caso contrario, clasificarlo como `RECHAZADO`.
6. **Redondeo:** Mostrar el valor de $C$ con cuatro decimales.
7. **Salida:** Para cada candidato, imprimir el valor de $C$ seguido de la clasificación. Al final, imprimir el número total de candidatos aceptados.

#### 6.0.2.2 📌 Restricciones Computacionales

- Utilizar la constante $\pi$ de la biblioteca estándar del lenguaje (por ejemplo, `math.pi`), sin aproximaciones.
- La comparación debe realizarse con el valor de $C$ en precisión completa, antes del redondeo para la visualización.
- El criterio de aceptación es estricto ($C>C_{\text{umbral}}$).
- Si $P=0$, no se debe realizar la división.

#### 6.0.2.3 🧠 Fundamentación Teórica

La circularidad es un descriptor geométrico definido por $C=\frac{4\pi A}{P^2}$, donde:

- $A$ es el área del componente;
- $P$ es el perímetro del componente;
- $C$ es la circularidad.

Para un círculo perfecto, $C=1$. A medida que la forma se vuelve más alargada o irregular, el perímetro crece más rápidamente que el área, reduciendo el valor de $C$.

| Forma | Circularidad aproximada | Interpretación |
|---|---:|---|
| Círculo | $1{,}0000$ | Forma circular. |
| Cuadrado | $0{,}7854$ | Forma aproximadamente compacta. |
| Forma alargada o irregular | $C\ll1$ | Baja circularidad. |
| $P=0$ | $0$ (convención adoptada) | Contorno degenerado. |

La circularidad es invariante a la traslación, la rotación y la escala, y se utiliza ampliamente para distinguir componentes aproximadamente circulares de otros formatos.

#### 6.0.2.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

- Línea 1: entero $N$.
- Línea 2: número real $C_{\text{umbral}}$.
- Siguientes $N$ líneas: área $A$ (entero) y perímetro $P$ (real), separados por espacio.

**Salida:**

- Una línea para cada candidato, en el formato `C ACEPTADO` o `C RECHAZADO`, con $C$ presentado con cuatro decimales.
- Última línea: `Total aceptados: X`.

#### 6.0.2.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 3<br>0.6<br>78 31.4<br>100 40<br>50 60 | 0.9941 ACEPTADO<br>0.7854 ACEPTADO<br>0.1745 RECHAZADO<br>Total aceptados: 2 | Candidato aproximadamente circular, forma compacta y forma alargada. |
| 1<br>0.9<br>10 0 | 0.0000 RECHAZADO<br>Total aceptados: 0 | Perímetro nulo: contorno degenerado. |

In [34]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0602" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0602 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0602 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0602 button:hover { background: #e8dfcf; }
  #sim-ep0602 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0602_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0602_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_02: Filtro de Marcadores por Circularidad</span>
  <span class="sim-ep0602_pill">C = 4&pi;A / P&sup2;</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0602_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Umbral de Circularidad (C_umbral): <span id="sim-ep0602_vl" style="font-family:monospace; color:#26241d;">0.60</span>
      </label>
    </div>
    
    <input id="sim-ep0602_sl" type="range" min="0.05" max="0.99" step="0.01" value="0.60">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste el umbral y observe qué candidatos (discos, cuadrados y formas irregulares) sobreviven al filtro.
    </div>
  </div>

  <!-- Cards de Candidatos -->
  <div id="sim-ep0602_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(100px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0602_debug" class="sim-ep0602_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep02(root){
    if (!root || root.dataset.sim06Ep02Init) return;
    root.dataset.sim06Ep02Init = "1";

    var candidatos = [
      {nome: "Disco", A: 78, P: 31.4},
      {nome: "Quadrado", A: 100, P: 40},
      {nome: "Retângulo", A: 60, P: 44},
      {nome: "Rasura", A: 50, P: 60},
      {nome: "Ponto", A: 10, P: 0}
    ];

    var slEl  = root.querySelector('#sim-ep0602_sl');
    var vlEl  = root.querySelector('#sim-ep0602_vl');
    var cards = root.querySelector('#sim-ep0602_cards');
    var dbg   = root.querySelector('#sim-ep0602_debug');

    function render(){
      var th = parseFloat(slEl.value);
      vlEl.textContent = th.toFixed(2);
      cards.innerHTML = '';
      var aceitos = 0;

      candidatos.forEach(function(c){
        var C = (c.P === 0) ? 0 : (4 * Math.PI * c.A) / (c.P * c.P);
        var ok = c.P !== 0 && C > th;
        if (ok) aceitos++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (ok ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;');
        
        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">' + c.nome + '</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px; opacity:0.8;">A = ' + c.A + '<br>P = ' + c.P + '</div>' +
          '<div style="font-family:monospace; font-weight:700; margin-bottom:4px;">C = ' + C.toFixed(4) + '</div>' +
          '<div style="font-weight:700; font-size:10px; letter-spacing:0.04em;">' + (ok ? 'ACEITO' : 'REJEITADO') + '</div>';
        
        cards.appendChild(div);
      });

      dbg.textContent = 'C_umbral = ' + th.toFixed(2) + '  |  Candidatos aceitos: ' + aceitos + ' / ' + candidatos.length;
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep02(){
    var root = document.getElementById('sim-ep0602');
    if (root) initSim06Ep02(root); else setTimeout(tryInitSim06Ep02, 200);
  }
  tryInitSim06Ep02();
})();
</script>
""")

**Figura 6.22:** Simulador EP06_02: Filtro de Marcadores por Circularidad


<figure id="fig-06-sim-ep0602">
  <img src="imagens/fig-06-sim-ep0602.png" alt=" Simulador EP06_02: Filtro de Marcadores por Circularidad " style="max-width:80%" />
  <figcaption><strong>Figura 6.22:</strong>  Simulador EP06_02: Filtro de Marcadores por Circularidad </figcaption>
</figure>

In [35]:
%%writefile EP06_02.py
# Código Python

Overwriting EP06_02.py


In [36]:
TestSuite("EP06_02.py").run()

### 6.0.3 EP06_03 🟡 Clasificación de Marcaciones en Hojas de Respuesta (OMR)

Tras la corrección de la hoja y la segmentación de los cuadros de respuestas, el MCTest estima, para cada burbuja, un **grado de relleno**, representado por un valor entre $0$ y $100$. A partir de estos valores, el sistema debe determinar automáticamente la alternativa marcada, identificando también preguntas en blanco y casos de múltiples marcaciones.

En este ejercicio, implementará esta etapa de decisión del *pipeline* de OMR. La clasificación depende de un umbral de relleno: pequeñas variaciones en este valor pueden alterar el resultado de la lectura automática.

#### 6.0.3.1 📋 Directrices de Implementación

1. **Parámetros:** Leer los enteros $Q$ (número de preguntas) y $K$ (número de alternativas por pregunta, con $2 \le K \le 26$) y el umbral de relleno $\mathrm{Th}$ (número real entre $0$ y $100$).
2. **Grados de relleno:** Para cada una de las $Q$ preguntas, leer los $K$ valores reales correspondientes a las alternativas `A`, `B`, `C`, ..., en el orden de entrada.
3. **Conteo de marcaciones:** Para cada pregunta, contar cuántas alternativas poseen un grado de relleno **estrictamente mayor** que $\mathrm{Th}$.
4. **Clasificación:**
   - Si ninguna alternativa excede $\mathrm{Th}$, clasificar la pregunta como `BRANCO`.
   - Si exactamente una alternativa excede $\mathrm{Th}$, imprimir la letra correspondiente (`A`, `B`, `C`, ...).
   - Si dos o más alternativas exceden $\mathrm{Th}$, clasificar la pregunta como `DUPLA_MARCACAO`.
5. **Salida por pregunta:** Imprimir, en el orden de lectura, la clasificación de cada pregunta.
6. **Totales:** Al final, imprimir el número de preguntas `OK` (una única marcación), `BRANCO` y `DUPLA_MARCACAO`.

#### 6.0.3.2 📌 Restricciones Computacionales

* **Comparación estricta:** solo los valores mayores que $\mathrm{Th}$ se consideran marcaciones válidas; los valores exactamente iguales al umbral no deben contabilizarse.
* **Letras de las alternativas:** el índice $0$ corresponde a la alternativa `A`, el índice $1$ a la alternativa `B` y así sucesivamente.
* **Múltiples marcaciones:** siempre que dos o más alternativas excedan el umbral, la clasificación debe ser `DUPLA_MARCACAO`, independientemente de los respectivos grados de relleno.

#### 6.0.3.3 🧠 Fundamentación Teórica

| Situación | Clasificación | Interpretación |
|---|---|---|
| Exactamente una alternativa por encima del umbral | Letra de la alternativa | Respuesta válida |
| Ninguna alternativa por encima del umbral | `BRANCO` | Pregunta no respondida |
| Dos o más alternativas por encima del umbral | `DUPLA_MARCACAO` | Respuesta ambigua |

El umbral de relleno controla la sensibilidad del algoritmo. Valores muy bajos tienden a aumentar el número de `DUPLA_MARCACAO`, mientras que valores muy altos pueden aumentar la cantidad de preguntas clasificadas como `BRANCO`.

#### 6.0.3.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $Q$.
* Línea 2: Entero $K$.
* Línea 3: Número real $\mathrm{Th}$.
* Siguientes $Q$ líneas: $K$ números reales, correspondientes a los grados de relleno de las alternativas.
* Línea 1: Entero $Q$ y $K$.

**Salida:**

* $Q$ líneas, cada una conteniendo la clasificación de la respectiva pregunta.
* Línea final: `OK: x  BRANCO: y  DUPLA_MARCACAO: z`.

#### 6.0.3.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 3<br>4<br>50<br>10 85 5 12<br>20 15 18 22<br>90 88 10 5 | B<br>BRANCO<br>DUPLA_MARCACAO<br>OK: 1  BRANCO: 1  DUPLA_MARCACAO: 1 | En la primera pregunta solo `B` supera el umbral; en la segunda ninguna alternativa lo supera; en la tercera, `A` y `B` exceden el umbral. |
| 1<br>2<br>50.0<br>50 50 | BRANCO<br>OK: 0  BRANCO: 1  DUPLA_MARCACAO: 0 | Los valores iguales al umbral no se consideran marcaciones válidas. |

In [37]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0603" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0603 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0603 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0603 button:hover { background: #e8dfcf; }
  #sim-ep0603 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0603_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0603_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_03: Clasificación de Marcas OMR</span>
  <span class="sim-ep0603_pill">4 Alternativas</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0603_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Umbral de Relleno (Th): <span id="sim-ep0603_vth" style="font-family:monospace; color:#26241d;">50</span>%
      </label>
    </div>
    
    <input id="sim-ep0603_th" type="range" min="0" max="100" step="1" value="50">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste el grado de relleno de cada burbuja (A&ndash;D) y el umbral para observar la clasificación resultante.
    </div>
  </div>

  <!-- Sliders das Bolhas (A-D) -->
  <div class="sim-ep0603_panel" style="margin-bottom:14px;">
    <div id="sim-ep0603_bubbles" style="display:grid; grid-template-columns:repeat(4, 1fr); gap:12px;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0603_debug" class="sim-ep0603_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep03(root){
    if (!root || root.dataset.sim06Ep03Init) return;
    root.dataset.sim06Ep03Init = "1";

    var letras = ['A', 'B', 'C', 'D'];
    var valores = [10, 85, 5, 12];
    var thEl  = root.querySelector('#sim-ep0603_th');
    var vthEl = root.querySelector('#sim-ep0603_vth');
    var box   = root.querySelector('#sim-ep0603_bubbles');
    var dbg   = root.querySelector('#sim-ep0603_debug');

    box.innerHTML = '';
    var sliders = [];

    letras.forEach(function(L, i){
      var col = document.createElement('div');
      col.style.cssText = 'text-align:center; background:#fafaf7; border:1px solid #e9e3d3; padding:10px; border-radius:8px;';
      col.innerHTML = '<div style="font-weight:700; font-size:12px; color:#5e5a4a; margin-bottom:6px;">' + L + '</div>' +
        '<input type="range" min="0" max="100" step="1" value="' + valores[i] + '" id="sim-ep0603_b' + i + '">' +
        '<div id="sim-ep0603_v' + i + '" style="font-family:monospace; font-weight:700; font-size:11px; color:#26241d; margin-top:6px;">' + valores[i] + '%</div>';
      box.appendChild(col);
      sliders.push(col.querySelector('#sim-ep0603_b' + i));
    });

    function render(){
      var th = parseFloat(thEl.value);
      vthEl.textContent = th.toFixed(0);
      var marcadas = [];

      sliders.forEach(function(s, i){
        var v = parseFloat(s.value);
        root.querySelector('#sim-ep0603_v' + i).textContent = v.toFixed(0) + '%';
        if (v > th) marcadas.push(letras[i]);
      });

      var resultado;
      if (marcadas.length === 0) {
        resultado = 'BRANCO';
        dbg.style.borderColor = '#e4dcc8';
        dbg.style.background  = '#fafaf7';
        dbg.style.color       = '#8a8371';
      } else if (marcadas.length === 1) {
        resultado = 'RESPOSTA: ' + marcadas[0];
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        resultado = 'DUPLA_MARCACAO (' + marcadas.join(', ') + ')';
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = 'Clasificación de la pregunta: ' + resultado;
    }

    sliders.forEach(function(s){ s.addEventListener('input', render); });
    thEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep03(){
    var root = document.getElementById('sim-ep0603');
    if (root) initSim06Ep03(root); else setTimeout(tryInitSim06Ep03, 200);
  }
  tryInitSim06Ep03();
})();
</script>
""")

**Figura 6.23:** Simulador EP06_03: Clasificación de Marcaciones OMR


<figure id="fig-06-sim-ep0603">
  <img src="imagens/fig-06-sim-ep0603.png" alt=" Simulador EP06_03: Clasificación de Marcaciones OMR " style="max-width:80%" />
  <figcaption><strong>Figura 6.23:</strong>  Simulador EP06_03: Clasificación de Marcaciones OMR </figcaption>
</figure>

In [38]:
%%writefile EP06_03.py
# Código Python

Overwriting EP06_03.py


In [39]:
TestSuite("EP06_03.py").run()

### 6.0.4 EP06_04 🟡 Estimador de Inclinación por Mediana Angular (*Deskew*)

Tras la detección de bordes y la aplicación de la Transformada de Hough, se obtiene un conjunto de rectas candidatas a la orientación predominante del documento. Cada recta proporciona una estimación del ángulo de inclinación, calculada por

$$
\text{ángulo} = \operatorname{rad2deg}(\theta) - 90.
$$

Sin embargo, no todas las rectas corresponden a las líneas del documento: algunas resultan de ruidos, sombras u otros elementos de la imagen. En este ejercicio, implementará la etapa de estimación robusta del ángulo de inclinación, filtrando los valores plausibles y calculando su mediana.

#### 6.0.4.1 📋 Directrices de Implementación

1. **Cantidad:** Leer el entero $M$, correspondiente al número de ángulos estimados.
2. **Ángulos:** Leer los $M$ valores reales, en grados.
3. **Filtrado:** Mantener únicamente los ángulos que satisfagan **estrictamente** $-45 < \text{ángulo} < 45$.
4. **Ausencia de candidatos:** Si ningún ángulo permanece tras el filtrado, imprimir exactamente `SEM_CORRECAO`.
5. **Mediana:** Si existen ángulos válidos:
   - si la cantidad es impar, la mediana es el elemento central de la secuencia ordenada;
   - si es par, la mediana es la media aritmética de los dos elementos centrales.
6. **Salida:** Imprimir la mediana redondeada a dos cifras decimales (redondeo estándar, *round half away from zero*, con `np.floor(img + 0.5)`).

#### 6.0.4.2 📌 Restricciones Computacionales

* **Intervalo abierto:** los ángulos iguales a $-45$ o $45$ no deben considerarse.
* **Precisión:** calcular la mediana utilizando los valores originales; el redondeo debe realizarse únicamente en la salida.
* **Caso vacío:** si no hay ángulos válidos, no debe calcularse ninguna mediana.

#### 6.0.4.3 🧠 Fundamentación Teórica

| Situación | Resultado |
|---|---|
| Mayoría de los ángulos concentrados en torno a la inclinación real | La mediana aproxima la orientación del documento. |
| Pocos ángulos discrepantes (*outliers*) | La mediana sufre poca influencia de esos valores. |
| Ángulos fuera del intervalo $(-45^\circ,45^\circ)$ | Se descartan antes del cálculo. |
| Ningún ángulo válido | No se aplica corrección (`SEM_CORRECAO`). |

La mediana se utiliza por ser más robusta que la media en presencia de pocos valores discrepantes, produciendo una estimación más estable de la inclinación predominante del documento.

#### 6.0.4.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $M$.
* Línea 2: $M$ números reales, correspondientes a los ángulos en grados.

**Salida:**

* Una única línea que contenga el ángulo estimado, con dos cifras decimales, o la palabra `SEM_CORRECAO` si ningún ángulo es válido.

#### 6.0.4.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 5<br>-50 -10.5 2.3 2.3 47 | 2.30 | Solo se consideran los ángulos en el intervalo $(-45,45)$; la mediana es $2{,}3$. |
| 4<br>-46 50 45 -45 | SEM_CORRECAO | Ningún ángulo pertenece al intervalo abierto $(-45,45)$. |

In [40]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0604" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0604 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0604 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0604 button:hover { background: #e8dfcf; }
  #sim-ep0604 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0604_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0604_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_04: Estimador de Inclinación por Mediana Angular (Deskew)</span>
  <span class="sim-ep0604_pill">mediana(-45&deg; &lt; &theta; &lt; 45&deg;)</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0604_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Ángulo del Ruido Extra (&theta;_ruido): <span id="sim-ep0604_vl" style="font-family:monospace; color:#26241d;">47</span>&deg;
      </label>
    </div>
    
    <input id="sim-ep0604_sl" type="range" min="-80" max="80" step="1" value="47">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Arrastra el ángulo del ruido extra hacia dentro o fuera del intervalo [-45&deg;, +45&deg;] y observa cómo la mediana permanece estable.
    </div>
  </div>

  <!-- Exibição dos Ângulos Amostrados -->
  <div class="sim-ep0604_panel" style="margin-bottom:14px;">
    <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; text-align:center; letter-spacing:0.04em;">
      Muestras de Ángulos (Verde = Dentro del Rango, Rojo = Ruido Descartado)
    </div>
    <div id="sim-ep0604_pts" style="display:flex; gap:8px; flex-wrap:wrap; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0604_debug" class="sim-ep0604_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep04(root){
    if (!root || root.dataset.sim06Ep04Init) return;
    root.dataset.sim06Ep04Init = "1";

    var base = [-10.5, 2.3, 2.3];
    var slEl  = root.querySelector('#sim-ep0604_sl');
    var vlEl  = root.querySelector('#sim-ep0604_vl');
    var ptsEl = root.querySelector('#sim-ep0604_pts');
    var dbg   = root.querySelector('#sim-ep0604_debug');

    function median(arr){
      var a = arr.slice().sort(function(x, y){ return x - y; });
      var n = a.length;
      if (n === 0) return null;
      var mid = Math.floor(n / 2);
      return (n % 2 === 1) ? a[mid] : (a[mid - 1] + a[mid]) / 2;
    }

    function render(){
      var extra = parseFloat(slEl.value);
      vlEl.textContent = extra;
      var todos = base.concat([extra, -50]);
      var validos = todos.filter(function(a){ return a > -45 && a < 45; });
      
      ptsEl.innerHTML = '';
      todos.forEach(function(a){
        var ok = a > -45 && a < 45;
        var div = document.createElement('div');
        div.style.cssText = 'padding:8px 12px; border-radius:8px; font-family:monospace; font-size:12px; font-weight:700; transition:all 0.15s ease;' +
          (ok ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;');
        div.textContent = a + '°';
        ptsEl.appendChild(div);
      });

      var med = median(validos);

      if (med === null) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
        dbg.textContent = 'Válidos: []  |  Mediana estimada: SEM_CORRECAO';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
        dbg.textContent = 'Válidos: [' + validos.join(', ') + ']  |  Mediana estimada: ' + med.toFixed(2) + '°';
      }
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep04(){
    var root = document.getElementById('sim-ep0604');
    if (root) initSim06Ep04(root); else setTimeout(tryInitSim06Ep04, 200);
  }
  tryInitSim06Ep04();
})();
</script>
""")

**Figura 6.24:** Simulador EP06_04: Estimador de Pendiente por Mediana Angular


<figure id="fig-06-sim-ep0604">
  <img src="imagens/fig-06-sim-ep0604.png" alt=" Simulador EP06_04: Estimador de Pendiente por Mediana Angular " style="max-width:80%" />
  <figcaption><strong>Figura 6.24:</strong>  Simulador EP06_04: Estimador de Pendiente por Mediana Angular </figcaption>
</figure>

In [41]:
%%writefile EP06_04.py
# Código Python

Overwriting EP06_04.py


In [42]:
TestSuite("EP06_04.py").run()

### 6.0.5 EP06_05 🟠 Normalización de Fondo por División (Corrección de Iluminación)

Un formulario fue fotografiado bajo iluminación no uniforme, lo que hace que un lado de la hoja aparezca más claro que el otro. En estas condiciones, la umbralización global por Otsu puede producir resultados insatisfactorios, ya que un único umbral no separa adecuadamente el texto y el fondo en toda la imagen. La solución presentada en el capítulo consiste en **normalizar el fondo**, dividiendo la imagen original por una versión fuertemente suavizada de sí misma, que representa la iluminación de baja frecuencia.

En este ejercicio, la imagen original y el fondo suavizado (equivalente al resultado de un `cv2.GaussianBlur` con $\sigma$ elevado) ya son proporcionados. Su tarea es implementar la etapa de normalización que produce la imagen corregida.

#### 6.0.5.1 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Imagen original:** Leer los $L \times C$ valores enteros de la matriz `img` (intensidades entre 0 y 255).
3. **Fondo estimado:** Leer los $L \times C$ valores enteros de la matriz `bg` (intensidades entre 0 y 255, siempre estrictamente mayores que cero).
4. **Normalización:** Para cada posición $(i,j)$, calcular
$$
\text{valor}(i,j)=
\frac{\text{img}(i,j)}{\text{bg}(i,j)}\times255.
$$
5. **Redondeo:** Redondear el resultado al entero más cercano (*round half away from zero*, con `np.floor(img + 0.5)`).
6. **Saturación:** Limitar el valor obtenido al intervalo $[0,255]$.
7. **Salida:** Imprimir la matriz `img_norm` resultante.

#### 6.0.5.2 📌 Restricciones Computacionales

* **División por cero:** la entrada garantiza $\text{bg}(i,j)>0$ en todas las posiciones.
* **Orden de las operaciones:** primero redondear, luego aplicar la saturación.
* **Procesamiento independiente:** cada píxel debe normalizarse individualmente, sin utilizar información de los píxeles vecinos.

#### 6.0.5.3 🧠 Fundamentación Teórica

| Situación | Efecto de la normalización |
|---|---|
| $\text{img}(i,j)=\text{bg}(i,j)$ | Resultado igual a $255$, correspondiente al fondo normalizado. |
| $\text{img}(i,j)<\text{bg}(i,j)$ | Resultado menor que $255$, preservando regiones más oscuras, como el texto. |
| $\text{img}(i,j)>\text{bg}(i,j)$ | Resultado superior a $255$, posteriormente saturado. |
| Fondo con iluminación no uniforme | La división reduce las variaciones lentas de iluminación, haciendo la imagen más homogénea. |

La división por el fondo estimado reduce los efectos de la iluminación no uniforme y preserva el contraste entre el primer plano y el fondo, facilitando las etapas posteriores de segmentación.

#### 6.0.5.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Siguientes $L$ líneas: elementos de la matriz `img`.
* Siguientes $L$ líneas: elementos de la matriz `bg`.

**Salida:**

* Matriz `img_norm`, con $L$ filas y $C$ columnas, conteniendo valores enteros separados por espacios.

#### 6.0.5.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 2<br>2<br>60 120<br>180 40<br>100 100<br>200 80 | 153 255<br>230 128 | Los valores superiores a $255$ deben saturarse; $180/200\times255=229{,}5$ resulta en $230$ después del redondeo. |
| 1<br>3<br>30 60 90<br>60 60 60 | 128 255 255 | Solo el primer valor permanece por debajo de $255$ después de la normalización. |

In [43]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0605" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0605 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0605 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0605 button:hover { background: #e8dfcf; }
  #sim-ep0605 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0605_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0605_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim05_ep05_cell { width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 9px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_05: Normalización de Fondo por División</span>
  <span class="sim-ep0605_pill">(img / bg) &times; 255</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0605_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Intensidad del Fondo a la Izquierda (bg_izq): <span id="sim-ep0605_vl" style="font-family:monospace; color:#26241d;">100</span>
      </label>
    </div>
    
    <input id="sim-ep0605_sl" type="range" min="40" max="220" step="5" value="100">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajusta el gradiente de fondo (izquierda &rarr; derecha) y observa cómo la división cancela la variación de iluminación.
    </div>
  </div>

  <!-- Exibição das Matrizes 1x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim-ep0605_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        img (Original)
      </div>
      <div id="sim-ep0605_g_img" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0605_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        bg (Fondo Suavizado)
      </div>
      <div id="sim-ep0605_g_bg" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0605_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        img_norm (Salida)
      </div>
      <div id="sim-ep0605_g_out" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0605_debug" class="sim-ep0605_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep05(root){
    if (!root || root.dataset.sim06Ep05Init) return;
    root.dataset.sim06Ep05Init = "1";

    var linha_img = [90, 90, 90, 90];
    var slEl = root.querySelector('#sim-ep0605_sl');
    var vlEl = root.querySelector('#sim-ep0605_vl');
    var gImg = root.querySelector('#sim-ep0605_g_img');
    var gBg  = root.querySelector('#sim-ep0605_g_bg');
    var gOut = root.querySelector('#sim-ep0605_g_out');
    var dbg  = root.querySelector('#sim-ep0605_debug');

    function roundHalfAway(x){
      return x >= 0 ? Math.floor(x + 0.5) : Math.ceil(x - 0.5);
    }

    function cellStyle(v){
      var g = Math.max(0, Math.min(255, v));
      return 'background:rgb(' + g + ',' + g + ',' + g + '); color:' + (g > 140 ? '#000000' : '#ffffff') + ';';
    }

    function render(){
      var bgEsq = parseInt(slEl.value, 10);
      vlEl.textContent = bgEsq;

      // Gradiente linear de bgEsq até 200 na direita, 4 colunas
      var bg = [];
      for (var j = 0; j < 4; j++){
        bg.push(Math.round(bgEsq + (200 - bgEsq) * j / 3));
      }

      gImg.innerHTML = '';
      gBg.innerHTML  = '';
      gOut.innerHTML = '';
      
      var out = [];
      for (var j = 0; j < 4; j++){
        var v = (linha_img[j] / bg[j]) * 255;
        var r = roundHalfAway(v);
        var sat = Math.max(0, Math.min(255, r));
        out.push(sat);

        var ci = document.createElement('div');
        ci.className = 'sim05_ep05_cell';
        ci.style.cssText = cellStyle(linha_img[j]);
        ci.textContent = linha_img[j];
        gImg.appendChild(ci);

        var cb = document.createElement('div');
        cb.className = 'sim05_ep05_cell';
        cb.style.cssText = cellStyle(bg[j]);
        cb.textContent = bg[j];
        gBg.appendChild(cb);

        var co = document.createElement('div');
        co.className = 'sim05_ep05_cell';
        co.style.cssText = cellStyle(sat);
        co.textContent = sat;
        gOut.appendChild(co);
      }

      dbg.textContent = 'bg = [' + bg.join(', ') + ']  |  img_norm = [' + out.join(', ') + ']';
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep05(){
    var root = document.getElementById('sim-ep0605');
    if (root) initSim06Ep05(root); else setTimeout(tryInitSim06Ep05, 200);
  }
  tryInitSim06Ep05();
})();
</script>
""")

**Figura 6.25:** Simulador EP06_05: Normalización de Fondo por División


<figure id="fig-06-sim-ep0605">
  <img src="imagens/fig-06-sim-ep0605.png" alt=" Simulador EP06_05: Normalización de Fondo por División " style="max-width:80%" />
  <figcaption><strong>Figura 6.25:</strong>  Simulador EP06_05: Normalización de Fondo por División </figcaption>
</figure>

In [44]:
%%writefile EP06_05.py
# Código Python

Overwriting EP06_05.py


In [45]:
TestSuite("EP06_05.py").run()

### 6.0.6 EP06_06 🔴 Mapa de Varianza Local para Detección de Textura

Una fábrica de tejidos necesita inspeccionar rollos de tela en tiempo real, sin disponer de una imagen de referencia — cada rollo presenta pequeñas variaciones naturales. En esta situación, la estrategia presentada en el capítulo consiste en analizar la **homogeneidad local de la textura**: las regiones uniformes presentan baja varianza de intensidad en pequeñas vecindades, mientras que rasguños, manchas y fallas de fabricación producen aumentos locales de dicha varianza.

En este ejercicio, implementará el núcleo de ese método, calculando la varianza local en una ventana deslizante y generando una máscara binaria que identifica las regiones cuya varianza supera un umbral.

#### 6.0.6.1 📋 Directrices de Implementación

1. **Dimensiones y parámetros:** Leer los enteros $L$, $C$, $k$ (tamaño de la ventana, siempre impar) y $T$ (umbral de varianza).
2. **Imagen:** Leer los $L \times C$ valores enteros de la matriz de textura (intensidades entre 0 y 255).
3. **Tratamiento de bordes:** Cuando la ventana sobrepase los límites de la imagen, utilizar **replicación de borde**, es decir, repetir el valor del píxel válido más cercano.
4. **Media local:** Para cada posición $(i,j)$, calcular
$$
\mu(i,j)=
\frac{1}{k^2}
\sum_{(p,q)\in\text{ventana}}
\text{textura}(p,q).
$$
5. **Varianza local:** Calcular la varianza poblacional de la ventana,
$$
\sigma^2(i,j)=
\frac{1}{k^2}
\sum_{(p,q)\in\text{ventana}}
\left(\text{textura}(p,q)-\mu(i,j)\right)^2,
$$
o, de forma equivalente,
$$
\sigma^2(i,j)=\overline{x^2}-\mu(i,j)^2,
$$
donde $\overline{x^2}$ representa la media de los cuadrados de las intensidades.

6. **Redondeo:** Redondear la varianza al entero más cercano (*round half away from zero*, con `np.floor(res_norm + 0.5)`).

7. **Umbralización:** Definir $\text{máscara}(i,j)=1$ si la varianza redondeada es **estrictamente mayor** que $T$; de lo contrario, definir $\text{máscara}(i,j)=0$.

8. **Salida:** Imprimir la máscara binaria resultante.

#### 6.0.6.2 📌 Restricciones Computacionales

* **Replicación de borde:** utilizar el valor del píxel válido más cercano siempre que la ventana sobrepase los límites de la imagen.
* **Varianza poblacional:** utilizar denominador $k^2$, nunca $k^2-1$.
* **Comparación estricta:** la máscara debe calcularse utilizando la condición $\sigma^2_{\text{redondeada}}>T$.
* **Ventana impar:** el valor de $k$ es siempre impar, garantizando un píxel central.

#### 6.0.6.3 🧠 Fundamentación Teórica

| Situación | Varianza local | Interpretación |
|---|---|---|
| Región uniforme | Baja | Intensidades similares en la vecindad. |
| Región que contiene un defecto | Alta | La presencia de intensidades distintas aumenta la dispersión de los valores. |
| Ventana pequeña | Mayor sensibilidad a detalles y ruido | Detecta alteraciones localizadas. |
| Ventana grande | Respuesta más suave | Evidencia defectos mayores, pero reduce la precisión de su localización. |

La varianza local mide la dispersión de las intensidades en una vecindad. Las regiones homogéneas presentan baja varianza, mientras que las alteraciones en la textura aumentan esta medida, permitiendo identificar posibles defectos mediante una simple umbralización.

#### 6.0.6.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $k$ (impar).
* Línea 4: Entero $T$.
* Siguientes $L$ líneas: elementos enteros de la matriz de textura.

**Salida:**

* Máscara binaria (valores 0 o 1), con $L$ filas y $C$ columnas.

#### 6.0.6.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 3<br>3<br>3<br>50<br>10 10 10<br>10 10 10<br>10 90 10 | 0 0 0<br>1 1 1<br>1 1 1 | El defecto aumenta la varianza en todas las ventanas que lo contienen. |
| 2<br>2<br>3<br>5<br>100 100<br>100 100 | 0 0<br>0 0 | La textura es uniforme; la varianza es nula en toda la imagen. |

In [46]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0606" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0606 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0606 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0606 button:hover { background: #e8dfcf; }
  #sim-ep0606 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0606_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0606_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0606_cell { width: 44px; height: 44px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_06: Variância Local (Detección de Textura)</span>
  <span class="sim-ep0606_pill">&sigma;&sup2; = m&eacute;día(x&sup2;) &minus; m&eacute;día(x)&sup2;</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0606_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Intensidad del Defecto (Posición Central): <span id="sim-ep0606_vl_def" style="font-family:monospace; color:#26241d;">90</span>
      </label>
    </div>
    <input id="sim-ep0606_sl_def" type="range" min="10" max="255" step="5" value="90">

    <div style="display:flex; justify-content:space-between; align-items:center; margin:10px 0 4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Umbral (T): <span id="sim-ep0606_vl_t" style="font-family:monospace; color:#26241d;">50</span>
      </label>
    </div>
    <input id="sim-ep0606_sl_t" type="range" min="0" max="2000" step="10" value="50">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste el valor del defecto y el umbral T; observe cómo la ventana 3&times;3 propaga la detección por la vecindad.
    </div>
  </div>

  <!-- Exibição das Grades 3x3 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0606_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Textura (3&times;3)
      </div>
      <div id="sim-ep0606_g_tex" style="display:grid; grid-template-columns:repeat(3, 44px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0606_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Máscara de Defecto
      </div>
      <div id="sim-ep0606_g_mask" style="display:grid; grid-template-columns:repeat(3, 44px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0606_debug" class="sim-ep0606_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep06(root){
    if (!root || root.dataset.sim06Ep06Init) return;
    root.dataset.sim06Ep06Init = "1";

    var slDef = root.querySelector('#sim-ep0606_sl_def');
    var vlDef = root.querySelector('#sim-ep0606_vl_def');
    var slT   = root.querySelector('#sim-ep0606_sl_t');
    var vlT   = root.querySelector('#sim-ep0606_vl_t');
    var gTex  = root.querySelector('#sim-ep0606_g_tex');
    var gMask = root.querySelector('#sim-ep0606_g_mask');
    var dbg   = root.querySelector('#sim-ep0606_debug');

    function roundHalfAway(x){
      return x >= 0 ? Math.floor(x + 0.5) : Math.ceil(x - 0.5);
    }

    function clampIdx(v, n){
      return Math.max(0, Math.min(n - 1, v));
    }

    function render(){
      var defeito = parseInt(slDef.value, 10);
      var T       = parseInt(slT.value, 10);
      vlDef.textContent = defeito;
      vlT.textContent   = T;

      var N = 3;
      var tex = [[10, 10, 10], [10, defeito, 10], [10, 10, 10]];

      gTex.innerHTML  = '';
      gMask.innerHTML = '';
      
      var mask = [];
      for (var i = 0; i < N; i++){
        var row = [];
        for (var j = 0; j < N; j++){
          var vals = [];
          for (var di = -1; di <= 1; di++){
            for (var dj = -1; dj <= 1; dj++){
              var pi = clampIdx(i + di, N);
              var pj = clampIdx(j + dj, N);
              vals.push(tex[pi][pj]);
            }
          }
          var mean = vals.reduce(function(a, b){ return a + b; }, 0) / vals.length;
          var meanSq = vals.reduce(function(a, b){ return a + b * b; }, 0) / vals.length;
          var varr = meanSq - mean * mean;
          var varRound = roundHalfAway(varr);
          row.push(varRound > T ? 1 : 0);
        }
        mask.push(row);
      }

      var total = 0;
      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var g = tex[i][j];
          var ct = document.createElement('div');
          ct.className = 'sim-ep0606_cell';
          ct.style.cssText = 'background:rgb(' + g + ',' + g + ',' + g + '); color:' + (g > 140 ? '#000000' : '#ffffff') + ';';
          ct.textContent = g;
          gTex.appendChild(ct);

          var m = mask[i][j];
          if (m) total++;

          var cm = document.createElement('div');
          cm.className = 'sim-ep0606_cell';
          if (m) {
            cm.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          } else {
            cm.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
          }
          cm.textContent = m;
          gMask.appendChild(cm);
        }
      }

      if (total > 0) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'defecto = ' + defeito + '  |  T = ' + T + '  |  Pixels marcados: ' + total + ' / 9';
    }

    slDef.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep06(){
    var root = document.getElementById('sim-ep0606');
    if (root) initSim06Ep06(root); else setTimeout(tryInitSim06Ep06, 200);
  }
  tryInitSim06Ep06();
})();
</script>
""")

**Figura 6.26:** Simulador EP06_06: Mapa de Variância Local para Detecção de Textura


<figure id="fig-06-sim-ep0606">
  <img src="imagens/fig-06-sim-ep0606.png" alt=" Simulador EP06_06: Mapa de Variância Local para Detecção de Textura " style="max-width:80%" />
  <figcaption><strong>Figura 6.26:</strong>  Simulador EP06_06: Mapa de Variância Local para Detecção de Textura </figcaption>
</figure>

In [47]:
%%writefile EP06_06.py
# Código Python

Overwriting EP06_06.py


In [48]:
TestSuite("EP06_06.py").run()

### 6.0.7 EP06_07 🟣 *Pipeline* de Inspección Industrial: Registro por Traslación y Sustracción

En una línea de producción, una cámara fija fotografía cada pieza que pasa por la cinta transportadora, comparándola con una imagen de referencia sin defectos. El problema: pequeñas vibraciones de la cinta desplazan la pieza en relación con la posición de referencia en cada captura. Si la sustracción de imágenes se aplica directamente, sin corrección, el desplazamiento por sí solo ya genera diferencias enormes — **falsos positivos** que enmascaran los defectos reales.

Este es el ejercicio más completo del capítulo: debes **primero registrar** (alinear geométricamente) la imagen capturada usando un desplazamiento conocido $(dx, dy)$, proporcionado por un sensor de posición de la cinta, y **solo entonces aplicar la sustracción** con umbralización, exactamente como se describe en la sección de inspección industrial.

#### 6.0.7.1 📋 Directrices de Implementación

1. **Dimensiones y parámetros:** Leer $L$, $C$ (dimensiones de las imágenes), el desplazamiento entero conocido $dx, dy$ (pudiendo ser negativos) y el umbral de detección $T$ (entero).
2. **Imágenes:** Leer la matriz de referencia (`ref`, $L\times C$, sin defectos) y la matriz capturada (`cap`, $L\times C$, posiblemente desplazada y con defecto).
3. **Registro por traslación:** Construir la imagen alineada `alin` aplicando el desplazamiento $(dx,dy)$ recibido:
$$
\text{alin}(i,j) = \begin{cases} \text{cap}(i+dy,\; j+dx), & \text{si } (i+dy,\ j+dx) \in [0,L)\times[0,C) \\ 0, & \text{en caso contrario} \end{cases}
$$
4. **Relleno de borde:** Las posiciones que "salen" de la imagen capturada después del desplazamiento reciben el valor **0** (*zero-padding* — fuera del campo de visión de la cámara; **nota que este ejercicio usa cero, diferente de la replicación de borde del EP06_06**).
5. **Diferencia absoluta:** Calcular, píxel a píxel,
$$
\text{diff}(i,j) = |\text{ref}(i,j) - \text{alin}(i,j)|
$$
6. **Umbralización:** Definir $\text{máscara}(i,j) = 1$ si $\text{diff}(i,j) > T$; en caso contrario, $\text{máscara}(i,j) = 0$.
7. **Salida:** En este orden — (a) la matriz `alin` ($L\times C$); (b) la máscara de defecto ($L\times C$); (c) una última línea con el total de píxeles clasificados como defectuosos.

#### 6.0.7.2 📌 Restricciones Computacionales

* ***Zero-padding*, no replicación:** posiciones fuera de los límites de la imagen capturada, después del desplazamiento, valen exactamente 0 — este es el punto que más diferencia este ejercicio del EP06_06.
* **Comparación estricta:** $\text{diff}(i,j) > T$.
* **Signo de $(dx,dy)$:** el desplazamiento puede ser positivo o negativo; la fórmula del paso 3 debe aplicarse literalmente, sin invertir los signos.
* **Todos los valores son enteros:** no hay redondeo en esta etapa.

#### 6.0.7.3 🧠 Fundamentación Teórica

| Etapa omitida | Consecuencia |
|---|---|
| Omitir el registro geométrico | Todo el borde de la imagen (introducido por el desplazamiento) se marca como "defecto" — falso positivo sistemático |
| Registro con $(dx,dy)$ incorrecto | Pieza y referencia quedan desalineadas; la sustracción detecta contornos desplazados, no defectos reales |
| Umbral $T$ demasiado bajo | Ruido de captura (variaciones de 1–2 niveles de gris) se confunde con defecto |
| Umbral $T$ demasiado alto | Defectos sutiles dejan de ser detectados |

El registro geométrico y la sustracción son etapas complementarias: el primero garantiza que ambas imágenes representen exactamente la misma escena en el mismo referencial espacial; la segunda aísla lo que realmente cambió entre ellas — idealmente, solo los defectos.

#### 6.0.7.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Dos enteros $dx$ y $dy$, separados por espacio.
* Línea 4: Entero $T$.
* Siguientes $L$ líneas: elementos enteros de la matriz `ref`.
* Siguientes $L$ líneas: elementos enteros de la matriz `cap`.

**Salida:**

* $L$ líneas con la matriz `alin`.
* $L$ líneas con la máscara de defecto (0/1).
* Última línea: `Total de píxeles defectuosos: X`.

#### 6.0.7.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 3<br>3<br>1 0<br>30<br>50 50 50<br>50 50 50<br>50 50 50<br>0 50 50<br>0 50 90<br>0 50 50 | 50 50 0<br>50 90 0<br>50 50 0<br>0 0 1<br>0 1 1<br>0 0 1<br>Total de píxeles defectuosos: 4 | $dx=1$ desplaza la lectura una columna a la derecha; la última columna de `alin` queda sin correspondencia <br> (se convierte en 0) y se marca sistemáticamente; el defecto real (90) también se detecta. |
| 2<br>2<br>0 0<br>20<br>10 10<br>10 10<br>10 10<br>10 60 | 10 10<br>10 60<br>0 0<br>0 1<br>Total de píxeles defectuosos: 1 | Sin desplazamiento ($dx=dy=0$): `alin` es idéntica a `cap`; solo el defecto real (60) se detecta. |

In [49]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0607" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0607 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0607 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0607 button:hover { background: #e8dfcf; }
  #sim-ep0607 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0607_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0607_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0607_cell { width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 10px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_07: Registro por Traslación + Resta</span>
  <span class="sim-ep0607_pill">|ref &minus; alin(dx,dy)| &gt; T</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0607_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Desplazamiento Horizontal (dx): <span id="sim-ep0607_vl_dx" style="font-family:monospace; color:#26241d;">1</span>
      </label>
    </div>
    <input id="sim-ep0607_sl_dx" type="range" min="-2" max="2" step="1" value="1">

    <div style="display:flex; justify-content:space-between; align-items:center; margin:10px 0 4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Umbral (T): <span id="sim-ep0607_vl_t" style="font-family:monospace; color:#26241d;">30</span>
      </label>
    </div>
    <input id="sim-ep0607_sl_t" type="range" min="0" max="100" step="5" value="30">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste el desplazamiento de la correa (dx) y el umbral T. Observe cómo el borde "fantasma" desaparece cuando dx = 0.
    </div>
  </div>

  <!-- Exibição das Grades 3x3 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim-ep0607_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        ref
      </div>
      <div id="sim-ep0607_g_ref" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0607_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        alin (registrada)
      </div>
      <div id="sim-ep0607_g_alin" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0607_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        máscara
      </div>
      <div id="sim-ep0607_g_mask" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0607_debug" class="sim-ep0607_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep07(root){
    if (!root || root.dataset.sim06Ep07Init) return;
    root.dataset.sim06Ep07Init = "1";

    var N = 3;
    var ref = [[50, 50, 50], [50, 50, 50], [50, 50, 50]];
    // cap representa a peça deslocada 1 px à direita (col 0 = 0) mais um defeito em (1,2)
    var cap = [[0, 50, 50], [0, 50, 90], [0, 50, 50]];

    var slDx  = root.querySelector('#sim-ep0607_sl_dx');
    var vlDx  = root.querySelector('#sim-ep0607_vl_dx');
    var slT   = root.querySelector('#sim-ep0607_sl_t');
    var vlT   = root.querySelector('#sim-ep0607_vl_t');
    var gRef  = root.querySelector('#sim-ep0607_g_ref');
    var gAlin = root.querySelector('#sim-ep0607_g_alin');
    var gMask = root.querySelector('#sim-ep0607_g_mask');
    var dbg   = root.querySelector('#sim-ep0607_debug');

    function cellStyle(g){
      var v = Math.max(0, Math.min(255, g));
      return 'background:rgb(' + v + ',' + v + ',' + v + '); color:' + (v > 140 ? '#000000' : '#ffffff') + ';';
    }

    function render(){
      var dx = parseInt(slDx.value, 10);
      var T  = parseInt(slT.value, 10);
      vlDx.textContent = dx;
      vlT.textContent  = T;

      gRef.innerHTML  = '';
      gAlin.innerHTML = '';
      gMask.innerHTML = '';

      var alin = [], mask = [], total = 0;

      for (var i = 0; i < N; i++){
        var rowA = [], rowM = [];
        for (var j = 0; j < N; j++){
          var pj = j + dx;
          var v = (pj >= 0 && pj < N) ? cap[i][pj] : 0;
          rowA.push(v);

          var diff = Math.abs(ref[i][j] - v);
          var m = diff > T ? 1 : 0;
          if (m) total++;
          rowM.push(m);
        }
        alin.push(rowA);
        mask.push(rowM);
      }

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var cr = document.createElement('div');
          cr.className = 'sim-ep0607_cell';
          cr.style.cssText = cellStyle(ref[i][j]);
          cr.textContent = ref[i][j];
          gRef.appendChild(cr);

          var ca = document.createElement('div');
          ca.className = 'sim-ep0607_cell';
          ca.style.cssText = cellStyle(alin[i][j]);
          ca.textContent = alin[i][j];
          gAlin.appendChild(ca);

          var m = mask[i][j];
          var cm = document.createElement('div');
          cm.className = 'sim-ep0607_cell';
          if (m) {
            cm.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          } else {
            cm.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
          }
          cm.textContent = m;
          gMask.appendChild(cm);
        }
      }

      if (total > 0) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'dx = ' + dx + '  |  T = ' + T + '  |  Total de pixels defeituosos: ' + total + ' / 9';
    }

    slDx.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep07(){
    var root = document.getElementById('sim-ep0607');
    if (root) initSim06Ep07(root); else setTimeout(tryInitSim06Ep07, 200);
  }
  tryInitSim06Ep07();
})();
</script>
""")

**Figura 6.27:** Simulador EP06_07: *Pipeline* de Inspección — Registro por Traslación y Sustracción


<figure id="fig-06-sim-ep0607">
  <img src="imagens/fig-06-sim-ep0607.png" alt=" Simulador EP06_07: *Pipeline* de Inspección — Registro por Traslación y Sustracción " style="max-width:80%" />
  <figcaption><strong>Figura 6.27:</strong>  Simulador EP06_07: *Pipeline* de Inspección — Registro por Traslación y Sustracción </figcaption>
</figure>

In [50]:
%%writefile EP06_07.py
# Código Python

Overwriting EP06_07.py


In [51]:
TestSuite("EP06_07.py").run()

### 6.0.8 EP06_08 ⚫ Segmentación y Decodificación Real de *QRCode* con OpenCV

En los ejercicios anteriores, las magnitudes intermedias del *pipeline* de procesamiento de imágenes — como áreas, perímetros, varianzas y desplazamientos — se proporcionaron directamente o se calcularon a partir de matrices numéricas, sin necesidad de bibliotecas especializadas de Visión por Computador. En este ejercicio de cierre del capítulo, esta restricción se elimina de forma intencional: se utilizará la biblioteca **OpenCV** (`cv2`) para localizar y decodificar un *QRCode* real presente en una escena.

La propuesta reproduce un flujo simplificado de sistemas empleados en inspección visual, automatización industrial y lectura automática de documentos. Para mantener la entrada de datos accesible al contexto educativo, la carga de la imagen se integrará a la biblioteca didáctica `morph`, mediante la función `mm.readImg`.

La escena se proporciona en el formato **PGM ASCII (P2)** y contiene un único *QRCode* válido, además de diversos **objetos distractores**, como rectángulos, regiones de ruido texturizado y bloques aislados. La segmentación basada únicamente en propiedades geométricas — como área y forma aproximadamente cuadrada — es necesaria para reducir el espacio de búsqueda, pero no es suficiente para identificar el código correcto. La confirmación final se realizará exclusivamente mediante el intento de decodificación utilizando `cv2.QRCodeDetector`, procedimiento compatible con aplicaciones reales de reconocimiento automático.

#### 6.0.8.1 📋 Directrices de Implementación

1. **Lectura de las dimensiones y parámetros**

   Leer, en este orden, mediante la entrada estándar:

   - una línea que contenga el número de filas $L$;
   - una línea que contenga el número de columnas $C$;
   - una línea que contenga los cuatro parámetros del algoritmo separados por espacios:
     - umbral de binarización $T$ (entero);
     - área mínima $A_{\text{min}}$ (entero);
     - tolerancia de aspecto $\text{tol}$ (real);
     - margen $M$ (entero, en píxeles).

2. **Carga de la imagen**

   Utilizar la función didáctica `f = mm.readImg(L, C)` para leer los $L \times C$ valores de la imagen en tonos de gris, obteniendo un *array* de NumPy de tipo `uint8`.

3. **Binarización**

   Aplicar umbralización binaria invertida utilizando el umbral $T$. Todo píxel de la imagen original con intensidad estrictamente mayor que $T$ debe convertirse a 255, mientras que los demás deben asumir el valor 0.

4. **Detección de contornos**

   Extraer los componentes conectados externos utilizando `cv2.findContours(...)` con los parámetros:

   * `cv2.RETR_EXTERNAL`;
   * `cv2.CHAIN_APPROX_SIMPLE`.

5. **Filtrado geométrico**

   Para cada contorno encontrado:

   * calcular el rectángulo delimitador `(x, y, w, h)` mediante `cv2.boundingRect`;
   * mantener únicamente los candidatos que satisfagan simultáneamente:

     **Área mínima**

     $$
     w \times h > A_{\text{min}}
     $$

     **Razón de aspecto**

     $$
     \left|\frac{w}{h}-1\right| \le \text{tol}
     $$

6. **Ordenación de los candidatos**

   Ordenar los candidatos por el área del rectángulo delimitador

   $$
   w \times h
   $$

   en orden descendente.

   En caso de empate, preservar el orden originalmente devuelto por `cv2.findContours`.

7. **Verificación por decodificación**

   Para cada candidato, siguiendo el orden establecido:

   * expandir el rectángulo en $M$ píxeles en las cuatro direcciones;
   * limitar los índices para permanecer dentro de la imagen;
   * extraer el recorte directamente de la imagen original `f`;
   * aplicar `cv2.QRCodeDetector().detectAndDecode(...)` sobre ese recorte.

8. **Criterio de detención**

   Interrumpir inmediatamente el procesamiento cuando el primer candidato produzca una *cadena* decodificada no vacía.

9. **Caso no encontrado**

   Si ningún candidato se decodifica con éxito, imprimir exactamente: `QRCODE_NAO_ENCONTRADO`

10. **Salida (caso encontrado)**

    Imprimir dos líneas.

    Primera línea: `linha coluna altura largura` utilizando el rectángulo delimitador **original**, antes de la expansión por el margen $M$.

    Segunda línea: `texto_decodificado`


#### 6.0.8.2 📌 Restricciones Computacionales

* Utilizar funciones de OpenCV para realizar la binarización, la detección de contornos, el cálculo del rectángulo delimitador y la decodificación del QRCode.
* El filtrado geométrico debe ocurrir obligatoriamente antes de la etapa de decodificación.
* Utilizar exclusivamente el umbral fijo $T$ proporcionado en la entrada. No se permite utilizar métodos automáticos de umbralización, como Otsu o umbralización adaptativa.
* Garantizar que los recortes enviados al decodificador permanezcan dentro de los límites de la imagen.


#### 6.0.8.3 🧠 Fundamentación Teórica

| Etapa                    | Papel en el pipeline                                                                                      | Consecuencia si se omite                                                                      |
| ------------------------ | --------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------- |
| **Filtrado geométrico**  | Reduce el espacio de búsqueda seleccionando solo regiones compatibles con la geometría esperada de un QRCode. | El decodificador procesaría todos los contornos, incluyendo ruidos y objetos distractores.    |
| **Decodificación**       | Confirma semánticamente si el candidato contiene un QRCode válido.                                        | Objetos geométricamente similares podrían clasificarse incorrectamente como QRCode.           |
| **Margen $M$**           | Preserva la *zona de silencio* alrededor del código, facilitando su detección.                            | La ausencia de este margen puede impedir la alineación y la lectura correcta del código.      |

Este ejercicio integra conceptos estudiados a lo largo del capítulo en un único *pipeline* de Visión por Computador. La segmentación reduce el conjunto de regiones candidatas mediante características geométricas, mientras que la etapa de decodificación valida el contenido de la región utilizando un algoritmo especializado de reconocimiento.


#### 6.0.8.4 📦 Especificación de Entrada y Salida (VPL)

**Estructura de Entrada**

```
L
C
T A_min tol M
[matriz de la imagen]
```

**Estructura de Salida (Éxito)**

```
linha coluna altura largura
texto_decodificado
```

**Estructura de Salida (Fallo)**

```
QRCODE_NAO_ENCONTRADO
```

#### 6.0.8.5 📌 Archivos de Referencia (.pgm)


Para fines de validación, depuración local y análisis de matrices reales de píxeles, los archivos de imagen generados en el estándar ASCII P2 se encuentran disponibles en el directorio del proyecto. Puede utilizarlos para probar con decodificadores de su teléfono móvil la adherencia de su código (guardar *.pgm localmente para visualizar):

* 📥 **[Caso 1: Patrón Normal](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso1_Normal.pgm)** – Contiene un único código perfectamente centrado con distractores geométricos simples en la periferia.
* 📥 **[Caso 2: Escenario Complejo](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso2_Complexo.pgm)** – Presenta mayor densidad de ruido texturizado y múltiples distractores candidatos que ponen a prueba los límites del filtrado por aspecto.
* 📥 **[Caso 3: Mensaje Expandido](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso3_MensagemSecreta.pgm)** – Contiene un QRCode estructurado a partir de una cadena de caracteres de mayor longitud, generando una mayor densidad de módulos internos.
* 📥 **[Caso 4: Geometría Compacta](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso4_Excelente.pgm)** – Evalúa el comportamiento del pipeline bajo condiciones optimizadas de contraste y posicionamiento límite.
* 📥 **[Caso 5: Escenario de Exclusión](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso5_Nao_Encontrado.pgm)** – Imagen compuesta puramente por elementos distractores de alta área, diseñada para validar el comportamiento de fallo controlado del programa.

In [52]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0608" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<!-- Cabeçalho no padrão institucional -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">📋 Simulador EP06_08: Segmentación y Decodificación de Código QR</span>
  <span style="font-size:10px;font-weight:700;padding:3px 10px;border-radius:40px;border:1px solid #e4dcc8;background:#26241d;color:#7ee7c6;font-family:monospace;">Filtro Geométrico &rarr; Parada Semántica</span>
</div>

<div style="padding:16px;background:#ffffff;">
  <p style="margin:0 0 14px 0;font-size:11px;color:#8a8371;line-height:1.5;text-align:center;font-weight:600;">
    Ajusta interactivamente los parámetros de entrada del algoritmo (A_min y tol) para verificar qué componentes se filtran geométricamente y cómo el criterio de parada por análisis semántico interrumpe el escaneo de la cola.
  </p>
  
  <div style="display:flex;gap:14px;margin-bottom:14px;flex-wrap:wrap;">
    <!-- Slider Area Minima -->
    <div style="flex:1;min-width:200px;background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#5e5a4a;">Área mínima (A_min, px&sup2;)</label>
        <span id="sim-ep0608_vl_amin" style="font-family:monospace;font-weight:700;color:#26241d;">250</span>
      </div>
      <input id="sim-ep0608_sl_amin" style="width:100%;accent-color:#26241d;cursor:pointer;height:4px;" max="3000" min="0" step="50" type="range" value="250">
    </div>
    
    <!-- Slider Tolerancia -->
    <div style="flex:1;min-width:200px;background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#5e5a4a;">Tolerancia de aspecto (tol)</label>
        <span id="sim-ep0608_vl_tol" style="font-family:monospace;font-weight:700;color:#26241d;">0.22</span>
      </div>
      <input id="sim-ep0608_sl_tol" style="width:100%;accent-color:#26241d;cursor:pointer;height:4px;" max="1.0" min="0.05" step="0.01" type="range" value="0.22">
    </div>
  </div>

  <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(240px, 1fr));gap:14px;margin-bottom:14px;">
    <!-- Canvas da Cena -->
    <div style="text-align:center;background:#fafaf7;border:1px solid #e9e3d3;padding:14px;border-radius:12px;">
      <div style="font-size:10px;font-weight:700;color:#8a8371;text-transform:uppercase;margin-bottom:10px;letter-spacing:0.04em;">Visualización de la Escena (Matriz f)</div>
      <canvas id="sim-ep0608_canvas" width="260" height="260" style="border:1px solid #e4dcc8;border-radius:10px;background:#ffffff;margin:0 auto;display:block;"></canvas>
    </div>
    
    <!-- Lista de Candidatos -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;padding:14px;border-radius:12px;">
      <div style="font-size:10px;font-weight:700;color:#8a8371;text-transform:uppercase;margin-bottom:10px;text-align:center;letter-spacing:0.04em;">Componentes Conectados en la Cola</div>
      <div id="sim-ep0608_lista" style="font-family:monospace;font-size:11px;display:flex;flex-direction:column;gap:8px;"></div>
    </div>
  </div>
  
  <!-- Console de Saída VPL -->
  <div id="sim-ep0608_debug" style="background:#fafaf7;border-radius:12px;padding:12px;border:1px solid #e9e3d3;font-family:monospace;font-size:11px;color:#26241d;text-align:center;"></div>
</div>

<script>
(function(){
  function initSim06Ep08(root){
    if(!root || root.dataset.sim06Ep08Init) return;
    root.dataset.sim06Ep08Init = "1";

    var formas = [
      {x: 145, y: 35,  w: 76, h: 76, tipo: "Componente QRCode Real", cor: "#cbd5e1", decodifica: true, padrao: "qr"},
      {x: 35,  y: 145, w: 55, h: 68, tipo: "Falso QRCode (Assimétrico)", cor: "#e2e8f0", decodifica: false, padrao: "falso_qr"},
      {x: 45,  y: 35,  w: 44, h: 44, tipo: "Círculo / Distrator", cor: "#f1f5f9", decodifica: false, padrao: "circulo"},
      {x: 160, y: 175, w: 68, h: 26, tipo: "Retângulo Distrator", cor: "#e2e8f0", decodifica: false, padrao: "retangulo"},
      {x: 65,  y: 220, w: 14, h: 14, tipo: "Ruído Isolado", cor: "#f8fafc", decodifica: false, padrao: "ruido"}
    ];

    var slA = root.querySelector('#sim-ep0608_sl_amin');
    var vlA = root.querySelector('#sim-ep0608_vl_amin');
    var slT = root.querySelector('#sim-ep0608_sl_tol');
    var vlT = root.querySelector('#sim-ep0608_vl_tol');
    var canvas = root.querySelector('#sim-ep0608_canvas');
    var ctx = canvas.getContext('2d');
    var lista = root.querySelector('#sim-ep0608_lista');
    var dbg = root.querySelector('#sim-ep0608_debug');

    function desenhaForma(f, estado){
      ctx.save();
      
      var corBorda = '#94a3b8';
      if (estado === 'candidato_ok') corBorda = '#10b981';
      if (estado === 'candidato_falhou') corBorda = '#f43f5e';
      if (estado === 'rejeitado') corBorda = '#cbd5e1';

      ctx.lineWidth = (estado === 'candidato_ok' || estado === 'candidato_falhou') ? 3 : 1.5;
      ctx.strokeStyle = corBorda;

      if (estado === 'rejeitado') {
        ctx.fillStyle = '#f8fafc';
      } else {
        if(f.padrao === 'qr') ctx.fillStyle = '#e2e8f0';
        else if(f.padrao === 'retangulo') ctx.fillStyle = '#fffbeb';
        else if(f.padrao === 'falso_qr') ctx.fillStyle = '#f0f9ff';
        else ctx.fillStyle = '#fdf4ff';
      }

      if(f.padrao === 'circulo'){
        ctx.beginPath();
        ctx.arc(f.x + f.w/2, f.y + f.h/2, f.w/2, 0, 2 * Math.PI);
        ctx.fill(); ctx.stroke();
      } else {
        ctx.fillRect(f.x, f.y, f.w, f.h);
        ctx.strokeRect(f.x, f.y, f.w, f.h);
        
        if(f.padrao === 'qr' || f.padrao === 'falso_qr'){
          var c = f.w / 5;
          ctx.fillStyle = '#ffffff';
          [[f.x + 3, f.y + 3], [f.x + f.w - c - 3, f.y + 3], [f.x + 3, f.y + f.h - c - 3]].forEach(function(p){
            ctx.fillRect(p[0], p[1], c, c);
            ctx.strokeRect(p[0], p[1], c, c);
          });
          
          ctx.fillStyle = (f.padrao === 'qr') ? '#334155' : '#64748b';
          [[f.x + 5, f.y + 5], [f.x + f.w - c + 1, f.y + 5], [f.x + 5, f.y + f.h - c + 1]].forEach(function(p){
            ctx.fillRect(p[0], p[1], c - 4, c - 4);
          });
        }
      }
      ctx.restore();
    }

    function render(){
      var amin = parseInt(slA.value, 10);
      var tol = parseFloat(slT.value);
      vlA.textContent = amin;
      vlT.textContent = tol.toFixed(2);

      ctx.clearRect(0, 0, canvas.width, canvas.height);
      ctx.fillStyle = '#ffffff';
      ctx.fillRect(0, 0, canvas.width, canvas.height);

      var candidatos = formas.map(function(f){
        var area = f.w * f.h;
        var aspecto = f.w / f.h;
        var passaArea = area > amin;
        var passaAspecto = Math.abs(aspecto - 1.0) <= tol;
        return {f: f, area: area, aspecto: aspecto, passa: passaArea && passaAspecto};
      }).sort(function(a, b){ return b.area - a.area; });

      lista.innerHTML = '';
      var encontrado = null;
      var flagParada = false;

      candidatos.forEach(function(c){
        var estado, texto, bgBox, txBox;
        
        if(!c.passa){
          estado = 'rejeitado';
          texto = 'REJEITADO (Área = ' + c.area + ' px&sup2;, Aspeto = ' + c.aspecto.toFixed(2) + ')';
          bgBox = '#f1f5f9';
          txBox = '#94a3b8';
        } else if(flagParada){
          estado = 'rejeitado';
          texto = 'FILA INTERROMPIDA (Critério de Parada Ativo)';
          bgBox = '#f8fafc';
          txBox = '#cbd5e1';
        } else if(c.f.decodifica){
          estado = 'candidato_ok';
          texto = 'SUCESSO: DECODIFICADO &#10004;';
          bgBox = '#ecfdf5';
          txBox = '#059669';
          encontrado = c.f;
          flagParada = true;
        } else {
          estado = 'candidato_falhou';
          texto = 'GEOMETRIA OK &rarr; FALHA NA DECODIFICAÇÃO &#10008;';
          bgBox = '#fff5f5';
          txBox = '#e11d48';
        }
        
        desenhaForma(c.f, estado);
        
        var div = document.createElement('div');
        div.style.cssText = 'padding:8px 10px;border-radius:8px;background:' + bgBox + ';border:1px solid #edf2f7;color:' + txBox + ';display:flex;flex-direction:column;gap:2px;';
        
        var nameSpan = document.createElement('strong');
        nameSpan.style.fontSize = '11px';
        nameSpan.textContent = c.f.tipo + ' (' + c.area + ' px²)';
        
        var statusSpan = document.createElement('span');
        statusSpan.style.fontSize = '10px';
        statusSpan.style.opacity = '0.9';
        statusSpan.innerHTML = texto;

        div.appendChild(nameSpan);
        div.appendChild(statusSpan);
        lista.appendChild(div);
      });

      if (encontrado) {
        dbg.style.backgroundColor = '#eafaf1';
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.color = '#04342C';
        dbg.innerHTML = '<div style="text-align:left;font-weight:700;color:#04342C;margin-bottom:4px;">&#128994; SAÍDA PADRÃO (VPL):</div>' +
                        'y=' + encontrado.y + ' x=' + encontrado.x + ' h=' + encontrado.h + ' w=' + encontrado.w + '<br>' +
                        '<span style="color:#04342C;font-weight:700;">"EP06_08 - PDI-VC | Parabens! Voce decodificou este QR Code!"</span>';
      } else {
        dbg.style.backgroundColor = '#fdecea';
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.color = '#c0392b';
        dbg.innerHTML = '<div style="text-align:left;font-weight:700;color:#c0392b;margin-bottom:4px;">&#128308; SAÍDA PADRÃO (VPL):</div>' +
                        'QRCODE_NAO_ENCONTRADO';
      }
    }

    slA.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }
  
  function tryInitSim06Ep08(){
    var root = document.getElementById('sim-ep0608');
    if(root) initSim06Ep08(root); else setTimeout(tryInitSim06Ep08, 200);
  }
  tryInitSim06Ep08();
})();
</script>
</div>
""")

**Figura 6.28:** Simulador EP06_08: Segmentación Geométrica + Verificación por Decodificación de Código QR


<figure id="fig-06-sim-ep0608">
  <img src="imagens/fig-06-sim-ep0608.png" alt=" Simulador EP06_08: Segmentación Geométrica + Verificación por Decodificación de Código QR " style="max-width:80%" />
  <figcaption><strong>Figura 6.28:</strong>  Simulador EP06_08: Segmentación Geométrica + Verificación por Decodificación de Código QR </figcaption>
</figure>

In [53]:
%%writefile EP06_08.py
# Código Python

Overwriting EP06_08.py


In [54]:
TestSuite("EP06_08.py").run()

## Referências do Capítulo


BAHDANAU, D.; CHO, K.; BENGIO, Y. **Neural Machine Translation by Jointly Learning to Align and Translate**. 2015.

BERGMANN, P. *et al*. **MVTec AD -- A Comprehensive Real-World Dataset for Unsupervised Anomaly Detection**. 2019.

BRADSKI, Gary; KAEHLER, Adrian. **Learning OpenCV: Computer vision with the OpenCV library**. " O'Reilly Media, Inc.", 2008.

GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. New York, Pearson, 2018.

SMITH, R. **An Overview of the Tesseract OCR Engine**. IEEE Computer Society, 2007.

SMITH, R. **History of the Tesseract OCR Engine: What Worked and What Didn't**. 2013.

SONG, K.; YAN, Y. **A Noise Robust Method Based on Completed Local Binary Patterns for Hot-Rolled Steel Strip Surface Defects**. 2013.

SZELISKI, Richard. **Computer Vision: Algorithms and Applications**. Springer, 2022.

TABERNIK, D. *et al*. **Segmentation-Based Deep-Learning Approach for Surface-Defect Detection**. 2020.

VASWANI, A. *et al*. **Attention Is All You Need**. 2017.

ZAMPIROLLI, F. A. **MCTest: Como Criar e Corrigir Exames Parametrizados**. Independente, 2023.